# TCC — template estático _new

**Objetivo:** comparar CNNs como extratoras congeladas de imagens da poça de fusão em PTA-DED, usando o mesmo regressor e a mesma base.

**Fluxo:** configuração → metadados ordenados → rótulos da Sofia → validação → features → split aleatório → blocos com teste final → blocos com teste central → LOEO → resultado principal.

Execute as células em ordem. Para outra CNN, altere ARQUITETURA na etapa 1. As validações ficam em células próprias. O teste com o CSV _old_adaptado foi preservado, mas está **comentado**.

- O perfil denso da Sofia é reamostrado nos frames. Os limites e a correspondência física entre vídeo e perfil continuam provisórios; ordenação não comprova alinhamento físico.
- USAR_PARAMETROS controla CNN isolada ou CNN + parâmetros. StandardScaler é ajustado apenas no treino de cada divisão.
- Regressor: Dense(32, ReLU) → Dense(1), Adam e MAE; dimensão de entrada automática.
- Quatro protocolos estáticos, sem construção de features temporais. Blocos cronológicos são uma forma de avaliar, não um modelo temporal.
- **Onde olhar:** a tabela principal da seção 10. Detalhes por experimento e custo do extrator ficam recolhidos.

**Após esta reorganização:** reinicie o ambiente e execute em ordem para não reutilizar a variável dataset de testes anteriores. REGERAR_FEATURES=False só pode reutilizar artefatos da mesma base/configuração; se a conferência falhar, selecione a pasta correta ou reextraia. Não é necessário apagar resultados antigos.

## 1. Configuração única

Selecione GPU no ambiente do Colab. Troque somente `ARQUITETURA` para comparar CNNs.
A resolução padrão é a recomendada por cada arquitetura: 299×299 para InceptionResNetV2
e 224×224 para as demais. Isso compara cada configuração completa; para controlar a
resolução, use `TAMANHO_COMUM=(299,299)` em todas. Pooling e regressor ficam iguais.


In [ ]:
from pathlib import Path, PurePosixPath
from zipfile import ZipFile
from datetime import datetime, timezone
from time import perf_counter
import hashlib, json, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from IPython.display import display
from google.colab import drive

drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/TCC')
ZIP_PATH = PROJECT_ROOT / 'Dados Experimentos/experimentos_taguchi_400p_cropped_images.zip'
SOFIA_PATH = PROJECT_ROOT / 'Dados Experimentos/cordon_width_profiles_dense_oriented.csv'
OUTPUT_ROOT = PROJECT_ROOT / 'resultados/cnn_new_estatico_v2'

POOLING = 'max'
TAMANHO_COMUM = None

ARQUITETURA = 'InceptionResNetV2'  # , , EfficientNetB0,InceptionResNetV2,ResNet50,DenseNet169
USAR_DATASET_ANTIGO = False
USAR_PARAMETROS = False

REGERAR_FEATURES = True


SEMENTE = 42
BATCH_EXTRACAO = 30
BATCH_REGRESSAO = 128
EPOCAS = 300
PACIENCIA = 20
TAXA_APRENDIZADO = 0.001
EXPERIMENTOS = [f'L{i}' for i in range(1, 10)]
INTERVALOS = {
    'L1': (1450, 5713), 'L2': (600, 4550), 'L3': (350, 4200),
    'L4': (750, 3800), 'L5': (600, 3700), 'L6': (650, 3600),
    'L7': (650, 2400), 'L8': (800, 2650), 'L9': (750, 2650),
}


COLUNAS_PARAMETROS = ['current', 'nozzle_speed', 'mass_flow_rate']
# Mesmos valores registrados no notebook anterior; unidades da fonte devem ser
# mantidas no texto do TCC. Aqui são apenas os valores numéricos originais.
PARAMETROS = pd.DataFrame([
    ['L1',120,60,10], ['L2',160,60,15], ['L3',200,60,20],
    ['L4',120,80,10], ['L5',160,80,15], ['L6',200,80,20],
    ['L7',120,100,10], ['L8',160,100,15], ['L9',200,100,20],
], columns=['experimento'] + COLUNAS_PARAMETROS)
keras.utils.set_random_seed(SEMENTE)
tf.config.experimental.enable_op_determinism()
print('Arquitetura:', ARQUITETURA, '| Parâmetros como entrada:', USAR_PARAMETROS)
print('TensorFlow:', tf.__version__, '| GPUs:', tf.config.list_physical_devices('GPU'))


## 2. Imagens e metadados ordenados

O ZIP permanece no Drive. As imagens são extraídas no disco temporário do Colab para
evitar milhares de leituras remotas. O inventário do ZIP identifica a origem do conjunto.
Um PNG já extraído é reaproveitado apenas se o tamanho esperado estiver presente.
As features serão sempre calculadas novamente na etapa 5.


In [ ]:
with ZipFile(ZIP_PATH) as arquivo:
    membros = []
    for item in arquivo.infolist():
        partes = PurePosixPath(item.filename).parts
        if (len(partes) == 3 and partes[0] == 'experimentos_taguchi_400p_cropped_images'
                and partes[1] in EXPERIMENTOS and partes[2].lower().endswith('.png')):
            membros.append(item)
    inventario = sorted((m.filename, m.file_size, m.CRC) for m in membros)
    zip_id = hashlib.sha256(json.dumps(inventario).encode()).hexdigest()[:16]
    EXTRACAO_ROOT = Path('/content/tcc_estatico_new') / zip_id
    DATASET_ROOT = EXTRACAO_ROOT / 'experimentos_taguchi_400p_cropped_images'
    for item in membros:
        destino = EXTRACAO_ROOT / item.filename
        if not destino.is_file() or destino.stat().st_size != item.file_size:
            arquivo.extract(item, EXTRACAO_ROOT)

linhas = []
for experimento in EXPERIMENTOS:
    arquivos = sorted((DATASET_ROOT / experimento).glob('*.png'), key=lambda p: int(p.stem))
    for posicao, caminho in enumerate(arquivos):
        linhas.append({'experimento': experimento, 'frame': int(caminho.stem),
                       'posicao_no_experimento': posicao,
                       'caminho_relativo': caminho.relative_to(DATASET_ROOT).as_posix()})
metadados = pd.DataFrame(linhas)
metadados.insert(0, 'indice_global', np.arange(len(metadados)))
perfil_sofia = pd.read_csv(SOFIA_PATH)
display(metadados.head())
print(f'{len(metadados):,} imagens | origem ZIP: {zip_id}')


In [ ]:
# VALIDAÇÃO 1 — imagens, ordem e perfil de entrada (não modifica rótulos)
assert len(metadados) == len(membros) > 0, 'Inventário de imagens incompleto.'
assert set(metadados.experimento) == set(EXPERIMENTOS), 'Experimento ausente.'
assert not metadados.duplicated(['experimento','frame']).any(), 'Frame duplicado.'
assert metadados.groupby('experimento').frame.apply(lambda s: s.is_monotonic_increasing).all()
assert {'experimento','posicao_temporal_normalizada','largura_mm'} <= set(perfil_sofia)
assert set(perfil_sofia.experimento) == set(EXPERIMENTOS)
for exp, perfil in perfil_sofia.groupby('experimento'):
    perfil = perfil.sort_values('posicao_temporal_normalizada')
    x = perfil.posicao_temporal_normalizada.to_numpy()
    y = perfil.largura_mm.to_numpy()
    assert len(x) > 1 and np.isfinite(x).all() and np.isfinite(y).all(), exp
    assert (np.diff(x) > 0).all() and (y > 0).all(), exp
    assert np.allclose([x[0],x[-1]], [0,1]), f'{exp}: cobertura incompleta do perfil.'
display(metadados.groupby('experimento').agg(
    imagens=('frame','size'), primeiro_frame=('frame','min'), ultimo_frame=('frame','max')))
print('Integridade estrutural conferida. Correspondência física continua como hipótese documentada.')
display(metadados.head(6))


## 3. Associar o perfil da Sofia

Selecionamos os frames pelos intervalos da configuração. A posição é calculada por
`(frame − início)/(fim − 1 − início)`, preservando possíveis lacunas da numeração.
Nos dados completos e consecutivos, isso equivale ao `linspace` usado anteriormente.
O começo e o fim selecionados são associados aos extremos do perfil: essa é uma
**hipótese de alinhamento**, não uma medição física comprovada. A orientação do CSV é preservada.


In [ ]:
def associar_perfis(metadados, perfis, intervalos):
    partes = []
    for exp, (inicio, fim) in intervalos.items():
        imagens = metadados.loc[(metadados.experimento == exp)
                               & (metadados.frame >= inicio) & (metadados.frame < fim)].copy()
        perfil = perfis.loc[perfis.experimento == exp].sort_values('posicao_temporal_normalizada')
        posicao = (imagens.frame.to_numpy() - inicio) / (fim - 1 - inicio)
        imagens['posicao_normalizada'] = posicao
        imagens['y'] = np.interp(posicao, perfil.posicao_temporal_normalizada,
                                 perfil.largura_mm).astype(np.float32)
        imagens['origem_y'] = 'sofia_perfil_denso_orientado_interpolacao_linear'
        partes.append(imagens)
    base = pd.concat(partes, ignore_index=True)
    base.insert(0, 'indice_dataset', np.arange(len(base)))
    return base

dataset = associar_perfis(metadados, perfil_sofia, INTERVALOS)
dataset = dataset.merge(PARAMETROS, on='experimento', how='left', validate='many_to_one', sort=False)
display(dataset.groupby('experimento', sort=False).head(2))


In [ ]:
# VALIDAÇÃO 2 — alinhamento estrutural da base e identificação reprodutível
assert np.array_equal(dataset.indice_dataset, np.arange(len(dataset)))
assert not dataset.duplicated(['experimento','frame']).any()
assert np.isfinite(dataset[['y'] + COLUNAS_PARAMETROS].to_numpy()).all()
assert (dataset.y > 0).all()
for exp, (inicio, fim) in INTERVALOS.items():
    frames = dataset.loc[dataset.experimento == exp, 'frame'].to_numpy()
    assert np.array_equal(frames, np.arange(inicio, fim)), f'{exp}: frames ausentes ou fora de ordem.'

sofia_sha256 = hashlib.sha256(SOFIA_PATH.read_bytes()).hexdigest()
assinatura = dataset.to_csv(index=False, float_format='%.9g').encode()
DATASET_ID = hashlib.sha256(assinatura + zip_id.encode() + sofia_sha256.encode()).hexdigest()[:16]
BASE_DIR = OUTPUT_ROOT / 'bases' / DATASET_ID
BASE_DIR.mkdir(parents=True, exist_ok=True)
dataset.to_csv(BASE_DIR / 'dataset_rotulado.csv', index=False)
metadados.to_csv(BASE_DIR / 'metadados_imagens.csv', index=False)
(BASE_DIR / 'origem.json').write_text(json.dumps({
    'dataset_id': DATASET_ID, 'zip_inventario_id': zip_id, 'sofia_sha256': sofia_sha256,
    'intervalos_frame_fim_exclusivo': INTERVALOS,
    'associacao': 'linear_por_numero_frame_entre_extremos_provisorios',
    'correspondencia_fisica_confirmada': False,
}, indent=2), encoding='utf-8')
display(dataset.groupby('experimento').agg(n=('y','size'), frame_inicial=('frame','min'),
    frame_final=('frame','max'), y_min_mm=('y','min'), y_medio_mm=('y','mean'), y_max_mm=('y','max')))
print('Base:', DATASET_ID, '|', dataset.shape, '| Salva em:', BASE_DIR)


### Visualizar rótulos e imagens

As curvas mostram o perfil fornecido e a reamostragem nos frames. Os pontos laranja
são rótulos interpolados, **não medições independentes**. A grade de imagens mostra
início, meio e fim do recorte. Esses gráficos ajudam a encontrar problemas, mas não
estabelecem sozinhos a correspondência física entre cordão e vídeo.


In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 10))
for ax, exp in zip(axes.flat, EXPERIMENTOS):
    p = perfil_sofia.loc[perfil_sofia.experimento == exp].sort_values('posicao_temporal_normalizada')
    d = dataset.loc[dataset.experimento == exp]
    ax.plot(p.posicao_temporal_normalizada, p.largura_mm, label='Perfil Sofia', color='navy')
    amostra = d.iloc[np.linspace(0, len(d)-1, 35).astype(int)]
    ax.scatter(amostra.posicao_normalizada, amostra.y, s=12, color='darkorange', label='Y nos frames')
    ax.set(title=exp, xlabel='Posição normalizada', ylabel='Largura (mm)')
    ax.grid(alpha=.25)
axes.flat[0].legend(fontsize=8)
fig.tight_layout()
fig.savefig(BASE_DIR / 'perfis_e_rotulos.png', dpi=160)
plt.show()

fig, axes = plt.subplots(9, 3, figsize=(10, 22))
for linha, exp in enumerate(EXPERIMENTOS):
    d = dataset.loc[dataset.experimento == exp]
    for ax, pos in zip(axes[linha], [0, len(d)//2, len(d)-1]):
        r = d.iloc[pos]
        ax.imshow(plt.imread(DATASET_ROOT / r.caminho_relativo))
        ax.set_title(f'{exp} | frame {r.frame} | y={r.y:.2f} mm', fontsize=9)
        ax.axis('off')
fig.tight_layout()
fig.savefig(BASE_DIR / 'imagens_inicio_meio_fim.png', dpi=120)
plt.show()


In [ ]:
if USAR_DATASET_ANTIGO:
    print("Aviso: teste ativado; carregando a base antiga adaptada.")

    CAMINHO_DATASET_OLD = (
        PROJECT_ROOT / "resultados" / "inceptionresnetv2"
        / "metadados_rotulados_old_adaptado.csv"
    )
    dataset_old = pd.read_csv(CAMINHO_DATASET_OLD)
    dataset = dataset_old.rename(columns={"indice_old": "indice_dataset"}).copy()

    # A BASE_DIR da etapa 6 pertence à base da Sofia. Cria uma pasta própria
    # para a base _old, evitando comparar ou identificar bases diferentes como iguais.
    assinatura_old = dataset.to_csv(index=False, float_format="%.9g").encode()
    DATASET_ID = hashlib.sha256(
        assinatura_old + CAMINHO_DATASET_OLD.read_bytes()
    ).hexdigest()[:16]
    BASE_DIR = OUTPUT_ROOT / "bases" / f"{DATASET_ID}_old"
    BASE_DIR.mkdir(parents=True, exist_ok=True)
    dataset.to_csv(BASE_DIR / "dataset_rotulado.csv", index=False)
    (BASE_DIR / "origem.json").write_text(json.dumps({
        "dataset_id": DATASET_ID,
        "origem": str(CAMINHO_DATASET_OLD),
        "metodologia_label": "dataset_rotulado_old_adaptado",
        "linhas": len(dataset),
        "observacao": "Base antiga escolhida explicitamente para teste; não é a base rotulada da Sofia.",
    }, indent=2), encoding="utf-8")

    print(f"Base _old: {dataset.shape} | ID: {DATASET_ID} | Salva em: {BASE_DIR}")
    display(dataset.head(10))
else:
    print("Teste _old desativado. A execução continua com a base rotulada da Sofia.")


## 4. CNN configurável e leitura em lotes

Pesos ImageNet congelados; cada arquitetura usa seu próprio pré-processamento.
EfficientNetB0 recebe pixels entre 0 e 255: a normalização já integra o modelo.
`NUMERO_FEATURES_CNN` vem da saída real da CNN. A leitura paralela conserva a ordem
dos metadados. O pequeno piloto aquece a execução antes da medição completa.


In [ ]:
CATALOGO = {
    'InceptionResNetV2': (keras.applications.InceptionResNetV2, (299,299), keras.applications.inception_resnet_v2.preprocess_input),
    'ResNet50': (keras.applications.ResNet50, (224,224), keras.applications.resnet50.preprocess_input),
    'DenseNet169': (keras.applications.DenseNet169, (224,224), keras.applications.densenet.preprocess_input),
    'EfficientNetB0': (keras.applications.EfficientNetB0, (224,224), keras.applications.efficientnet.preprocess_input),
}
construtor, tamanho_padrao, preprocessar = CATALOGO[ARQUITETURA]
TAMANHO_ENTRADA = TAMANHO_COMUM or tamanho_padrao
keras.backend.clear_session()
keras.utils.set_random_seed(SEMENTE)
extrator = construtor(weights='imagenet', include_top=False, pooling=POOLING,
                     input_shape=(*TAMANHO_ENTRADA, 3))
extrator.trainable = False
NUMERO_FEATURES_CNN = int(extrator.output_shape[-1])

def criar_dataset_imagens(tabela):
    caminhos = [str(DATASET_ROOT / c) for c in tabela.caminho_relativo]
    def carregar(caminho):
        imagem = tf.image.decode_png(tf.io.read_file(caminho), channels=3)
        imagem = tf.image.resize(imagem, TAMANHO_ENTRADA, method='bilinear', antialias=True)
        return preprocessar(tf.cast(imagem, tf.float32))
    return (tf.data.Dataset.from_tensor_slices(caminhos)
            .map(carregar, num_parallel_calls=tf.data.AUTOTUNE, deterministic=True)
            .batch(BATCH_EXTRACAO).prefetch(tf.data.AUTOTUNE))

piloto = dataset.iloc[:min(60, len(dataset))]
features_piloto = extrator.predict(criar_dataset_imagens(piloto), verbose=0)
print(f'{ARQUITETURA}: {TAMANHO_ENTRADA} → {NUMERO_FEATURES_CNN} features por imagem')


In [ ]:
# VALIDAÇÃO 3 — dimensão, valores e ordem usando inferências individuais
assert features_piloto.shape == (len(piloto), NUMERO_FEATURES_CNN)
assert np.isfinite(features_piloto).all()
for i in [0, len(piloto)-1]:
    individual = extrator.predict(criar_dataset_imagens(piloto.iloc[[i]]), verbose=0)[0]
    np.testing.assert_allclose(features_piloto[i], individual, rtol=2e-3, atol=2e-4)
print('Piloto conferido: primeira e última imagem correspondem aos respectivos vetores.')


## 5. Extração completa e custo computacional

Defina `REGERAR_FEATURES = True` para recalcular todas as features ou `False` para carregar a extração indicada em `PASTA_FEATURES_SALVAS`. Ao reutilizar, conferimos a identidade da base, a ordem das imagens, a arquitetura, a resolução, o pooling e a dimensão dos vetores. As etapas anteriores de preparação e o piloto continuam necessários.

Cada execução recebe uma pasta exclusiva para preservar os resultados anteriores. A próxima célula valida e salva a matriz `.npy`, o índice e os custos. Na reutilização, tempo, vazão, batch e dispositivo mantêm os valores da extração original; a origem fica registrada.

O tempo de extração inclui leitura, redimensionamento, pré-processamento e CNN; exclui download dos pesos e gravação no Drive. `ms/imagem` é o tempo médio amortizado em lotes. Compare custo no mesmo hardware, batch, precisão e política de resolução.


In [ ]:
# True: recalcula todas as features. False: carrega a extração indicada abaixo.

PASTA_FEATURES_SALVAS = OUTPUT_ROOT / 'execucoes' / 'InceptionResNetV2_20260928T213536_441901Z'

if REGERAR_FEATURES:
    inicio = perf_counter()
    features = extrator.predict(criar_dataset_imagens(dataset), verbose=1).astype(np.float32)
    tempo_extracao = perf_counter() - inicio
    gpus = tf.config.list_physical_devices('GPU')
    dispositivo = tf.config.experimental.get_device_details(gpus[0]).get('device_name', 'GPU') if gpus else 'CPU'
    custo_extrator = {
        'arquitetura': ARQUITETURA, 'entrada': str(TAMANHO_ENTRADA), 'pooling': POOLING,
        'features_cnn': NUMERO_FEATURES_CNN, 'parametros_extrator': int(extrator.count_params()),
        'tempo_extracao_s': tempo_extracao, 'imagens_s': len(dataset) / tempo_extracao,
        'ms_imagem_lote': 1000 * tempo_extracao / len(dataset),
        'memoria_features_MiB': features.nbytes / 1024**2, 'batch_extracao': BATCH_EXTRACAO,
        'dispositivo': dispositivo, 'dtype': str(features.dtype),
    }
else:
    # Shape sozinho não garante que os vetores pertencem às imagens atuais.
    necessarios = ['features.npy', 'indice_features.csv', 'custo_extrator.csv', 'configuracao.json']
    faltantes = [nome for nome in necessarios if not (PASTA_FEATURES_SALVAS / nome).is_file()]
    if faltantes:
        raise FileNotFoundError(f'Extração incompleta em {PASTA_FEATURES_SALVAS}: {faltantes}. Confira a pasta ou use REGERAR_FEATURES=True.')
    config_salva = json.loads((PASTA_FEATURES_SALVAS / 'configuracao.json').read_text())
    custo_extrator = pd.read_csv(PASTA_FEATURES_SALVAS / 'custo_extrator.csv').iloc[0].to_dict()
    if (config_salva['dataset_id'] != DATASET_ID
            or custo_extrator['arquitetura'] != ARQUITETURA
            or custo_extrator['entrada'] != str(TAMANHO_ENTRADA)
            or custo_extrator['pooling'] != POOLING
            or int(custo_extrator['features_cnn']) != NUMERO_FEATURES_CNN):
        raise ValueError('A extração salva não corresponde à base ou à CNN atual. Escolha outra pasta ou use REGERAR_FEATURES=True.')
    indice_salvo = pd.read_csv(PASTA_FEATURES_SALVAS / 'indice_features.csv')
    colunas_identidade = ['indice_dataset', 'experimento', 'frame', 'caminho_relativo']
    pd.testing.assert_frame_equal(
        indice_salvo[colunas_identidade].reset_index(drop=True),
        dataset[colunas_identidade].reset_index(drop=True), check_dtype=False,
    )
    features = np.load(PASTA_FEATURES_SALVAS / 'features.npy', mmap_mode='r', allow_pickle=False)
    if features.shape != (len(dataset), NUMERO_FEATURES_CNN) or features.dtype != np.float32:
        raise ValueError('Dimensão ou dtype das features salvas incompatível.')
    # Preserva tempo e dispositivo da extração original, não o tempo de carregar o arquivo.
    tempo_extracao = float(custo_extrator['tempo_extracao_s'])
    dispositivo = custo_extrator['dispositivo']
    print(f'Features reutilizadas de: {PASTA_FEATURES_SALVAS}')
    print('Tempo, vazão e dispositivo referem-se à extração original.')

# Nova pasta para os resultados desta execução, inclusive quando reutilizamos features.
EXECUCAO_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
RUN_DIR = OUTPUT_ROOT / 'execucoes' / f'{ARQUITETURA}_{EXECUCAO_ID}'
RUN_DIR.mkdir(parents=True, exist_ok=False)
custo_extrator['features_reutilizadas'] = not REGERAR_FEATURES
custo_extrator['origem_features'] = str(RUN_DIR) if REGERAR_FEATURES else custo_extrator.get('origem_features', str(PASTA_FEATURES_SALVAS))
print('Features disponíveis; execute a próxima validação antes de continuar.')


In [ ]:
# VALIDAÇÃO 4 — matriz completa antes de salvar os artefatos
assert features.shape == (len(dataset), NUMERO_FEATURES_CNN)
assert features.dtype == np.float32 and np.isfinite(features).all()
np.testing.assert_allclose(features[:len(piloto)], features_piloto, rtol=2e-3, atol=2e-4)
np.save(RUN_DIR / 'features.npy', features)
dataset.to_csv(RUN_DIR / 'indice_features.csv', index=False)
pd.DataFrame([custo_extrator]).to_csv(RUN_DIR / 'custo_extrator.csv', index=False)
display(pd.DataFrame([custo_extrator]).T.rename(columns={0:'valor'}))
print('Artefatos:', RUN_DIR)


## 6. Quatro metodologias — mesma base, mesma CNN e mesmo regressor

| Ordem | Metodologia | O que estamos avaliando? |
|---|---|---|
| 1 | Split aleatório 70/30 (abordagem Leonel) | Frames separados aleatoriamente de condições já conhecidas. |
| 2 | Blocos — teste final (70/15/15 antes dos gaps) | Trechos finais não vistos de condições conhecidas; gap de 100 frames. |
| 3 | Blocos — teste central (30% centrais) | Trechos intermediários entre regiões de treino; validação nas pontas e gaps retirados do treino. |
| 4 | LOEO | Generalização para um experimento inteiro não visto. |

**Mantemos fixos:** dados/labels, features, parâmetros de processo opcionais, Dense(32, ReLU) → Dense(1), Adam, MAE e StandardScaler ajustado somente no treino de cada divisão. Não há features temporais.

**Importante:** no protocolo 1, os mesmos 30% orientam o early stopping e são avaliados, como na abordagem de referência; portanto não são teste independente. Frames vizinhos podem ficar em conjuntos diferentes. Nos protocolos 2, 3 e 4, validação e teste são separados. A padronização é mantida: não é reprodução literal de todo o código do Leonel.

Execute a preparação, a validação e depois as quatro análises na ordem. Os resultados novos vão para uma pasta própria; arquivos antigos no Drive não são apagados. Ao final, olhe primeiro a **tabela principal**. Detalhes ficam recolhidos.

In [ ]:
# Preparação comum: as quatro análises usam exatamente a mesma matriz X e os mesmos y.
from sklearn.model_selection import train_test_split
from IPython.display import HTML
from html import escape

GAP_FRAMES = 100
ORDEM_ANALISES = ['aleatorio', 'blocos', 'blocos_central', 'loeo']
NOMES_ANALISES = {'aleatorio': '1. Aleatório 70/30', 'blocos': '2. Blocos — teste final', 'blocos_central': '3. Blocos — teste central', 'loeo': '4. LOEO'}
X = np.asarray(features, dtype=np.float32)
if USAR_PARAMETROS:
    X = np.concatenate([X, dataset[COLUNAS_PARAMETROS].to_numpy(dtype=np.float32)], axis=1)
y = dataset.y.to_numpy(dtype=np.float32)
grupos = dataset.experimento.to_numpy()
frames = dataset.frame.to_numpy()
NUMERO_ENTRADAS = X.shape[1]

# 1) Referência: os mesmos 30% são usados para early stopping e avaliação.
tr, te = train_test_split(np.arange(len(dataset)), train_size=0.7, random_state=SEMENTE)
protocolos = {'aleatorio': [{'nome': '70_30', 'treino': tr, 'validacao': te, 'teste': te}]}

# 2) Ordenamos somente os índices: não movemos features nem alteramos os rótulos.
partes = {k: [] for k in ['treino', 'validacao', 'teste']}
for exp in EXPERIMENTOS:
    idx = np.flatnonzero(grupos == exp)
    idx = idx[np.argsort(frames[idx], kind='stable')]
    a, b = int(len(idx) * 0.70), int(len(idx) * 0.85)
    tr, va, te = idx[:a], idx[a:b], idx[b:]
    va = va[frames[va] > frames[tr[-1]] + GAP_FRAMES]
    te = te[frames[te] > frames[idx[b-1]] + GAP_FRAMES]
    for nome, trecho in [('treino', tr), ('validacao', va), ('teste', te)]:
        partes[nome].append(trecho)
protocolos['blocos'] = [{'nome': '70_15_15', **{k: np.concatenate(v) for k, v in partes.items()}}]

# 3) LOEO: 7 experimentos de treino, 1 de validação e 1 de teste.
protocolos['loeo'] = []
for i, exp in enumerate(EXPERIMENTOS):
    validacao = EXPERIMENTOS[(i-1) % len(EXPERIMENTOS)]
    protocolos['loeo'].append({'nome': exp,
        'treino': np.flatnonzero((grupos != exp) & (grupos != validacao)),
        'validacao': np.flatnonzero(grupos == validacao),
        'teste': np.flatnonzero(grupos == exp)})

import numpy as np
import pandas as pd

def criar_blocos_centrais(grupos, frames, experimentos, gap):
    """Teste central 30%; validação 10% em cada ponta; gaps retirados do treino."""
    if not isinstance(gap, (int, np.integer)) or gap < 0:
        raise ValueError('GAP_FRAMES deve ser um inteiro não negativo.')
    partes = {k: [] for k in ['treino', 'validacao', 'teste']}
    resumo = []
    for exp in experimentos:
        idx = np.flatnonzero(grupos == exp)
        idx = idx[np.argsort(frames[idx], kind='stable')]
        n = len(idx)
        if n < 10 or not np.isfinite(frames[idx]).all() or not (np.diff(frames[idx]) > 0).all():
            raise ValueError(f'{exp}: frames ausentes, duplicados ou inválidos.')
        a, b, c, d = [int(n * fracao) for fracao in (0.10, 0.35, 0.65, 0.90)]
        va = np.concatenate([idx[:a], idx[d:]])
        te = idx[b:c]
        esquerda, direita = idx[a:b], idx[c:d]
        esquerda = esquerda[(frames[esquerda] > frames[idx[a-1]] + gap)
                            & (frames[esquerda] < frames[te[0]] - gap)]
        direita = direita[(frames[direita] > frames[te[-1]] + gap)
                          & (frames[direita] < frames[idx[d]] - gap)]
        if min(len(esquerda), len(direita), len(va), len(te)) < 2:
            raise ValueError(f'{exp}: poucos frames após os gaps; não reduzimos o gap automaticamente.')
        tr = np.concatenate([esquerda, direita])
        for nome, trecho in [('treino', tr), ('validacao', va), ('teste', te)]:
            partes[nome].append(trecho)
        resumo.append({'experimento': exp, 'total': n, 'treino': len(tr),
            'validacao': len(va), 'teste': len(te), 'gap_descartados': n-len(tr)-len(va)-len(te),
            'teste_frame_inicial': int(frames[te[0]]), 'teste_frame_final': int(frames[te[-1]])})
    divisao = {'nome': 'central_30_validacao_pontas_10_10',
               **{k: np.concatenate(v) for k, v in partes.items()}}
    return [divisao], pd.DataFrame(resumo)

protocolos['blocos_central'], resumo_blocos_centrais = criar_blocos_centrais(
    grupos, frames, EXPERIMENTOS, GAP_FRAMES)

# Uma nova pasta a cada rodada, sem sobrescrever resultados anteriores.
ANALISE_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
ANALISE_DIR = RUN_DIR / 'tres_metodologias' / ANALISE_ID
config_analise = {'dataset_id': DATASET_ID, 'arquitetura': ARQUITETURA,
    'parametros_processo': USAR_PARAMETROS, 'numero_entradas': NUMERO_ENTRADAS,
    'padronizacao': 'StandardScaler_fit_somente_treino', 'semente': SEMENTE,
    'gap_frames': GAP_FRAMES, 'epocas': EPOCAS, 'paciencia': PACIENCIA,
    'batch_regressao': BATCH_REGRESSAO, 'taxa_aprendizado': TAXA_APRENDIZADO,
    'regressor': 'Dense(32,relu)-Dense(1)', 'origem_features': str(RUN_DIR),
    'aleatorio_validacao_igual_teste': True,
    'blocos_central': {'versao': 1, 'teste_fracao': [0.35, 0.65],
        'validacao_fracoes': [[0.0, 0.10], [0.90, 1.0]],
        'gap_retirado_de': 'treino_nas_quatro_fronteiras',
        'ordenacao': 'numero_frame_por_experimento', 'arredondamento': 'int(n*fracao)'}}
resultados = {}
VALIDACAO_ANALISES_OK = False
print(f'Base: {DATASET_ID} | CNN: {ARQUITETURA} | Parâmetros: {USAR_PARAMETROS}')
print('Preparação concluída. Execute a validação abaixo antes dos treinos.')

In [ ]:
# VALIDAÇÃO — integridade e isolamento das divisões (sem treinar).
VALIDACAO_ANALISES_OK = False
assert X.shape == (len(dataset), NUMERO_ENTRADAS)
assert np.isfinite(X).all() and np.isfinite(y).all() and (y > 0).all()
assert np.array_equal(dataset.indice_dataset, np.arange(len(dataset)))
assert not dataset.duplicated(['experimento', 'frame']).any()
assert set(grupos) == set(EXPERIMENTOS)
# Evita usar no runtime uma base diferente daquela validada/salva com as features.
colunas = ['indice_dataset', 'experimento', 'frame', 'caminho_relativo', 'y'] + COLUNAS_PARAMETROS
for arquivo in [BASE_DIR / 'dataset_rotulado.csv', RUN_DIR / 'indice_features.csv']:
    salvo = pd.read_csv(arquivo)[colunas].reset_index(drop=True)
    try:
        pd.testing.assert_frame_equal(dataset[colunas].reset_index(drop=True),
            salvo, check_dtype=False, rtol=1e-6, atol=1e-6)
    except AssertionError as erro:
        raise ValueError(
            f'Base incompatível em {arquivo}: ativa={len(dataset)} linhas; '
            f'salva={len(salvo)} linhas. Confira a seleção da base e a origem '
            'das features antes de continuar. Nenhuma linha será descartada.'
        ) from erro
for protocolo, divisoes in protocolos.items():
    for f in divisoes:
        tr, va, te = [f[k] for k in ['treino', 'validacao', 'teste']]
        assert min(len(tr), len(va), len(te)) > 1
        assert not np.intersect1d(tr, va).size and not np.intersect1d(tr, te).size
        if protocolo == 'aleatorio':
            assert np.array_equal(va, te)  # Exceção deliberada da referência.
        else:

            assert not np.intersect1d(va, te).size
        if protocolo == 'loeo':
            assert not set(grupos[tr]) & (set(grupos[va]) | set(grupos[te]))
            assert not set(grupos[va]) & set(grupos[te])
        if protocolo == 'blocos':
            for exp in EXPERIMENTOS:
                t, v, s = [frames[idx[grupos[idx] == exp]] for idx in [tr, va, te]]
                assert min(len(t), len(v), len(s)) > 1, exp
                assert v.min() - t.max() > GAP_FRAMES and s.min() - v.max() > GAP_FRAMES, exp
        if protocolo == 'blocos_central':
            for exp in EXPERIMENTOS:
                idx = np.flatnonzero(grupos == exp)
                idx = idx[np.argsort(frames[idx], kind='stable')]
                n = len(idx)
                t, v, s = [q[grupos[q] == exp] for q in [tr, va, te]]
                assert np.array_equal(s, idx[int(.35*n):int(.65*n)]), exp
                assert np.array_equal(v, np.concatenate([idx[:int(.10*n)], idx[int(.90*n):]])), exp
                assert (frames[t] < frames[s].min()).any() and (frames[t] > frames[s].max()).any(), exp
                # Qualquer transição entre conjuntos precisa respeitar o gap em números de frame.
                juntos = np.concatenate([t, v, s])
                etiquetas = np.repeat([0, 1, 2], [len(t), len(v), len(s)])
                ordem = np.argsort(frames[juntos])
                troca = np.diff(etiquetas[ordem]) != 0
                assert (np.diff(frames[juntos][ordem])[troca] > GAP_FRAMES).all(), exp
assert np.array_equal(np.sort(np.concatenate([f['teste'] for f in protocolos['loeo']])), np.arange(len(dataset)))
ANALISE_DIR.mkdir(parents=True, exist_ok=True)
(ANALISE_DIR / 'configuracao.json').write_text(json.dumps(config_analise, indent=2), encoding='utf-8')
# Metadados da extração necessários à opção REGERAR_FEATURES=False.
if not (RUN_DIR / 'configuracao.json').exists():
    (RUN_DIR / 'configuracao.json').write_text(json.dumps(config_analise, indent=2), encoding='utf-8')
display(resumo_blocos_centrais)
resumo_blocos_centrais.to_csv(ANALISE_DIR / 'divisao_blocos_centrais.csv', index=False)
VALIDACAO_ANALISES_OK = True
print('Validação OK: mesma base/features nas quatro metodologias; divisões conferidas.')

In [ ]:
# Rotina compartilhada: evita repetir o mesmo treinamento nas quatro metodologias.
def metricas(real, previsto):
    real, previsto = np.asarray(real).ravel(), np.asarray(previsto).ravel()
    return {'MAE_mm': float(mean_absolute_error(real, previsto)),
        'MAPE_pct': float(100 * np.mean(np.abs((real-previsto)/real))),
        'RMSE_mm': float(np.sqrt(mean_squared_error(real, previsto))),
        'R2': float(r2_score(real, previsto)) if np.var(real) > 0 else np.nan}

def construir_regressor(n_entradas):
    modelo = keras.Sequential([keras.layers.Input(shape=(n_entradas,)),
        keras.layers.Dense(32, activation='relu'), keras.layers.Dense(1)])
    modelo.compile(optimizer=keras.optimizers.Adam(learning_rate=TAXA_APRENDIZADO),
        loss='mean_absolute_error')
    return modelo

def executar_protocolo(chave):
    assert VALIDACAO_ANALISES_OK, 'Execute a validação antes de treinar.'
    pasta = ANALISE_DIR / (chave + '_' + datetime.now(timezone.utc).strftime('%H%M%S_%f'))
    pasta.mkdir(parents=True, exist_ok=False)
    partes, diagnostico, tempos = [], [], []
    for numero, f in enumerate(protocolos[chave], start=1):
        tr, va, te = [f[k] for k in ['treino', 'validacao', 'teste']]
        scaler = StandardScaler().fit(X[tr])
        x_tr, x_va, x_te = [scaler.transform(X[idx]).astype(np.float32) for idx in [tr, va, te]]
        keras.backend.clear_session()
        keras.utils.set_random_seed(SEMENTE + numero if chave == 'loeo' else SEMENTE)
        modelo = construir_regressor(NUMERO_ENTRADAS)
        parada = keras.callbacks.EarlyStopping(monitor='val_loss', patience=PACIENCIA, restore_best_weights=True)
        inicio = perf_counter()
        historico = modelo.fit(x_tr, y[tr], validation_data=(x_va, y[va]), epochs=EPOCAS,
            batch_size=BATCH_REGRESSAO, callbacks=[parada], verbose=0, shuffle=True)
        tempos.append(perf_counter() - inicio)
        previsto = modelo.predict(x_te, batch_size=BATCH_REGRESSAO, verbose=0).ravel()
        assert len(previsto) == len(te) and np.isfinite(previsto).all()
        # Baseline: média do TREINO do mesmo experimento quando ele é conhecido;
        # no LOEO, usa a média global do treino, pois o experimento de teste é novo.
        medias = pd.DataFrame({'experimento': grupos[tr], 'y': y[tr]}).groupby('experimento').y.mean()
        baseline = pd.Series(grupos[te]).map(medias).fillna(float(y[tr].mean())).to_numpy()
        p = dataset.iloc[te][['indice_dataset', 'experimento', 'frame', 'y']].copy()
        p['y_predito'], p['y_baseline'], p['fold'] = previsto, baseline, f['nome']
        partes.append(p)
        # Detalhes são salvos, não despejados na saída principal.
        for conjunto, idx, entrada in [('treino', tr, x_tr), ('validacao', va, x_va), ('teste', te, x_te)]:
            pred = previsto if conjunto == 'teste' else modelo.predict(entrada, batch_size=BATCH_REGRESSAO, verbose=0).ravel()
            diagnostico.append({'fold': f['nome'], 'conjunto': conjunto, 'n': len(idx), **metricas(y[idx], pred)})
        prefixo = pasta / f['nome']
        modelo.save(str(prefixo) + '.keras')
        np.savez(str(prefixo) + '_scaler.npz', media=scaler.mean_, escala=scaler.scale_)
        np.savez(str(prefixo) + '_indices.npz', treino=tr, validacao=va, teste=te)
        pd.DataFrame(historico.history).to_csv(str(prefixo) + '_historico.csv', index=False)
        pd.concat(partes, ignore_index=True).to_csv(pasta / 'predicoes.csv', index=False)
        pd.DataFrame(diagnostico).to_csv(pasta / 'treino_validacao_teste.csv', index=False)
        print(f'{NOMES_ANALISES[chave]} | divisão {numero}/{len(protocolos[chave])} concluída.', flush=True)
    predicoes = pd.concat(partes, ignore_index=True).sort_values('indice_dataset')
    por_exp = pd.DataFrame([{'experimento': exp, **metricas(d.y, d.y_predito),
        'MAE_baseline_mm': metricas(d.y, d.y_baseline)['MAE_mm']}
        for exp, d in predicoes.groupby('experimento', sort=True)])
    resumo = {'metodologia': NOMES_ANALISES[chave], **metricas(predicoes.y, predicoes.y_predito),
        'MAE_baseline_mm': metricas(predicoes.y, predicoes.y_baseline)['MAE_mm'],
        'experimentos_R2_positivo': int((por_exp.R2 > 0).sum()), 'n_avaliado': len(predicoes),
        'tempo_regressor_s': sum(tempos), 'arquitetura': ARQUITETURA,
        'parametros_processo': USAR_PARAMETROS, 'dataset_id': DATASET_ID}
    por_exp.to_csv(pasta / 'metricas_experimentos.csv', index=False)
    pd.DataFrame([resumo]).to_csv(pasta / 'resumo.csv', index=False)
    (pasta / 'configuracao.json').write_text(json.dumps({**config_analise, 'protocolo': chave}, indent=2), encoding='utf-8')
    print(f'Concluído — MAE: {resumo["MAE_mm"]:.4f} mm | R² global: {resumo["R2"]:.4f}')
    return {'resumo': resumo, 'por_experimento': por_exp, 'predicoes': predicoes,
        'diagnostico': pd.DataFrame(diagnostico), 'pasta': pasta}

print('Rotina pronta. Execute primeiro o split aleatório abaixo.')

## 7. Metodologia 1 — split aleatório 70/30

**Pergunta:** o modelo prevê frames separados aleatoriamente dentro de condições conhecidas?

70% para treino e 30% para validação/avaliação, com semente fixa. O conjunto de avaliação participa do early stopping e pode conter vizinhos dos frames de treino: trate os números como **referência otimista**, não como generalização para um experimento novo.

In [ ]:
# Treina um regressor com split aleatório; não extrai novamente as features.
resultados['aleatorio'] = executar_protocolo('aleatorio')

## 8. Metodologia 2 — blocos com teste final

**Pergunta:** o modelo prevê trechos finais não vistos de condições conhecidas?

Dentro de cada experimento: primeiros 70% para treino, próximos 15% para validação e últimos 15% para teste. Removemos dos inícios da validação e do teste os frames que não respeitam um **gap de 100 números de frame**. As frações efetivas ficam menores após o gap. Um único regressor reúne os blocos de treino de todos os experimentos.

Validação e teste são independentes entre si. O gap reduz proximidade, mas não comprova ausência de toda dependência temporal.

In [ ]:
# Usa os mesmos X e y, mudando somente o protocolo de divisão.
resultados['blocos'] = executar_protocolo('blocos')

## 8B. Metodologia 3 — blocos com teste central

**Pergunta:** o modelo prevê um trecho central não visto usando trechos anteriores e posteriores do mesmo experimento?

Em cada experimento, ordenamos apenas os índices por número de frame, preservando a associação com X e y. As fronteiras são calculadas com int(n × fração):

| Posição na sequência | Uso antes dos gaps |
|---|---|
| 0–10% | Validação |
| 10–35% | Treino |
| 35–65% | **Teste central: aproximadamente 30%** |
| 65–90% | Treino |
| 90–100% | Validação |

Retiramos do **treino** os frames a até GAP_FRAMES de cada fronteira com validação/teste. Validação e teste ficam intactos; o treino efetivo é menor que 50%. Se algum trecho ficar insuficiente, a preparação interrompe sem reduzir o gap automaticamente. Um regressor reúne os trechos de todos os experimentos; o StandardScaler é ajustado somente no treino e o EarlyStopping usa somente a validação.

**Não há nova interpolação de y nem extração de features.** “Interpolação” aqui descreve a posição temporal do teste, não garante que suas larguras estejam na faixa do treino e não representa previsão futura em tempo real. O protocolo anterior permanece 70/15/15 antes dos gaps; esta comparação muda também o tamanho/localização dos conjuntos, não isola apenas a posição do teste.

Execute primeiro a preparação, validação e rotina compartilhada da seção 6, depois esta célula. Os resultados recebem a chave **blocos_central**, separada dos resultados antigos. Na seção 11, execute novamente a leitura do histórico e os gráficos após o treinamento.

In [ ]:
# Reutiliza X/y; treina somente o regressor do teste central.
resultados['blocos_central'] = executar_protocolo('blocos_central')

## 9. Metodologia 4 — LOEO (Leave-One-Experiment-Out)

**Pergunta:** o modelo funciona em um experimento que nunca viu?

Em cada uma das nove divisões, deixamos um experimento inteiro para teste e o anterior da lista para validação; os sete restantes entram no treino. Cada experimento é testado uma vez. O StandardScaler e o regressor são reajustados do zero em cada divisão, sem usar o teste.

In [ ]:
# Treina nove regressores; cada experimento é avaliado fora do treino.
resultados['loeo'] = executar_protocolo('loeo')

## 10. Resultado principal

MAE (erro em mm), RMSE (penaliza erros maiores), MAPE (erro percentual) e R² (quanto explica a variação). Menores erros são melhores; R² maior é melhor. R² negativo fica destacado em vermelho.

As quatro linhas respondem a perguntas diferentes e avaliam conjuntos diferentes: **não são um ranking para escolher a divisão que dá o maior R²**. O split aleatório é a referência otimista; blocos e LOEO testam generalizações mais exigentes. O R² global pode esconder erros dentro dos experimentos — confira o painel recolhido se necessário.

Para comparar CNNs, compare a mesma metodologia, base, uso de parâmetros e configuração. Custo de extração e detalhes por experimento ficam recolhidos para não sobrecarregar a leitura.

In [ ]:
# Quadro principal: somente as quatro métricas essenciais, uma linha por metodologia.
faltantes = [k for k in ORDEM_ANALISES if k not in resultados]
if faltantes:
    raise ValueError('Execute antes: ' + ', '.join(faltantes))
comparacao = pd.DataFrame([resultados[k]['resumo'] for k in ORDEM_ANALISES])
comparacao.to_csv(ANALISE_DIR / 'comparacao_metodologias.csv', index=False)
principal = comparacao.set_index('metodologia')[['MAE_mm', 'MAPE_pct', 'RMSE_mm', 'R2']]
principal.columns = ['MAE (mm) ↓', 'MAPE (%) ↓', 'RMSE (mm) ↓', 'R² global ↑']
contexto = f'{ARQUITETURA} | {"CNN + parâmetros" if USAR_PARAMETROS else "Somente CNN"} | Base {DATASET_ID}'
display(HTML('<h3>Resultado principal</h3><p>' + escape(contexto) + '</p>'))
estilo = (principal.style.format('{:.4f}')
    .set_properties(**{'text-align': 'center', 'padding': '12px', 'font-size': '15px'})
    .set_properties(subset=['MAE (mm) ↓'], **{'font-weight': 'bold', 'background-color': '#eaf2ff', 'color': '#12335b'})
    .set_table_styles([{'selector': 'th', 'props': [('background-color', '#17324d'), ('color', 'white'), ('padding', '12px')]}])
    .apply(lambda s: ['background-color:#fff0f0;color:#a31515;font-weight:bold' if v < 0 else 'font-weight:bold'
        for v in s], subset=['R² global ↑']))
display(estilo)
display(HTML('<p><b>Atenção:</b> aleatório = avaliação reutilizada no early stopping; blocos e LOEO = teste separado. '
    'Os conjuntos avaliados diferem. R² global não substitui a análise por experimento.</p>'))

# Detalhes recolhidos: não competem visualmente com a tabela principal.
for chave in ORDEM_ANALISES:
    r = resultados[chave]
    d = r['por_experimento'].copy()
    d.columns = ['Experimento', 'MAE (mm)', 'MAPE (%)', 'RMSE (mm)', 'R²', 'MAE baseline (mm)']
    positivos = int((d['R²'] > 0).sum())
    titulo = f'{NOMES_ANALISES[chave]} — detalhes por experimento ({positivos}/{len(d)} com R² positivo)'
    display(HTML('<details><summary><b>' + escape(titulo) + '</b></summary>' +
        d.to_html(index=False, float_format=lambda v: f'{v:.4f}') +
        '<p>Baseline: média do treino do mesmo experimento, quando conhecido; média global do treino no LOEO.</p></details>'))

campos_custo = ['arquitetura', 'features_cnn', 'parametros_extrator', 'tempo_extracao_s',
    'imagens_s', 'ms_imagem_lote', 'memoria_features_MiB', 'batch_extracao', 'dispositivo']
custo = pd.DataFrame({'Parâmetro': campos_custo, 'Valor': [custo_extrator.get(k, np.nan) for k in campos_custo]})
display(HTML('<details><summary><b>Custo do extrator e eficiência computacional</b></summary>' +
    custo.to_html(index=False, float_format=lambda v: f'{v:.3f}') +
    '<p>Compare imagens/s (maior é melhor) e ms/imagem em lote (menor é melhor) no mesmo hardware e batch. '
    'Incluem leitura, pré-processamento e inferência; não são latência isolada da CNN. '
    'Quando há reuso, os tempos pertencem à extração original.</p></details>'))
pd.DataFrame([custo_extrator]).to_csv(ANALISE_DIR / 'custo_extrator.csv', index=False)
print('Resultados e detalhes salvos em:', ANALISE_DIR)

## 11 — Comparação entre execuções salvas

Execute apenas estas duas células para atualizar o histórico, sem treinar novamente. O painel mostra **CNN, parâmetros=True/False e data da execução**. Escolha metodologia, base, parâmetros e CNN; MAE, MAPE e RMSE devem diminuir, enquanto R² deve aumentar.

A tabela **Erro × variabilidade da largura** acompanha os mesmos filtros. Ela lê o arquivo de previsões de cada análise e compara **desvio-padrão de y, RMSE e R² global**, destacando a razão RMSE/desvio. Não depende dos modelos nem das previsões na memória. Arquivos ausentes ou incompatíveis são sinalizados, sem usar dados de outra execução.

As bases são separadas pelo conteúdo do índice de imagens e rótulos (precisão de seis casas decimais), não apenas pelo nome. São lidos os resumos finais antigos e os novos resumos das quatro metodologias, sem duplicar arquivos parciais. **Este é um histórico, não um ranking controlado**: versões antigas podem ter configurações declaradas desatualizadas. Confira os detalhes antes de atribuir diferenças à CNN ou aos parâmetros. R² exibido é global e não substitui a análise por experimento.

**Blocos — teste final** identifica o protocolo antigo. **Blocos — teste central** é separado nos filtros, na tabela de variabilidade e no gráfico de custo × erro. Só aparecerão valores após concluir e salvar essa nova análise; resultados antigos não são convertidos em teste central.

In [ ]:
# HISTÓRICO — somente leitura dos resultados concluídos; não treina modelos.
from pathlib import Path as HistPath
import json as hist_json
import hashlib as hist_hash
import pandas as hist_pd
import numpy as hist_np
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
PASTA_EXECUCOES = HistPath('/content/drive/MyDrive/TCC/resultados/cnn_new_estatico_v2/execucoes')
METRICAS_HISTORICO = ['MAE_mm', 'MAPE_pct', 'RMSE_mm', 'R2']

def ler_config_historico(pasta):
    arquivo = pasta / 'configuracao.json'
    return hist_json.loads(arquivo.read_text()) if arquivo.is_file() else {}

def identificar_base_historico(pasta):
    # Confere o conteúdo: alguns arquivos antigos reutilizaram o mesmo dataset_id.
    indice = hist_pd.read_csv(pasta / 'indice_features.csv')
    colunas = ['experimento', 'frame', 'y', 'current', 'nozzle_speed', 'mass_flow_rate']
    base = indice[colunas].sort_values(['experimento', 'frame']).reset_index(drop=True)
    assinatura = base.to_csv(index=False, float_format='%.6f').encode()
    return hist_hash.sha256(assinatura).hexdigest()[:10], len(base)

linhas_historico, avisos_historico = [], []
for pasta in sorted(PASTA_EXECUCOES.iterdir()):
    if not pasta.is_dir():
        continue
    arquivos = list(pasta.glob('resumo_final.csv'))
    arquivos += sorted(pasta.glob('tres_metodologias/*/*/resumo.csv'))
    if not arquivos:
        avisos_historico.append(f'{pasta.name}: sem resumo final reconhecido.')
        continue
    try:
        base_hash, n_base = identificar_base_historico(pasta)
        config_raiz = ler_config_historico(pasta)
    except Exception as erro:
        avisos_historico.append(f'{pasta.name}: {erro}')
        continue
    for arquivo in arquivos:
        try:
            config = {**config_raiz, **ler_config_historico(arquivo.parent)}
            resumo = hist_pd.read_csv(arquivo)
            if len(resumo) != 1:
                raise ValueError('O resumo precisa conter exatamente uma análise.')
            r = resumo.iloc[0].to_dict()
            valores = [float(r[m]) for m in METRICAS_HISTORICO]
            if not hist_np.isfinite(valores).all():
                raise ValueError('Métricas ausentes ou não finitas.')
            protocolo = str(config.get('protocolo', r.get('protocolo', ''))).lower()
            metodo = ('LOEO' if 'loeo' in protocolo else 'Blocos — teste central' if 'blocos_central' in protocolo
                      else 'Blocos — teste final' if 'bloco' in protocolo
                      else 'Aleatório 70/30' if 'aleatorio' in protocolo else None)
            if metodo is None:
                raise ValueError(f'Protocolo não reconhecido: {protocolo}')
            parametro = str(r.get('parametros_processo', config.get('parametros_processo', '?'))).lower()
            parametro = {'true': 'True', 'false': 'False', '1': 'True', '0': 'False'}.get(parametro, 'não informado')
            cnn = str(r.get('arquitetura', config.get('arquitetura', 'não informada')))
            data_id = pasta.name.split('_', 1)[-1]
            data = hist_pd.to_datetime(data_id, format='%Y%m%dT%H%M%S_%fZ', utc=True)
            linhas_historico.append({**dict(zip(METRICAS_HISTORICO, valores)),
                'metodologia': metodo, 'CNN': cnn, 'parametros': parametro,
                'base': f'{base_hash} | {n_base:,} imagens', 'data_utc': data,
                'execucao': pasta.name, 'dataset_id_declarado': config.get('dataset_id', ''),
                'padronizacao_declarada': config.get('padronizacao', 'não informada'),
                'configuracao': hist_json.dumps(config, ensure_ascii=False), 'arquivo': str(arquivo)})
        except Exception as erro:
            avisos_historico.append(f'{arquivo.relative_to(PASTA_EXECUCOES)}: {erro}')

historico_execucoes = hist_pd.DataFrame(linhas_historico)
if historico_execucoes.empty:
    raise ValueError('Nenhum resultado final válido encontrado. Confira PASTA_EXECUCOES e avisos_historico.')
historico_execucoes = historico_execucoes.sort_values(['data_utc', 'arquivo']).reset_index(drop=True)
print(f'{len(historico_execucoes)} análises concluídas | {historico_execucoes.execucao.nunique()} execuções | '
      f'{historico_execucoes.base.nunique()} bases identificadas pelo conteúdo.')
print(f'{len(avisos_historico)} pasta(s)/arquivo(s) ignorados; detalhes disponíveis em avisos_historico.')


In [ ]:
# PAINEL — filtros e um gráfico legível por vez.
import matplotlib.pyplot as hist_plt
import ipywidgets as hist_widgets
from IPython.display import display as hist_display, HTML as HistHTML, clear_output as hist_clear
from html import escape as hist_escape

# Calcula somente a partir das previsões da própria análise, nunca do dataset em memória.
def mostrar_variabilidade_salva(d, rotulos):
    linhas, avisos = [], []
    for (_, r), rotulo in zip(d.iterrows(), rotulos):
        arquivo = HistPath(r['arquivo']).parent / 'predicoes.csv'
        try:
            if not arquivo.is_file():
                raise ValueError('predicoes.csv não encontrado na pasta desta análise')
            p = hist_pd.read_csv(arquivo, usecols=['y', 'y_predito'])
            valores = p[['y', 'y_predito']].to_numpy(dtype=float)
            if len(p) < 2 or not hist_np.isfinite(valores).all():
                raise ValueError('previsões insuficientes ou não finitas')
            real, previsto = valores.T
            rmse = float(hist_np.sqrt(hist_np.mean((real-previsto)**2)))
            desvio = float(hist_np.std(real, ddof=0))
            razao = rmse/desvio if desvio > 0 else hist_np.nan
            r2 = 1-razao**2 if desvio > 0 else hist_np.nan
            mae = float(hist_np.mean(hist_np.abs(real-previsto)))
            if not hist_np.allclose([mae, rmse], [r['MAE_mm'], r['RMSE_mm']], rtol=1e-4, atol=1e-5):
                raise ValueError('previsões não reproduzem MAE/RMSE do resumo salvo')
            if desvio > 0 and not hist_np.isclose(r2, r['R2'], rtol=1e-4, atol=1e-5):
                raise ValueError('R² recalculado difere do resumo salvo')
            linhas.append({'Execução': rotulo, 'N': len(real), 'Desvio y (mm)': desvio,
                'RMSE (mm)': rmse, 'RMSE / desvio': razao, 'R² global': r2})
        except Exception as erro:
            avisos.append(f'{rotulo}: {erro}. Arquivo: {arquivo}')
    hist_display(HistHTML('<h3>Erro × variabilidade da largura</h3>'))
    if linhas:
        tabela_diag = hist_pd.DataFrame(linhas)
        hist_display(tabela_diag.style.hide(axis='index').format({
            'Desvio y (mm)': '{:.4f}', 'RMSE (mm)': '{:.4f}',
            'RMSE / desvio': '{:.2f}', 'R² global': '{:.4f}'}, na_rep='Indefinido (y constante)')
            .set_properties(subset=['RMSE / desvio'], **{
                'font-weight': 'bold', 'background-color': '#fff2cc', 'color': '#222222'}))
        print('Razão < 1: R² positivo | = 1: R² zero | > 1: R² negativo (antes do arredondamento).')
        print('Desvio populacional (ddof=0) e RMSE usam os mesmos exemplos avaliados de cada análise.')
        print('A média real do conjunto é só a referência matemática do R², não uma entrada do modelo.')
        print('Isso explica o R², mas não prova a causa do erro. Protocolos podem avaliar exemplos diferentes.')
    if avisos:
        print(f'{len(avisos)} análise(s) sem diagnóstico verificável; gráficos históricos preservados.')
        hist_display(HistHTML('<details><summary>Diagnósticos indisponíveis — arquivos e motivos</summary><pre>'
            + hist_escape('\n'.join(avisos)) + '</pre></details>'))

ultima_base = historico_execucoes.iloc[-1]['base']
seletor_metodo = hist_widgets.Dropdown(options=['Aleatório 70/30', 'Blocos — teste final', 'Blocos — teste central', 'LOEO'],
    value='LOEO', description='Método:', layout=hist_widgets.Layout(width='290px'))
seletor_base = hist_widgets.Dropdown(options=historico_execucoes.base.drop_duplicates().tolist(),
    value=ultima_base, description='Base:', layout=hist_widgets.Layout(width='370px'))
seletor_param = hist_widgets.Dropdown(options=['Todos', 'False', 'True'], value='Todos',
    description='Parâmetros:', layout=hist_widgets.Layout(width='240px'))
seletor_metrica = hist_widgets.Dropdown(options=[('MAE (mm) ↓','MAE_mm'), ('MAPE (%) ↓','MAPE_pct'),
    ('RMSE (mm) ↓','RMSE_mm'), ('R² global ↑','R2')], description='Métrica:',
    layout=hist_widgets.Layout(width='260px'))
seletor_cnn = hist_widgets.Dropdown(options=['Todas'] + sorted(historico_execucoes.CNN.unique().tolist()),
    value='Todas', description='CNN:', layout=hist_widgets.Layout(width='300px'))
saida_historico = hist_widgets.Output()

def desenhar_historico(_=None):
    with saida_historico:
        hist_clear(wait=True)
        d = historico_execucoes.loc[(historico_execucoes.metodologia == seletor_metodo.value)
                                   & (historico_execucoes.base == seletor_base.value)].copy()
        if seletor_param.value != 'Todos':
            d = d.loc[d.parametros == seletor_param.value]
        if seletor_cnn.value != 'Todas':
            d = d.loc[d.CNN == seletor_cnn.value]
        if d.empty:
            print('Nenhuma execução concluída para estes filtros.')
            return
        d = d.reset_index(drop=True)
        datas = d.data_utc.dt.tz_convert('America/Sao_Paulo').dt.strftime('%d/%m %H:%M:%S')
        rotulos = [f'{cnn} | parâmetros={p} | {data}' for cnn,p,data in zip(d.CNN,d.parametros,datas)]
        metrica = seletor_metrica.value
        titulo = {'MAE_mm':'MAE (mm) — menor é melhor', 'MAPE_pct':'MAPE (%) — menor é melhor',
                  'RMSE_mm':'RMSE (mm) — menor é melhor', 'R2':'R² global — maior é melhor'}[metrica]
        cores = [{'False':'#386cb0','True':'#d17b21'}.get(p,'#888888') for p in d.parametros]
        fig, ax = hist_plt.subplots(figsize=(12, max(3, 0.47*len(d)+1.5)))
        barras = ax.barh(range(len(d)), d[metrica], color=cores, height=0.62)
        ax.set_yticks(range(len(d)), rotulos, fontsize=9)
        ax.invert_yaxis()
        ax.bar_label(barras, labels=[f'{v:.4f}' for v in d[metrica]], padding=5, fontsize=10)
        ax.axvline(0, color='#555555', linewidth=0.8)
        ax.set_title(f'{seletor_metodo.value} | {titulo}', loc='left', fontweight='bold', pad=15)
        ax.set_xlabel('Mesma base selecionada; datas no horário de Brasília')
        ax.xaxis.grid(True, alpha=0.18)
        ax.set_axisbelow(True)
        ax.margins(x=0.22)
        for borda in ['top','right','left']:
            ax.spines[borda].set_visible(False)
        hist_plt.tight_layout()
        hist_plt.show()
        print(f'{len(d)} análise(s). Azul: parâmetros=False | Laranja: parâmetros=True.')
        print('Histórico exploratório: confira configurações e divisões antes de comparar modelos.')
        if seletor_metodo.value == 'Aleatório 70/30':
            print('Referência: validação e teste compartilham o mesmo conjunto; resultado otimista.')
        mostrar_variabilidade_salva(d, rotulos)
        tabela = d[['CNN','parametros'] + METRICAS_HISTORICO + ['execucao','padronizacao_declarada','arquivo','configuracao']]
        hist_display(HistHTML('<details><summary>Ver métricas, configurações declaradas e arquivos de origem</summary>'
            + tabela.to_html(index=False, float_format=lambda v: f'{v:.4f}', escape=True) + '</details>'))
        if avisos_historico:
            hist_display(HistHTML('<details><summary>Arquivos ignorados</summary><pre>'
                + hist_escape('\n'.join(avisos_historico)) + '</pre></details>'))

for seletor in [seletor_metodo, seletor_base, seletor_param, seletor_metrica, seletor_cnn]:
    seletor.observe(desenhar_historico, names='value')
hist_display(hist_widgets.VBox([hist_widgets.HBox([seletor_metodo, seletor_base]),
    hist_widgets.HBox([seletor_param, seletor_metrica]), seletor_cnn, saida_historico]))
desenhar_historico()



### Tempo × erro — todas as execuções

Cada ponto representa uma execução em uma metodologia. **Esquerda e abaixo** indicam menor tempo e menor MAE. A **cor mostra o R²**; círculos representam `parâmetros=False` e losangos, `True`. Passe o mouse sobre os pontos para ver a **CNN, a data, a base e o dispositivo**. Os códigos E01, E02… identificam as execuções na tabela recolhida abaixo.

O eixo X soma a extração registrada ao treino da metodologia e usa escala logarítmica para comportar CPU e GPU. Em execuções com cache, considera o tempo da extração original. Todas as análises concluídas com tempos válidos aparecem inicialmente; use o menu para selecionar uma metodologia. Execute a célula de leitura do histórico da seção 11 antes desta célula.

In [ ]:
# DISPERSÃO — custo computacional registrado × erro, com todas as análises concluídas.
# Execute antes a célula HISTÓRICO da seção 11. Não há extração nem treino nesta célula.
import plotly.graph_objects as custo_go
from IPython.display import display as custo_display, HTML as CustoHTML
from html import escape as custo_escape

linhas_custo, avisos_custo = [], []
for _, registro in historico_execucoes.iterrows():
    try:
        pasta_run = PASTA_EXECUCOES / registro['execucao']
        resumo = hist_pd.read_csv(registro['arquivo']).iloc[0].to_dict()
        arquivo_custo = pasta_run / 'custo_extrator.csv'
        custo = hist_pd.read_csv(arquivo_custo).iloc[0].to_dict() if arquivo_custo.is_file() else resumo
        extracao = float(custo.get('tempo_extracao_s', hist_np.nan))
        treino = float(resumo.get('tempo_regressor_s', hist_np.nan))
        if not (hist_np.isfinite([extracao, treino]).all() and extracao > 0 and treino >= 0):
            raise ValueError('Tempo de extração ou treino ausente/inválido.')
        reutilizadas = str(custo.get('features_reutilizadas', 'não registrado'))
        linhas_custo.append({**registro.to_dict(), 'tempo_extracao_s': extracao,
            'tempo_regressor_s': treino, 'tempo_referencia_s': extracao + treino,
            'dispositivo_extracao': str(custo.get('dispositivo', 'não informado')),
            'features_reutilizadas': reutilizadas,
            'origem_features': str(custo.get('origem_features', 'não registrada'))})
    except Exception as erro:
        avisos_custo.append(f"{registro['execucao']} | {registro['metodologia']}: {erro}")

comparacao_custo = hist_pd.DataFrame(linhas_custo)
if comparacao_custo.empty:
    raise ValueError('Não há análises com tempos válidos. Consulte avisos_custo.')
comparacao_custo = comparacao_custo.sort_values(['data_utc','arquivo']).reset_index(drop=True)
# O mesmo ID identifica uma execução, mesmo quando ela tem três metodologias.
ids_execucao = {nome: f'E{i:02d}' for i, nome in enumerate(comparacao_custo.execucao.unique(), 1)}
comparacao_custo['id_grafico'] = comparacao_custo.execucao.map(ids_execucao)
comparacao_custo['data_brasilia'] = comparacao_custo.data_utc.dt.tz_convert('America/Sao_Paulo').dt.strftime('%d/%m/%Y %H:%M:%S')
comparacao_custo['rotulo'] = comparacao_custo['id_grafico'] + ' · ' + comparacao_custo['CNN']

fig_custo = custo_go.Figure()
metodos_custo = ['Aleatório 70/30', 'Blocos — teste final', 'Blocos — teste central', 'LOEO']
metodos_traces = []
campos_hover = ['id_grafico','CNN','parametros','data_brasilia','metodologia','base',
                'dispositivo_extracao','tempo_extracao_s','tempo_regressor_s','R2',
                'features_reutilizadas','origem_features']
for metodo in metodos_custo:
    for parametro, simbolo in [('False','circle'), ('True','diamond'), ('não informado','x')]:
        d = comparacao_custo.loc[(comparacao_custo.metodologia == metodo) & (comparacao_custo.parametros == parametro)]
        if d.empty: continue
        fig_custo.add_trace(custo_go.Scatter(
            x=d.tempo_referencia_s, y=d.MAE_mm, mode='markers+text',
            name=f'{metodo} | parâmetros={parametro}',
            text=d.id_grafico, textposition='top center', textfont=dict(size=10),
            customdata=d[campos_hover].to_numpy(),
            marker=dict(size=13, symbol=simbolo, color=d.R2, coloraxis='coloraxis',
                        line=dict(width=1, color='#333333'), opacity=0.9),
            hovertemplate='<b>%{customdata[0]} · %{customdata[1]}</b><br>'
                'Parâmetros=%{customdata[2]} | %{customdata[3]}<br>%{customdata[4]}<br>'
                'Base: %{customdata[5]}<br>Dispositivo da extração: %{customdata[6]}<br>'
                '<b>MAE: %{y:.4f} mm | R²: %{customdata[9]:.4f}</b><br>'
                'Extração + treino: %{x:.1f} s<br>Extração registrada: %{customdata[7]:.1f} s<br>'
                'Treino: %{customdata[8]:.1f} s<br>Features reutilizadas: %{customdata[10]}<br>'
                'Origem das features: %{customdata[11]}<extra></extra>'))
        metodos_traces.append(metodo)

r2_min, r2_max = comparacao_custo.R2.min(), comparacao_custo.R2.max()
if r2_min == r2_max: r2_min, r2_max = r2_min - 0.01, r2_max + 0.01
botoes = [dict(label='Todas as metodologias', method='update', args=[{'visible':[True]*len(metodos_traces)}])]
botoes += [dict(label=m, method='update', args=[{'visible':[t == m for t in metodos_traces]}])
           for m in metodos_custo if m in metodos_traces]
fig_custo.update_layout(
    title=dict(text='Custo computacional × erro das execuções<br><sup>Menor tempo e menor MAE: pontos mais à esquerda e abaixo</sup>', x=0.02),
    template='plotly_white', height=650, margin=dict(l=75,r=100,t=125,b=140),
    xaxis=dict(title='Extração + treino (segundos, escala logarítmica)', type='log'),
    yaxis=dict(title='MAE (mm) — menor é melhor', rangemode='tozero'),
    coloraxis=dict(colorscale='Viridis', cmin=r2_min, cmax=r2_max,
                   colorbar=dict(title='R² global', len=0.75)),
    legend=dict(orientation='h', y=-0.23, x=0, title=''),
    updatemenus=[dict(buttons=botoes, x=0, y=1.14, xanchor='left', yanchor='top', showactive=True)],
    hoverlabel=dict(bgcolor='white', font_size=12))



# Seletor de execução — filtra os pontos sem depender de ipywidgets.
# Mantém o menu de metodologias já existente.

def dados_da_execucao(id_selecionado=None):
    atualizacao = {
        'x': [],
        'y': [],
        'text': [],
        'customdata': [],
        'marker.color': [],
    }

    for trace in fig_custo.data:
        indices = [
            i for i, dados in enumerate(trace.customdata)
            if id_selecionado is None or dados[0] == id_selecionado
        ]

        atualizacao['x'].append([trace.x[i] for i in indices])
        atualizacao['y'].append([trace.y[i] for i in indices])
        atualizacao['text'].append([trace.text[i] for i in indices])
        atualizacao['customdata'].append(
            [list(trace.customdata[i]) for i in indices]
        )
        atualizacao['marker.color'].append(
            [trace.marker.color[i] for i in indices]
        )

    return atualizacao


botoes_execucao = [
    dict(
        label='Todas as execuções',
        method='restyle',
        args=[dados_da_execucao()],
    )
]

for _, registro in comparacao_custo.drop_duplicates('execucao').iterrows():
    botoes_execucao.append(dict(
        label=(
            f"{registro['id_grafico']} | {registro['CNN']} | "
            f"parâmetros={registro['parametros']} | "
            f"{registro['data_brasilia']}"
        ),
        method='restyle',
        args=[dados_da_execucao(registro['id_grafico'])],
    ))

menus_existentes = [
    menu.to_plotly_json() for menu in fig_custo.layout.updatemenus
]

fig_custo.update_layout(
    margin=dict(l=75, r=100, t=190, b=140),
    updatemenus=menus_existentes + [
        dict(
            buttons=botoes_execucao,
            direction='down',
            x=0,
            y=1.28,
            xanchor='left',
            yanchor='top',
            showactive=True,
        )
    ],
)

fig_custo.show(config={'displaylogo':False, 'toImageButtonOptions':{'format':'png','filename':'custo_mae_execucoes','scale':2}})

print(f'{len(comparacao_custo)} análises de {comparacao_custo.execucao.nunique()} execuções. Passe o mouse para ver CNN, data e configuração.')
print('Custo de referência = extração registrada + treino do protocolo (soma dos folds no LOEO).')
print('Features em cache mantêm o custo da extração de origem; o eixo X não é o tempo efetivamente gasto na reexecução.')
print('Os tempos excluem etapas não cronometradas, como montagem da base, avaliação e gravação de arquivos.')
print('Há bases, protocolos e dispositivos diferentes no histórico; compare essas condições antes de escolher uma CNN.')
colunas_detalhes = ['id_grafico','CNN','parametros','data_brasilia','metodologia','base',
    'dispositivo_extracao','MAE_mm','R2','tempo_extracao_s','tempo_regressor_s','tempo_referencia_s','features_reutilizadas']
custo_display(CustoHTML('<details><summary>Identificação das execuções e valores exatos</summary>'
    + comparacao_custo[colunas_detalhes].to_html(index=False, float_format=lambda v:f'{v:.4f}', escape=True) + '</details>'))
if avisos_custo:
    custo_display(CustoHTML('<details><summary>Análises sem tempos válidos</summary><pre>'
        + custo_escape('\n'.join(avisos_custo)) + '</pre></details>'))

# Artefatos da visualização: HTML interativo abre mesmo sem o runtime do Colab.
PASTA_GRAFICOS_CUSTO = PASTA_EXECUCOES.parent / 'comparacoes'
PASTA_GRAFICOS_CUSTO.mkdir(parents=True, exist_ok=True)
fig_custo.write_html(PASTA_GRAFICOS_CUSTO / 'custo_mae_execucoes.html', include_plotlyjs=True)
comparacao_custo.to_csv(PASTA_GRAFICOS_CUSTO / 'custo_mae_execucoes.csv', index=False)
print('Gráfico HTML e tabela salvos em:', PASTA_GRAFICOS_CUSTO)


## 12 — Análise temporal preliminar (definindo approach)

**Pergunta:** como construir uma coluna temporal útil, sem ensinar ao auxiliar os rótulos que ele precisa prever? Ainda estamos definindo o método: **uma feature, referente ao frame anterior**, não um dataset com 100 colunas.

Reutilizamos a matriz CNN e os metadados de uma execução estática salva. Não extraímos imagens novamente. A feature adicional é a **previsão do auxiliar para o frame anterior**, não a largura medida nem a imagem anterior.

### O que comparamos agora

O teste principal desta seção é **por blocos com teste central**: os trechos externos disponíveis de L1–L9 são reunidos para treinar um regressor final, avaliado nos blocos centrais escondidos desses mesmos experimentos. Validação e teste são separados; gaps são preservados.

| Regressor final | Entradas | Como a coluna de TREINO é produzida |
|---|---|---|
| Estático pareado | Features CNN e parâmetros, se ativados na origem | Não possui coluna temporal |
| Temporal — auxiliar por experimento | Mesmas entradas + uma previsão anterior | O auxiliar não vê nenhum trecho do experimento previsto |
| Temporal — auxiliar por blocos | Mesmas entradas + uma previsão anterior | O auxiliar pode ver outros trechos de treino do mesmo experimento, mas exclui o bloco previsto, seu predecessor e o gap |

**No auxiliar por blocos:** dividimos cada um dos dois trechos de treino em duas partes. Cada um dos quatro auxiliares esconde a parte correspondente de **todos os experimentos**, usando somente o restante do treino externo. Nenhum rótulo do teste de L2, por exemplo, é usado para gerar a coluna de L1.

**Controle importante:** só regeneramos a coluna do treino. As colunas de validação/teste são as mesmas do piloto anterior, produzidas por um auxiliar ajustado no treino externo. Reajustamos os três regressores finais nas mesmas linhas e com a mesma semente. O StandardScaler aprende exclusivamente no treino de cada ajuste. A seleção interna de épocas dos auxiliares continua igual à anterior (um experimento interno para validação e depois reajuste no pool permitido).

### Como executar sem rodar o estático

Execute **P1 → P6** abaixo. Por padrão, P3 **não treina**: P4 carrega a comparação salva. Para uma nova rodada, confira P1/P2, altere apenas `AB_EXECUTAR=True` em P3 e execute P3 → P5. São 11 ajustes nesta configuração, sem nova extração CNN. Os arquivos anteriores não são sobrescritos.

- **P4:** tabela principal dos três regressores; selecione validação ou teste.
- **P5:** gráficos originais de treino/validação/teste, com seletores de **método, divisão e experimento**: aleatório, blocos com teste final, blocos com teste central e LOEO. Usam o piloto original selecionado em P1; a comparação controlada da nova coluna por blocos fica em P4. Os pontos laranja são a feature auxiliar, **não a previsão final**.
- **P6:** histórico do piloto nas quatro metodologias, apenas para consulta; não inicia novamente o LOEO.

Uma melhora na coluna auxiliar não garante melhora no regressor final. A validação participa do EarlyStopping, o teste já foi explorado e uma única semente não demonstra superioridade robusta. Ordenação e curvas suaves interpoladas também não comprovam a correspondência física frame–largura.


In [ ]:
# P1 — preparação independente: reutiliza a base estática e o piloto salvo, sem treinar.
# Execute P1 a P6 desta seção. A parte estática não precisa ser reexecutada.
from pathlib import Path as ABPath
from datetime import datetime as ABDatetime, timezone as ABTimezone
from time import perf_counter as ab_clock
import json as ab_json
import hashlib as ab_hash
import numpy as ab_np
import pandas as ab_pd
from IPython.display import display as ab_display
from google.colab import drive as ab_drive

AB_PILOTO = ABPath('/content/drive/MyDrive/TCC/resultados/cnn_new_estatico_v2/temporal_pilotos/DenseNet169_20261014T204217_183926Z')
AB_RESULTADO_SALVO = ABPath('/content/drive/MyDrive/TCC/resultados/cnn_new_estatico_v2/temporal_ab_blocos/DenseNet169_20261017T135716_758966Z')
AB_PARTES_POR_TRECHO = 2  # Cada um dos dois trechos de treino vira dois blocos: quatro folds internos.
AB_VALIDADO = True
ab_resultado = None
ab_drive.mount('/content/drive')

def ab_ler(pasta):
    cfg = ab_json.loads((pasta/'configuracao.json').read_text())
    ab_json.loads((pasta/'concluido.json').read_text())  # Exige piloto concluído.
    candidatos = sorted((pasta/'blocos_central').glob('fold_*/configuracao.json'))
    if len(candidatos) != 1:
        raise ValueError('Esperada uma divisão central concluída, sem misturar execuções.')
    fold = candidatos[0].parent
    ab_json.loads((fold/'concluido.json').read_text())
    cfold = ab_json.loads(candidatos[0].read_text())
    origem = ABPath(cfg['analise_estatica_origem']).parent.parent
    cfg_x = ab_json.loads((origem/'configuracao.json').read_text())
    for k in ['dataset_id', 'arquitetura']:
        if cfg[k] != cfg_x[k] or cfg[k] != cfold[k]:
            raise ValueError('Origem incompatível: '+k)
    for k in ['parametros_processo','numero_entradas','semente','gap_frames',
              'epocas','paciencia','batch_regressao','taxa_aprendizado','regressor','padronizacao']:
        if cfg[k] != cfold[k]:
            raise ValueError('Configuração do fold diferente: '+k)
    if (cfg['regressor'] != 'Dense(32,relu)-Dense(1)'
        or cfg['padronizacao'] != 'StandardScaler_fit_somente_treino'
        or type(cfg['parametros_processo']) is not bool):
        raise ValueError('Configuração não suportada por este controle.')
    base = ab_pd.read_csv(pasta/'indice_base.csv')
    indice = ab_pd.read_csv(origem/'indice_features.csv')
    identidade = ['indice_dataset','experimento','frame','caminho_relativo','y',
                  'current','nozzle_speed','mass_flow_rate']
    ab_pd.testing.assert_frame_equal(base[identidade], indice[identidade],
                                     check_dtype=False, rtol=1e-6, atol=1e-6)
    if (not ab_np.array_equal(base.indice_dataset, ab_np.arange(len(base)))
        or base.duplicated(['experimento','frame']).any()):
        raise ValueError('Identidade/ordem inválida; não realinhamos silenciosamente.')
    X = ab_np.load(origem/'features.npy', mmap_mode='r', allow_pickle=False)
    custo = ab_pd.read_csv(origem/'custo_extrator.csv').iloc[0]
    if X.shape != (len(base), int(custo.features_cnn)) or custo.arquitetura != cfg['arquitetura']:
        raise ValueError('Matriz CNN incompatível.')
    if cfg['parametros_processo']:
        X = ab_np.column_stack([X, base[['current','nozzle_speed','mass_flow_rate']].to_numpy('float32')])
    y = base.y.to_numpy('float32')
    if X.shape[1] != cfg['numero_entradas'] or not ab_np.isfinite(X).all() or not (ab_np.isfinite(y)&(y>0)).all():
        raise ValueError('Entradas/rótulos inválidos.')
    with ab_np.load(fold/'divisao_comparacao.npz', allow_pickle=False) as z:
        div = {k: z[k].copy() for k in ['treino','validacao','teste']}
    trace = ab_pd.read_csv(fold/'rastreabilidade_temporal_1.csv')
    ab_pd.testing.assert_frame_equal(trace[['indice_dataset','experimento','frame','y']],
        base[['indice_dataset','experimento','frame','y']], check_dtype=False, rtol=1e-6, atol=1e-6)
    anterior = ab_np.load(fold/'temporal_1.npy', allow_pickle=False)
    if anterior.shape != (len(base),):
        raise ValueError('Feature temporal salva com dimensão errada.')
    usados = ab_np.concatenate(list(div.values()))
    if (usados.dtype.kind not in 'iu' or len(ab_np.unique(usados)) != len(usados)
        or (usados<0).any() or (usados>=len(base)).any()):
        raise ValueError('Divisões externas inválidas/sobrepostas.')
    fontes = ab_np.full(len(base), -1, dtype=int)
    flags = trace.utilizavel.astype(str).str.lower()
    if not flags.isin(['true','false']).all():
        raise ValueError('Flag utilizavel inválida.')
    for k, idx in div.items():
        esperado = ab_np.flatnonzero((trace.conjunto==k)&(flags=='true'))
        if not ab_np.array_equal(ab_np.sort(idx), esperado):
            raise ValueError('Contagens/índices divergentes: '+k)
        src = trace.iloc[idx].indice_origem.to_numpy(float)
        if not ab_np.isfinite(src).all() or not (src == ab_np.floor(src)).all():
            raise ValueError('Predecessor inválido.')
        src = src.astype(int)
        if (src<0).any() or (src>=len(base)).any():
            raise ValueError('Predecessor fora da base.')
        if (not ab_np.array_equal(base.experimento.to_numpy()[idx],base.experimento.to_numpy()[src])
            or not ab_np.array_equal(base.frame.to_numpy()[idx]-1,base.frame.to_numpy()[src])
            or not (trace.conjunto.to_numpy()[src]==k).all()):
            raise ValueError('Predecessor atravessa experimento, lacuna ou partição.')
        fontes[idx] = src
    if not ab_np.isfinite(anterior[usados]).all():
        raise ValueError('Feature antiga ausente nos exemplos pareados.')
    ab_np.testing.assert_allclose(anterior[usados],trace.temporal_1.to_numpy()[usados],rtol=1e-6,atol=1e-6)
    return cfg, base, X, y, div, fontes, anterior, fold, origem

def ab_planejar(base, treino, fontes, gap, partes):
    """Blocos de alvos; purga inclui predecessor E alvo atual, evitando vazamento na fronteira."""
    if not isinstance(gap, int) or gap < 1 or not isinstance(partes, int) or partes < 2:
        raise ValueError('Gap >=1 e pelo menos duas partes por trecho.')
    g, f = base.experimento.to_numpy(str), base.frame.to_numpy()
    grupos, spans = {}, {}
    for exp in sorted(ab_np.unique(g[treino])):
        ids = treino[g[treino]==exp]
        ids = ids[ab_np.argsort(f[ids], kind='stable')]
        trechos = ab_np.split(ids, ab_np.flatnonzero(ab_np.diff(f[ids]) != 1)+1)
        if len(trechos) != 2:
            raise ValueError(f'{exp}: esperado treino em dois trechos contíguos; confira a divisão central.')
        for t, trecho in enumerate(trechos):
            if len(trecho) < partes*2:
                raise ValueError(f'{exp}: trecho insuficiente para divisão interna.')
            for b, alvo in enumerate(ab_np.array_split(trecho, partes)):
                chave = t*partes+b
                grupos.setdefault(chave, []).append(alvo)
                lo, hi = int(f[fontes[alvo]].min()), int(f[alvo].max())
                spans.setdefault(chave, []).append((str(exp), lo, hi))
    plano = []
    for k in sorted(grupos):
        alvo = ab_np.concatenate(grupos[k])
        manter = ab_np.ones(len(treino), dtype=bool)
        for exp, lo, hi in spans[k]:
            manter &= ~((g[treino]==exp)&(f[treino]>=lo-gap)&(f[treino]<=hi+gap))
        pool = treino[manter]
        if len(pool)<2 or set(g[pool]) != set(g[treino]):
            raise ValueError('Purga removeu um experimento inteiro do pool.')
        plano.append({'fold':k+1,'alvos':alvo,'fontes':fontes[alvo],
                      'pool':pool,'intervalos':spans[k]})
    return plano

ab_cfg, ab_base, ab_X, ab_y, ab_div, ab_fontes, ab_anterior, ab_fold_origem, ab_origem = ab_ler(AB_PILOTO)
AB_GAP = int(ab_cfg['gap_frames'])  # Herda 100 desta execução; não otimizado pelo teste.
ab_plano = ab_planejar(ab_base, ab_div['treino'], ab_fontes, AB_GAP, AB_PARTES_POR_TRECHO)
print('Origem:', AB_PILOTO)
print('CNN:',ab_cfg['arquitetura'],'| parâmetros:',ab_cfg['parametros_processo'],'| base:',ab_cfg['dataset_id'])
print('Uma feature; somente teste central. Validação/teste mantêm EXATAMENTE a coluna temporal anterior.')
print('EarlyStopping interno dos auxiliares mantém a regra anterior: último experimento do pool, depois refit.')
print('Nenhum treinamento iniciado. Execute P2 para validar; P3 fica desativado por padrão.')


In [ ]:
# P2 — validação separada: cobertura, gap, alvo e predecessor fora do pool.
AB_VALIDADO = True
ab_g, ab_f = ab_base.experimento.to_numpy(str), ab_base.frame.to_numpy()
ab_tr, ab_va, ab_te = [ab_div[k] for k in ['treino','validacao','teste']]
ab_cobertura = ab_np.concatenate([p['alvos'] for p in ab_plano])
assert ab_np.array_equal(ab_np.sort(ab_cobertura), ab_np.sort(ab_tr))
assert len(ab_np.unique(ab_cobertura)) == len(ab_tr)
for p in ab_plano:
    assert ab_np.isin(p['pool'], ab_tr).all()
    assert not ab_np.intersect1d(p['pool'],ab_np.concatenate([p['alvos'],p['fontes'],ab_va,ab_te])).size
    for exp, lo, hi in p['intervalos']:
        frames_pool = ab_f[p['pool'][ab_g[p['pool']]==exp]]
        assert not ((frames_pool>=lo-AB_GAP)&(frames_pool<=hi+AB_GAP)).any()
    assert (ab_g[p['alvos']]==ab_g[p['fontes']]).all()
    assert (ab_f[p['alvos']]-ab_f[p['fontes']]==1).all()

def ab_assinatura():
    h = ab_hash.sha256(ab_json.dumps(ab_cfg,sort_keys=True).encode())
    h.update(str((AB_GAP,AB_PARTES_POR_TRECHO,AB_PILOTO)).encode())
    h.update(ab_base.to_csv(index=False).encode())
    h.update(ab_np.asarray(ab_X,dtype='<f4').tobytes())
    h.update(ab_np.asarray(ab_y,dtype='<f4').tobytes())
    h.update(ab_np.asarray(ab_anterior,dtype='<f4').tobytes())
    for idx in ab_div.values():
        h.update(ab_np.asarray(idx,dtype='<i8').tobytes())
    for p in ab_plano:
        for k in ['pool','alvos','fontes']:
            h.update(ab_np.asarray(p[k],dtype='<i8').tobytes())
    return h.hexdigest()

AB_ASSINATURA = ab_assinatura()
AB_VALIDADO = True
ab_display(ab_pd.DataFrame([{'fold_interno':p['fold'],'alvos_treino':len(p['alvos']),
    'pool_apos_gap':len(p['pool']),'gap_frames':AB_GAP} for p in ab_plano]))
print('Validação OK: nenhum alvo/predecessor no ajuste do seu auxiliar; divisões externas preservadas.')
print('Fits previstos:',2*len(ab_plano)+3,'(seleção/refit auxiliares + três regressores finais).')
print('Mesmos exemplos nos três braços:',{k:len(v) for k,v in ab_div.items()})


In [ ]:
# P3 — treinamento OPCIONAL. False permite apenas consultar os resultados salvos em P4.
AB_EXECUTAR = True  # True: após P1/P2, gera nova rodada de 11 fits; não sobrescreve a anterior.

def ab_executar():
    import tensorflow as ab_tf
    from tensorflow import keras as ab_keras
    from sklearn.preprocessing import StandardScaler as ABScaler
    from sklearn.metrics import mean_absolute_error as ab_mae, mean_squared_error as ab_mse, r2_score as ab_r2
    if not AB_VALIDADO or AB_ASSINATURA != ab_assinatura():
        raise RuntimeError('Configuração/dados alterados: execute P1 e P2 novamente.')
    ab_tf.config.experimental.enable_op_determinism()
    cfg = dict(ab_cfg)
    seed = int(cfg['semente'])
    pasta = AB_PILOTO.parent.parent/'temporal_ab_blocos'/(
        cfg['arquitetura']+'_'+ABDatetime.now(ABTimezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'))
    pasta.mkdir(parents=True,exist_ok=False)
    cfg.update({'piloto_referencia':str(AB_PILOTO),'protocolo':'blocos_central',
        'estrategia':'OOF_blocos_com_gap_somente_treino','partes_por_trecho':AB_PARTES_POR_TRECHO,
        'gap_interno':AB_GAP,'assinatura_ab':AB_ASSINATURA,'tensorflow':ab_tf.__version__,
        'validacao_teste_temporal':'reutilizados_sem_alteracao_do_piloto',
        'selecao_epocas_auxiliar':'ultimo_experimento_do_pool_como_no_piloto',
        'interpretacao':'teste exploratorio; teste externo ja inspecionado, nao confirmatorio'})
    (pasta/'configuracao.json').write_text(ab_json.dumps(cfg,indent=2),encoding='utf-8')
    ab_np.savez(pasta/'divisao_comparacao.npz',**ab_div)
    ab_base.to_csv(pasta/'indice_base.csv',index=False)
    print('Nova pasta (origem preservada):',pasta,flush=True)
    inicio_total = ab_clock()

    def ajustar(X,tr,va,nome,semente,fixas=None):
        ab_keras.backend.clear_session()
        ab_keras.utils.set_random_seed(semente)
        scaler = ABScaler().fit(X[tr])
        modelo = ab_keras.Sequential([ab_keras.layers.Input(shape=(X.shape[1],)),
            ab_keras.layers.Dense(32,activation='relu'),ab_keras.layers.Dense(1)])
        modelo.compile(optimizer=ab_keras.optimizers.Adam(learning_rate=cfg['taxa_aprendizado']),
                       loss='mean_absolute_error')
        kwargs = {} if fixas is not None else {
            'validation_data':(scaler.transform(X[va]).astype('float32'),ab_y[va]),
            'callbacks':[ab_keras.callbacks.EarlyStopping(monitor='val_loss',
                patience=cfg['paciencia'],restore_best_weights=True)]}
        t = ab_clock()
        h = modelo.fit(scaler.transform(X[tr]).astype('float32'),ab_y[tr],
            epochs=int(fixas if fixas is not None else cfg['epocas']),
            batch_size=cfg['batch_regressao'],shuffle=True,verbose=0,**kwargs)
        tempo = ab_clock()-t
        melhor = int(fixas) if fixas is not None else int(ab_np.argmin(h.history['val_loss'])+1)
        modelo.save(pasta/(nome+'.keras'))
        ab_np.savez(pasta/(nome+'_scaler.npz'),media=scaler.mean_,escala=scaler.scale_)
        ab_np.savez(pasta/(nome+'_indices.npz'),treino=tr,validacao=va)
        ab_pd.DataFrame(h.history).to_csv(pasta/(nome+'_historico.csv'),index=False)
        return modelo,scaler,melhor,tempo

    def prever(modelo,scaler,X,idx):
        p = modelo.predict(scaler.transform(X[idx]).astype('float32'),
            batch_size=cfg['batch_regressao'],verbose=0).ravel()
        if len(p)!=len(idx) or not ab_np.isfinite(p).all():
            raise ValueError('Previsão inválida.')
        return p

    def medidas(real,p):
        return {'MAE_mm':float(ab_mae(real,p)),
            'MAPE_pct':float(100*ab_np.mean(ab_np.abs((real-p)/real))),
            'RMSE_mm':float(ab_np.sqrt(ab_mse(real,p))),
            'R2':float(ab_r2(real,p)) if ab_np.var(real)>0 else ab_np.nan}

    nova = ab_anterior.copy()  # Validação/teste permanecem idênticos ao controle anterior.
    autoria = ab_np.full(len(ab_base),'auxiliar_externo_anterior',dtype=object)
    custos, auditoria = [], []
    for p in ab_plano:
        nome = f"aux_bloco_{p['fold']:02d}"
        pool = p['pool']
        exp_va = sorted(ab_np.unique(ab_g[pool]))[-1]
        iva, itr = pool[ab_g[pool]==exp_va],pool[ab_g[pool]!=exp_va]
        if len(iva)<2 or len(itr)<2:
            raise ValueError('Pool interno insuficiente.')
        seletor,_,epocas,t1 = ajustar(ab_X,itr,iva,nome+'_selecao',seed+p['fold']-1)
        del seletor
        auxiliar,scaler,_,t2 = ajustar(ab_X,pool,ab_np.array([],dtype=int),
            nome+'_refit',seed+p['fold']-1,fixas=epocas)
        nova[p['alvos']] = prever(auxiliar,scaler,ab_X,p['fontes'])
        autoria[p['alvos']] = nome
        del auxiliar
        ab_np.savez(pasta/(nome+'_rastreio.npz'),pool=pool,alvos=p['alvos'],fontes=p['fontes'])
        auditoria.append({'fold':p['fold'],'intervalos_purgados_antes_gap':p['intervalos'],
            'experimento_validacao_interna':str(exp_va),'epocas':epocas,'tempo_fit_s':t1+t2})
        custos.append(t1+t2)
        print(f"Auxiliar {p['fold']}/{len(ab_plano)} concluído | épocas={epocas}",flush=True)
    for idx in [ab_va,ab_te]:
        ab_np.testing.assert_array_equal(nova[idx],ab_anterior[idx])
    assert ab_np.isfinite(nova[ab_tr]).all()
    ab_np.save(pasta/'temporal_blocos.npy',nova)
    (pasta/'auditoria_auxiliares.json').write_text(ab_json.dumps(auditoria,indent=2),encoding='utf-8')
    usados = ab_np.concatenate([ab_tr,ab_va,ab_te])
    rastreio = ab_base.iloc[usados][['indice_dataset','experimento','frame','y']].copy()
    conjunto = ab_np.full(len(ab_base),'fora',dtype=object)
    for k,idx in ab_div.items():
        conjunto[idx] = k
    rastreio['conjunto'] = conjunto[usados]
    rastreio['indice_origem'] = ab_fontes[usados]
    rastreio['feature_anterior'] = ab_anterior[usados]
    rastreio['feature_blocos'] = nova[usados]
    rastreio['auxiliar_blocos'] = autoria[usados]
    rastreio.to_csv(pasta/'rastreabilidade.csv',index=False)

    # Reajusta os TRÊS regressores finais na mesma sessão/semente para comparação controlada.
    # Não reutiliza métricas antigas como se fossem uma execução idêntica.
    linhas, preds = [], []
    for nome,coluna in [('estatico',None),('temporal_experimentos',ab_anterior),('temporal_blocos',nova)]:
        X = ab_X if coluna is None else ab_np.column_stack([ab_X,coluna])
        modelo,scaler,epocas,tempo = ajustar(X,ab_tr,ab_va,nome,seed)
        for conjunto_nome,idx in [('validacao',ab_va),('teste',ab_te)]:
            pred = prever(modelo,scaler,X,idx)
            linhas.append({'modelo':nome,'conjunto':conjunto_nome,**medidas(ab_y[idx],pred),
                'N':len(idx),'epocas_melhor':epocas,'tempo_fit_final_s':tempo})
            d = ab_base.iloc[idx][['indice_dataset','experimento','frame','y']].copy()
            d['modelo'],d['conjunto'],d['y_predito'] = nome,conjunto_nome,pred
            preds.append(d)
        ab_pd.DataFrame(linhas).to_csv(pasta/'metricas_parciais.csv',index=False)
        print(nome,'concluído. MAE validação:',round(linhas[-2]['MAE_mm'],4),flush=True)
        del modelo
    resultado = ab_pd.DataFrame(linhas)
    resultado.to_csv(pasta/'metricas.csv',index=False)
    ab_pd.concat(preds,ignore_index=True).to_csv(pasta/'predicoes.csv',index=False)
    diagnostico = []
    for nome,coluna in [('temporal_experimentos',ab_anterior),('temporal_blocos',nova)]:
        for k,idx in ab_div.items():
            diagnostico.append({'modelo':nome,'conjunto':k,
                'MAE_auxiliar_anterior_mm':float(ab_np.mean(ab_np.abs(coluna[idx]-ab_y[ab_fontes[idx]]))),
                'MAE_feature_atual_mm':float(ab_np.mean(ab_np.abs(coluna[idx]-ab_y[idx]))),
                'vies_feature_mm':float(ab_np.mean(coluna[idx]-ab_y[idx]))})
    ab_pd.DataFrame(diagnostico).to_csv(pasta/'diagnostico_feature.csv',index=False)
    (pasta/'concluido.json').write_text(ab_json.dumps({'fits':2*len(ab_plano)+3,
        'tempo_novos_auxiliares_s':sum(custos),'tempo_total_s':ab_clock()-inicio_total,
        'custo':'Somente custo incremental; CNN e auxiliares anteriores reutilizados, não comparáveis como custo total.'},indent=2),encoding='utf-8')
    return {'pasta':pasta,'metricas':resultado,'diagnostico':ab_pd.DataFrame(diagnostico)}

if AB_EXECUTAR:
    ab_resultado = None
    ab_resultado = ab_executar()
else:
    print('Treino desativado. Siga para P4/P5: resultados e gráficos da pasta salva em P1.')


In [ ]:
# P4 — carregar resultados salvos e mostrar somente três linhas por vez. Não treina.
import ipywidgets as ap_widgets
from IPython.display import HTML as APHTML, clear_output as ap_clear
from html import escape as ap_escape

def ap_carregar(pasta):
    cfg = ab_json.loads((pasta/'configuracao.json').read_text())
    fim = ab_json.loads((pasta/'concluido.json').read_text())
    if cfg.get('piloto_referencia') != str(AB_PILOTO) or cfg.get('protocolo') != 'blocos_central':
        raise ValueError('A comparação não pertence ao piloto central selecionado.')
    for k in ['dataset_id','arquitetura','parametros_processo','numero_entradas','semente']:
        if cfg[k] != ab_cfg[k]:
            raise ValueError('Configuração incompatível: '+k)
    b = ab_pd.read_csv(pasta/'indice_base.csv')
    cols = ['indice_dataset','experimento','frame','y']
    ab_pd.testing.assert_frame_equal(b[cols],ab_base[cols],check_dtype=False,rtol=1e-6,atol=1e-6)
    with ab_np.load(pasta/'divisao_comparacao.npz',allow_pickle=False) as z:
        for k,idx in ab_div.items():
            ab_np.testing.assert_array_equal(z[k],idx)
    r = ab_pd.read_csv(pasta/'rastreabilidade.csv').set_index('indice_dataset',drop=False)
    usados = ab_np.concatenate(list(ab_div.values()))
    if r.index.has_duplicates or set(r.index) != set(usados):
        raise ValueError('Rastreabilidade incompleta ou duplicada.')
    r = r.loc[usados].copy()
    ab_pd.testing.assert_frame_equal(r[cols].reset_index(drop=True),ab_base.iloc[usados][cols].reset_index(drop=True),check_dtype=False,rtol=1e-6,atol=1e-6)
    ab_np.testing.assert_array_equal(r.indice_origem,ab_fontes[usados])
    ab_np.testing.assert_allclose(r.feature_anterior,ab_anterior[usados],rtol=1e-6,atol=1e-6)
    if not ab_np.isfinite(r[['feature_anterior','feature_blocos']]).all().all():
        raise ValueError('Feature não finita.')
    for k,idx in ab_div.items():
        if not (r.loc[idx,'conjunto']==k).all():
            raise ValueError('Partições divergentes.')
        if k != 'treino':
            ab_np.testing.assert_allclose(r.loc[idx,'feature_anterior'],r.loc[idx,'feature_blocos'],rtol=1e-6,atol=1e-6)
    m = ab_pd.read_csv(pasta/'metricas.csv')
    modelos = {'estatico','temporal_experimentos','temporal_blocos'}
    esperado = {(modelo,k) for modelo in modelos for k in ['validacao','teste']}
    if len(m)!=6 or set(zip(m.modelo,m.conjunto)) != esperado:
        raise ValueError('Métricas incompletas.')
    for k in ['validacao','teste']:
        if not (m.loc[m.conjunto==k,'N']==len(ab_div[k])).all():
            raise ValueError('Métricas usam outro conjunto de exemplos.')
    return {'pasta':pasta,'metricas':m,'rastreio':r,'custo':fim,
            'diagnostico':ab_pd.read_csv(pasta/'diagnostico_feature.csv')}

# Uma nova rodada de P3 tem precedência; caso contrário, usa a pasta explícita em P1.
ap_pasta = ab_resultado['pasta'] if ab_resultado is not None else AB_RESULTADO_SALVO
ap_resultado = ap_carregar(ap_pasta)
AP_NOMES = {'estatico':'Estático pareado','temporal_experimentos':'Temporal — auxiliar por experimento',
            'temporal_blocos':'Temporal — auxiliar por blocos'}
ap_particao = ap_widgets.Dropdown(options=[('Validação','validacao'),('Teste (exploratório)','teste')],description='Conjunto:')
ap_saida = ap_widgets.Output()
def ap_resultados(_=None):
    with ap_saida:
        ap_clear(wait=True)
        m = ap_resultado['metricas'].query('conjunto == @ap_particao.value').set_index('modelo').loc[list(AP_NOMES)].copy()
        referencia = m.loc['estatico','MAE_mm']
        m['Redução MAE (%)'] = 100*(referencia-m.MAE_mm)/referencia if referencia>0 else ab_np.nan
        m.index = m.index.map(AP_NOMES)
        ab_display(APHTML('<h3>Blocos — teste central | '+ap_escape(ap_particao.label)+'</h3>'))
        ab_display(m[['MAE_mm','MAPE_pct','RMSE_mm','R2','Redução MAE (%)']].style.format(precision=4)
                   .highlight_min(subset=['MAE_mm'],color='#d8efd3'))
        print('Mesmos',int(m.N.iloc[0]),'exemplos. Redução de MAE positiva = melhora em relação ao estático.')
        print('Validação orienta EarlyStopping; teste já explorado. Não são evidências confirmatórias.')
        d = ap_resultado['diagnostico'].query("conjunto == 'treino'").copy()
        d['modelo'] = d.modelo.map(AP_NOMES)
        ab_display(APHTML('<details><summary>Diagnóstico da coluna no treino e custos</summary>'+
            d[['modelo','MAE_auxiliar_anterior_mm','MAE_feature_atual_mm','vies_feature_mm']].to_html(index=False,float_format=lambda x:f'{x:.4f}')+
            '<p>'+ap_escape(str(ap_resultado['custo']))+'</p></details>'))
        print('Origem:',ap_resultado['pasta'])
ap_particao.observe(ap_resultados,names='value')
ab_display(ap_widgets.VBox([ap_particao,ap_saida]))
ap_resultados()


In [ ]:
# P5 — gráficos do piloto: selecione método, divisão e experimento. Somente leitura.
from pathlib import Path as TDPath
import json as td_json
import numpy as td_np
import pandas as td_pd
import matplotlib.pyplot as td_plt
import ipywidgets as td_widgets
from IPython.display import display as td_display, HTML as TDHTML, clear_output as td_clear
from html import escape as td_escape

# Usa a origem explícita de P1, nunca uma variável de outra rodada ainda na memória.
TD_PASTA = TDPath(AB_PILOTO)

def td_correlacao(a, b):
    a, b = td_np.asarray(a, dtype=float), td_np.asarray(b, dtype=float)
    if len(a) < 2 or td_np.std(a) < 1e-12 or td_np.std(b) < 1e-12:
        return td_np.nan
    return float(td_np.corrcoef(a, b)[0, 1])

def td_medidas(d):
    real, feature = d.y.to_numpy(dtype=float), d.temporal_1.to_numpy(dtype=float)
    erro = feature-real
    # Remove SOMENTE para diagnóstico as médias por experimento, nunca muda a entrada do modelo.
    centrado = d[['experimento', 'y', 'temporal_1']].copy()
    centrado[['y','temporal_1']] -= centrado.groupby('experimento')[['y','temporal_1']].transform('mean')
    return {'N': len(d), 'MAE_feature_mm': float(td_np.mean(td_np.abs(erro))),
        'vies_mm': float(td_np.mean(erro)),
        'corr_dentro_experimento': td_correlacao(centrado.y, centrado.temporal_1),
        'MAE_auxiliar_no_anterior_mm': float(td_np.mean(td_np.abs(feature-d.y_anterior.to_numpy()))),
        'MAE_y_anterior_real_mm': float(td_np.mean(td_np.abs(d.y_anterior.to_numpy()-real))),
        'media_y_mm': float(td_np.mean(real)), 'media_feature_mm': float(td_np.mean(feature)),
        'desvio_y_mm': float(td_np.std(real)), 'desvio_feature_mm': float(td_np.std(feature))}

def td_ler_piloto(pasta):
    cfg = td_json.loads((pasta/'configuracao.json').read_text())
    fim = td_json.loads((pasta/'concluido.json').read_text())
    base = td_pd.read_csv(pasta/'indice_base.csv')
    if not td_np.array_equal(base.indice_dataset, td_np.arange(len(base))):
        raise ValueError('Índice da base não é sequencial; não realinhamos silenciosamente.')
    traces = sorted(pasta.glob('*/fold_*/rastreabilidade_temporal_1.csv'))
    if len(traces) != int(fim['divisoes_concluidas']):
        raise ValueError('Quantidade de divisões difere do registro de conclusão.')
    if not traces:
        raise ValueError('Nenhuma rastreabilidade temporal salva.')
    dados, linhas, por_exp = {}, [], []
    for arquivo in traces:
        c = td_json.loads((arquivo.parent/'configuracao.json').read_text())
        for campo in ['dataset_id', 'arquitetura', 'parametros_processo']:
            if c[campo] != cfg[campo]:
                raise ValueError(f'Configuração incompatível em {arquivo}: {campo}')
        chave = (c['protocolo'], int(c['fold']))
        if chave in dados:
            raise ValueError('Divisão duplicada: ' + str(chave))
        d = td_pd.read_csv(arquivo)
        identidade = ['indice_dataset','experimento','frame','y']
        td_pd.testing.assert_frame_equal(d[identidade].reset_index(drop=True), base[identidade],
                                        check_dtype=False, rtol=1e-6, atol=1e-6)
        flags = d.utilizavel.astype(str).str.lower()
        if not flags.isin(['true','false']).all():
            raise ValueError('Flag utilizavel inválida.')
        d = d.loc[flags == 'true'].copy()
        if d.empty or not td_np.isfinite(d[['y','temporal_1','indice_origem']].to_numpy()).all():
            raise ValueError('Rótulos/features/fontes inválidos: ' + str(arquivo))
        if not td_np.equal(d.indice_origem, td_np.floor(d.indice_origem)).all():
            raise ValueError('Índice de origem não inteiro.')
        fontes = d.indice_origem.to_numpy(dtype=int)
        if (fontes < 0).any() or (fontes >= len(base)).any():
            raise ValueError('Índice de origem fora da base.')
        anterior = base.iloc[fontes]
        if (not td_np.array_equal(d.experimento.to_numpy(), anterior.experimento.to_numpy())
            or not td_np.array_equal(d.frame.to_numpy()-1, anterior.frame.to_numpy())):
            raise ValueError('Predecessor não é t-1 do mesmo experimento.')
        completo = td_pd.read_csv(arquivo, usecols=['conjunto'])
        if not td_np.array_equal(d.conjunto.to_numpy(), completo.iloc[fontes].conjunto.to_numpy()):
            raise ValueError('Predecessor pertence a outro conjunto.')
        d['y_anterior'] = anterior.y.to_numpy()  # Só diagnóstico; NÃO entra no treinamento.
        d['protocolo'], d['fold'], d['nome_divisao'] = chave[0], chave[1], c['nome_divisao']
        with td_np.load(arquivo.parent/'divisao_comparacao.npz', allow_pickle=False) as z:
            for conjunto in ['treino','validacao','teste']:
                p = d.loc[d.conjunto == conjunto]
                if len(p) < 2 or not td_np.array_equal(td_np.sort(z[conjunto]), td_np.sort(p.indice_dataset)):
                    raise ValueError('Linhas não correspondem à comparação pareada: ' + conjunto)
                linhas.append({'protocolo': chave[0], 'fold': chave[1], 'conjunto': conjunto, **td_medidas(p)})
                for exp, e in p.groupby('experimento', sort=True):
                    por_exp.append({'protocolo': chave[0], 'fold': chave[1], 'conjunto': conjunto,
                                    'experimento': exp, **td_medidas(e)})
        dados[chave] = d
    return cfg, dados, td_pd.DataFrame(linhas), td_pd.DataFrame(por_exp)

td_cfg, td_dados, td_resumo, td_por_exp = td_ler_piloto(TD_PASTA)
td_nomes = {'aleatorio':'Aleatório', 'blocos':'Blocos — teste na extremidade (final)',
            'blocos_central':'Blocos — teste no meio (central)', 'loeo':'LOEO'}
td_disponiveis = list(dict.fromkeys(k[0] for k in td_dados))
td_metodo = td_widgets.Dropdown(options=[(td_nomes[k],k) for k in td_disponiveis],
    value='blocos_central' if 'blocos_central' in td_disponiveis else td_disponiveis[0], description='Método:',
    layout=td_widgets.Layout(width='430px'))
td_fold = td_widgets.Dropdown(description='Divisão:')
td_exp = td_widgets.Dropdown(description='Experimento:')
td_saida = td_widgets.Output()
td_atualizando = False

def td_desenhar(_=None):
    if td_atualizando or td_fold.value is None or td_exp.value is None:
        return
    with td_saida:
        td_clear(wait=True)
        metodo, fold = td_metodo.value, td_fold.value
        d = td_dados[(metodo, fold)]
        r = td_resumo.loc[(td_resumo.protocolo == metodo) & (td_resumo.fold == fold)].copy()
        r = r.set_index('conjunto').reindex(['treino','validacao','teste'])
        tabela = r[['N','MAE_feature_mm','vies_mm','corr_dentro_experimento']].rename(columns={
            'MAE_feature_mm':'MAE feature → y(t) (mm)', 'vies_mm':'Viés (mm)',
            'corr_dentro_experimento':'Correlação das variações'})
        td_display(TDHTML('<h3>Qualidade da coluna temporal — divisão selecionada</h3>'))
        td_display(tabela.style.format({'N':'{:.0f}', 'MAE feature → y(t) (mm)':'{:.3f}',
            'Viés (mm)':'{:.3f}', 'Correlação das variações':'{:.3f}'}, na_rep='Indefinida')
            .set_properties(subset=['MAE feature → y(t) (mm)'],
                **{'font-weight':'bold','background-color':'#eaf2ff','color':'#12335b'}))
        print('Esta tabela avalia a feature isolada, NÃO a previsão do regressor temporal final.')
        print('Viés positivo: feature acima de y; negativo: abaixo. Correlação remove médias por experimento, só no diagnóstico.')
        if metodo == 'loeo':
            print('LOEO: treino, validação e teste contêm experimentos diferentes; diferenças não isolam a causa do erro.')
        else:
            print('Treino: auxiliar exclui o experimento. Validação/teste: auxiliar pode usar outros frames de treino desse experimento.')
        e = d.loc[d.experimento == td_exp.value].sort_values('frame')
        fig, axes = td_plt.subplots(3, 1, figsize=(11,7), sharex=True, sharey=True)
        for ax, conjunto in zip(axes, ['treino','validacao','teste']):
            p = e.loc[e.conjunto == conjunto]
            if p.empty:
                ax.text(.5,.5,'Experimento ausente deste conjunto',transform=ax.transAxes,ha='center')
            else:
                # Pontos, não linhas: não ligamos frames separados por lacunas/gaps.
                ax.plot(p.frame,p.y,'.',ms=2.5,color='#17324d',label='y real do frame atual')
                ax.plot(p.frame,p.temporal_1,'.',ms=2.5,alpha=.65,color='#d17b21',label='Feature: previsão de t−1')
                ax.legend(fontsize=8,loc='best')
            ax.set_title(conjunto,loc='left',fontsize=10)
            ax.set_ylabel('Largura (mm)')
            ax.grid(alpha=.2)
        axes[-1].set_xlabel('Número do frame — sem reordenar os artefatos salvos')
        fig.suptitle(f'{td_nomes[metodo]} | {td_exp.value} | y(t) × feature temporal',fontsize=12)
        fig.tight_layout()
        td_plt.show()
        td_plt.close(fig)
        detalhe = td_por_exp.loc[(td_por_exp.protocolo == metodo) & (td_por_exp.fold == fold)].drop(columns=['protocolo','fold'])
        td_display(TDHTML('<details><summary>Detalhes por experimento: auxiliar, deslocamento e distribuição</summary>'
            '<p>MAE_auxiliar_no_anterior_mm: previsão auxiliar versus y(t−1). '
            'MAE_y_anterior_real_mm: diferença entre y(t−1) e y(t), referência retrospectiva com rótulos reais; '
            'não é um modelo utilizável nem uma entrada fornecida ao regressor. '
            'As médias/desvios e correlações são descritivos: sozinhos não demonstram a causa da piora.</p>'
            + detalhe.to_html(index=False,float_format=lambda v:f'{v:.4f}',escape=True)+'</details>'))
        print('Diagnóstico exploratório. Não escolher hiperparâmetros repetidamente pelo resultado do teste.')

def td_atualizar_exp(_=None):
    global td_atualizando
    td_atualizando = True
    d = td_dados[(td_metodo.value, td_fold.value)]
    td_exp.options = sorted(d.experimento.unique().tolist())
    td_exp.value = sorted(d.loc[d.conjunto == 'teste','experimento'].unique().tolist())[0]
    td_atualizando = False
    td_desenhar()

def td_atualizar_fold(_=None):
    global td_atualizando
    td_atualizando = True
    td_fold.options = [(str(v.nome_divisao.iloc[0]), k[1]) for k,v in td_dados.items() if k[0] == td_metodo.value]
    td_fold.value = td_fold.options[0][1]
    td_atualizando = False
    td_atualizar_exp()

td_metodo.observe(td_atualizar_fold,names='value')
td_fold.observe(lambda _: None if td_atualizando else td_atualizar_exp(),names='value')
td_exp.observe(td_desenhar,names='value')
print('Leitura somente de:', TD_PASTA)
print('Gráficos do piloto original nas quatro análises; a comparação controlada por blocos permanece em P4.')
print('CNN:', td_cfg['arquitetura'], '| parâmetros:', td_cfg['parametros_processo'], '| base:', td_cfg['dataset_id'])
td_display(td_widgets.VBox([td_metodo,td_widgets.HBox([td_fold,td_exp]),td_saida]))
td_atualizar_fold()


In [ ]:
# P6 — histórico do piloto anterior, somente leitura. Não executa LOEO nem novos fits.
ap_historico = ab_pd.read_csv(AB_PILOTO/'comparacao_temporal.csv')
AP_PROTOCOL = {'aleatorio':'Aleatório independente','blocos':'Blocos — teste final',
    'blocos_central':'Blocos — teste central','loeo':'LOEO'}
ap_hist_metodo = ap_widgets.Dropdown(options=[(AP_PROTOCOL.get(p,p),p) for p in ap_historico.protocolo.unique()],description='Histórico:')
ap_hist_saida = ap_widgets.Output()
def ap_historico_mostrar(_=None):
    with ap_hist_saida:
        ap_clear(wait=True)
        d = ap_historico.loc[ap_historico.protocolo==ap_hist_metodo.value].copy()
        d.modelo = d.modelo.replace({'estatico_pareado':'Estático pareado','temporal_1':'Temporal — auxiliar por experimento'})
        ab_display(d.set_index('modelo')[['MAE_mm','MAPE_pct','RMSE_mm','R2','n_teste']].style.format(precision=4))
        print('Piloto original de 14/10: não confundir com os três reajustes de P4.')
        print('Teste e base variam entre protocolos; compare os dois modelos dentro de cada protocolo.')
        print('LOEO agrega previsões dos folds; não faz média dos R². Custos antigos permanecem no CSV.')
ap_hist_metodo.observe(ap_historico_mostrar,names='value')
ab_display(ap_widgets.VBox([ap_hist_metodo,ap_hist_saida]))
ap_historico_mostrar()


## 13 — Análise temporal: primeira implementação com 10 colunas

**Escopo desta rodada: teste no bloco central, comparando 0, 1, 5 e 10 colunas nos mesmos frames.** A seção estática e a análise temporal preliminar permanecem intactas. Esta é a primeira implementação do ciclo, ainda não uma validação científica concluída.

Reutilizamos as features CNN, a base e a divisão central da execução indicada. Não extraímos imagens novamente. Mantemos `Dense(32, relu) → Dense(1)`, parâmetros de processo conforme a origem e `StandardScaler` ajustado somente no respectivo treino.

**Como as dez colunas são construídas:** em cada etapa, um novo auxiliar recebe as features estáticas e as colunas temporais anteriores. Sua previsão do frame **t−1** vira a nova entrada do frame **t**. Portanto, a coluna 10 não é simplesmente a previsão estática do frame t−10: ela resulta de dez etapas de geração e retreinamento. Nenhum y real anterior é fornecido como feature.

**Proteção entre etapas:** quatro cadeias geram as colunas de treino, cada uma excluindo seus blocos-alvo, o histórico de dez frames e o gap durante **todas** as dez etapas. Uma quinta cadeia aprende somente no treino externo e produz as colunas de validação/teste. Não reutilizamos colunas produzidas por outra cadeia para treinar auxiliares posteriores.

**Limitação explícita:** dentro do pool de cada cadeia, a recursão utiliza previsões in-sample. A exclusão vale para a cadeia completa, mas não se trata de cross-fitting integralmente aninhado em cada camada. Essa escolha controla o custo e impede que os rótulos do bloco escondido entrem em sua cadeia; ainda pode existir diferença de distribuição entre treino e uso das features. Os auxiliares usam um orçamento fixo de **50 épocas**, pré-declarado, sem EarlyStopping externo. Os regressores finais usam validação separada e EarlyStopping. Não é reprodução exata do pipeline do Leonel nem da seleção de épocas do piloto.

### T10.1 — Origem e plano de execução

Em uma sessão nova, execute **somente P1 da seção preliminar** para montar o Drive e disponibilizar o carregador validado; depois siga T10.1 → T10.2 → T10.3 → T10.4. Não precisa rodar o estático nem P2–P6. Nesta célula, confira `T10_PILOTO`, CNN, base e parâmetros exibidos. São **10 colunas**, **54 fits** no total e uma semente. `T10_EXECUTAR=False` impede iniciar o treinamento por acidente. Não compare diretamente esta rodada com resultados antigos que utilizaram outros frames ou outro auxiliar.


In [ ]:
# T10.1 — carregar a execução estática e congelar os blocos; nenhum treinamento.
# Pré-requisito: execute somente P1 da seção preliminar para definir ab_ler e montar o Drive.
if 'ab_ler' not in globals():
    raise RuntimeError('Execute P1 da seção preliminar primeiro. Não precisa executar o estático nem P2–P6.')
from pathlib import Path as T10Path
from datetime import datetime as T10Datetime, timezone as T10Timezone
from time import perf_counter as t10_clock
import json as t10_json
import hashlib as t10_hash
import numpy as t10_np
import pandas as t10_pd
from IPython.display import display as t10_display

T10_PILOTO = T10Path('/content/drive/MyDrive/TCC/resultados/cnn_new_estatico_v2/temporal_pilotos/DenseNet169_20261014T204217_183926Z')
T10_COLUNAS = 10
T10_AVALIAR = (0, 1, 5, 10)
T10_EPOCAS_AUX = 50  # Orçamento fixo pré-declarado; não escolhido olhando o teste.
T10_RETOMAR = ''  # Opcional: pasta t10 de uma execução interrompida desta MESMA configuração.
T10_EXECUTAR = True  # Altere para True somente depois de conferir T10.1 e T10.2.

t10_cfg, t10_base, t10_X, t10_y, _, _, _, t10_fold, t10_origem = ab_ler(T10_PILOTO)
with t10_np.load(t10_fold/'divisao_original.npz', allow_pickle=False) as z:
    t10_div_original = {s: z[s].copy() for s in ('treino','validacao','teste')}
t10_trace_original = t10_pd.read_csv(t10_fold/'rastreabilidade_temporal_1.csv')
for t10_s, t10_ids in t10_div_original.items():
    if not t10_np.array_equal(t10_np.sort(t10_ids),t10_np.flatnonzero(t10_trace_original.conjunto==t10_s)):
        raise ValueError('Divisão original diverge da rastreabilidade: '+t10_s)

def t10_historico(base, div, profundidade):
    """Índices t-1...t-K: exatos, mesmo experimento e mesma partição externa."""
    n = len(base)
    grupos, frames = base.experimento.to_numpy(str), base.frame.to_numpy()
    if not t10_np.array_equal(frames, frames.astype(int)):
        raise ValueError('Frame não inteiro.')
    dono = t10_np.full(n, '', dtype=object)
    for nome, ids in div.items():
        if (ids.dtype.kind not in 'iu' or (ids < 0).any() or (ids >= n).any()
            or len(t10_np.unique(ids)) != len(ids) or (dono[ids] != '').any()):
            raise ValueError('Índices externos inválidos ou sobrepostos.')
        dono[ids] = nome
    lookup = {(str(e), int(f)): i for i, (e, f) in enumerate(zip(grupos, frames))}
    if len(lookup) != n:
        raise ValueError('Experimento/frame duplicado.')
    hist = t10_np.full((n, profundidade), -1, dtype=int)
    for i in t10_np.flatnonzero(dono != ''):
        for lag in range(1, profundidade+1):
            j = lookup.get((grupos[i], int(frames[i])-lag), -1)
            if j < 0 or dono[j] != dono[i]:
                break
            hist[i, lag-1] = j
    return hist, dono

def t10_planejar(base, div, hist, gap):
    """Quatro cadeias OOF: 2 partes de cada um dos 2 trechos de treino."""
    grupos, frames = base.experimento.to_numpy(str), base.frame.to_numpy()
    treino = div['treino']
    elegiveis = treino[(hist[treino] >= 0).all(axis=1)]
    alvos_por_cadeia = {k: [] for k in range(4)}
    for exp in sorted(set(grupos[treino])):
        ids = treino[grupos[treino] == exp]
        ids = ids[t10_np.argsort(frames[ids])]
        trechos = t10_np.split(ids, t10_np.flatnonzero(t10_np.diff(frames[ids]) != 1)+1)
        if len(trechos) != 2:
            raise ValueError(f'{exp}: esperado treino central em dois trechos contíguos.')
        for r, trecho in enumerate(trechos):
            for b, pedaco in enumerate(t10_np.array_split(trecho, 2)):
                alvo = t10_np.intersect1d(pedaco, elegiveis)
                if not len(alvo):
                    raise ValueError('Bloco sem dez predecessores disponíveis.')
                alvos_por_cadeia[2*r+b].append(alvo)
    planos = []
    for k, partes in alvos_por_cadeia.items():
        alvos = t10_np.concatenate(partes)
        permitido = t10_np.zeros(len(base), dtype=bool)
        permitido[treino] = True
        intervalos = []
        for exp in sorted(set(grupos[alvos])):
            a = alvos[grupos[alvos] == exp]
            lo, hi = int(frames[hist[a, -1]].min())-gap, int(frames[a].max())+gap
            permitido[(grupos == exp) & (frames >= lo) & (frames <= hi)] = False
            intervalos.append([str(exp), lo, hi])
        # Todos os dez predecessores dos frames usados em fit também ficam no pool permitido.
        pool = elegiveis[permitido[elegiveis] & permitido[hist[elegiveis]].all(axis=1)]
        if len(pool) < 100 or set(grupos[pool]) != set(grupos[treino]):
            raise ValueError('Pool auxiliar insuficiente depois de purgar o bloco e seu histórico.')
        planos.append({'nome':f'oof_{k+1}', 'pool':pool, 'alvos':alvos, 'purga':intervalos})
    planos.append({'nome':'externa', 'pool':elegiveis, 'alvos':t10_np.concatenate([
        ids[(hist[ids] >= 0).all(axis=1)] for s, ids in div.items() if s != 'treino']), 'purga':[]})
    return planos

t10_hist, t10_dono = t10_historico(t10_base, t10_div_original, T10_COLUNAS)
t10_div = {s: ids[(t10_hist[ids] >= 0).all(axis=1)] for s, ids in t10_div_original.items()}
t10_planos = t10_planejar(t10_base, t10_div_original, t10_hist, int(t10_cfg['gap_frames']))
t10_conf = {'versao':'cadeias_isoladas_v1', 'piloto':str(T10_PILOTO), 'origem_estatica':str(t10_origem),
    'base':t10_cfg['dataset_id'], 'cnn':t10_cfg['arquitetura'], 'parametros':t10_cfg['parametros_processo'],
    'colunas':T10_COLUNAS, 'avaliar':list(T10_AVALIAR), 'epocas_aux':T10_EPOCAS_AUX,
    'configuracao_herdada':t10_cfg, 'protocolo':'blocos_central',
    'limite_metodologico':'OOF por cadeia completa; recursao interna in-sample, nao crossfit aninhado integral'}
t10_resultado = None
print('CNN:',t10_conf['cnn'],'| parâmetros:',t10_conf['parametros'],'| base:',t10_conf['base'])
print('Features estáticas reutilizadas de:',t10_origem)
t10_display(t10_pd.DataFrame([{'conjunto':s,'antes':len(t10_div_original[s]),'pareado_0_1_5_10':len(ids)} for s,ids in t10_div.items()]))
t10_display(t10_pd.DataFrame([{'cadeia':p['nome'],'frames_fit':len(p['pool']),'frames_destino':len(p['alvos'])} for p in t10_planos]))
print('Orçamento: 5 cadeias × 10 etapas + 4 regressores finais = 54 fits. Sem nova extração CNN.')
print('Nenhum treinamento iniciado. Confira a origem e execute T10.2.')


### T10.2 — Conferir isolamento e identidade

Esta célula verifica as divisões, a ausência de sobreposição, os dez predecessores exatos, os blocos excluídos e seus gaps. Calcula uma assinatura incluindo a matriz de features e os índices, para impedir uma retomada com outra base. **Não há parâmetro para mudar aqui e nenhum modelo é treinado.** Ela verifica a estrutura do pipeline, não comprova o alinhamento físico dos rótulos.


In [ ]:
# T10.2 — verificações estruturais e assinatura dos artefatos; nenhum treinamento.
def t10_validar_plano(base, div, hist, planos, gap):
    grupos, frames = base.experimento.to_numpy(str), base.frame.to_numpy()
    usados = t10_np.concatenate(list(div.values()))
    if len(t10_np.unique(usados)) != len(usados):
        raise ValueError('Treino, validação e teste se sobrepõem.')
    for s, ids in div.items():
        if len(ids) < 2 or not (hist[ids] >= 0).all():
            raise ValueError('Partição sem histórico completo: '+s)
        for lag in range(hist.shape[1]):
            if not (grupos[hist[ids,lag]] == grupos[ids]).all() or not (frames[hist[ids,lag]] == frames[ids]-lag-1).all():
                raise ValueError('Histórico atravessou lacuna ou experimento.')
    tr = set(div['treino'])
    recebidos = []
    for p in planos:
        pool = p['pool']
        if not set(pool).issubset(tr):
            raise ValueError('Auxiliar usa dados externos ao treino.')
        if set(pool) & set(p['alvos']):
            raise ValueError('Auxiliar viu o rótulo do destino.')
        for exp, lo, hi in p['purga']:
            historico_pool = t10_np.concatenate([pool, hist[pool].ravel()])
            if ((grupos[historico_pool] == exp) & (frames[historico_pool] >= lo) & (frames[historico_pool] <= hi)).any():
                raise ValueError('Fit ou histórico do fit entra no bloco purgado.')
        if p['nome'] != 'externa':
            recebidos.extend(p['alvos'])
    if sorted(recebidos) != sorted(div['treino'].tolist()):
        raise ValueError('Cada frame de treino deve receber uma única cadeia OOF.')
    if not t10_np.array_equal(t10_np.sort(planos[-1]['alvos']), t10_np.sort(t10_np.r_[div['validacao'],div['teste']])):
        raise ValueError('Destinos externos divergentes.')

def t10_assinatura():
    h = t10_hash.sha256(t10_json.dumps(t10_conf,sort_keys=True).encode())
    for ar in [t10_X, t10_y, t10_hist, *t10_div.values()]:
        h.update(str(ar.shape).encode())
        h.update(str(ar.dtype).encode())
        for inicio in range(0, len(ar), 2048):
            h.update(t10_np.ascontiguousarray(ar[inicio:inicio+2048]).tobytes())
    h.update(t10_base.to_csv(index=False).encode())
    for p in t10_planos:
        h.update(p['pool'].tobytes()); h.update(p['alvos'].tobytes())
    return h.hexdigest()

if T10_COLUNAS != 10 or T10_AVALIAR != (0,1,5,10) or T10_EPOCAS_AUX < 1:
    raise ValueError('Esta primeira implementação foi validada para 10 colunas e avaliações 0/1/5/10.')
t10_validar_plano(t10_base,t10_div,t10_hist,t10_planos,int(t10_cfg['gap_frames']))
t10_selo = t10_assinatura()
print('Validação estrutural OK. Assinatura:',t10_selo[:16])
print('As mesmas linhas serão usadas no estático e nos três modelos temporais.')
print('Importante: cada cadeia mantém seu pool fixo nas dez etapas; não mistura colunas de outras cadeias.')
print('A validação estrutural não comprova o alinhamento físico entre imagens e rótulos.')


### T10.3 — Executar e salvar a cada etapa

Depois de aprovar as informações de T10.1/T10.2, defina `T10_EXECUTAR=True` (por exemplo, numa célula curta imediatamente antes desta) e execute. Serão treinados **50 auxiliares e 4 regressores finais**. O estático pareado também é reajustado, para comparar exatamente os mesmos frames.

Cada etapa salva modelo, scaler, histórico e checkpoint. Se a sessão cair, execute P1, T10.1 e T10.2; informe em `T10_RETOMAR` a pasta impressa anteriormente, defina `T10_EXECUTAR=True` e execute T10.3. A configuração precisa ser idêntica; a última etapa incompleta será refeita. Não sobrescrevemos resultados estáticos nem preliminares.

O teste central avalia interpolação em trechos novos de experimentos conhecidos. Como há treino dos dois lados do teste, **não representa previsão causal do futuro nem generalização para outro experimento**. LOEO e os outros protocolos não são executados nesta primeira rodada.


In [ ]:
# T10.3 — gerar dez colunas e ajustar os quatro modelos finais; desativado por padrão.
def t10_rodar():
    if 't10_selo' not in globals() or t10_assinatura() != t10_selo:
        raise RuntimeError('Dados/configuração mudaram. Reexecute T10.1 e T10.2.')
    import tensorflow as tf10
    from sklearn.preprocessing import StandardScaler as Scaler10
    from sklearn.metrics import mean_absolute_error as mae10, mean_squared_error as mse10, r2_score as r210
    cfg = t10_conf['configuracao_herdada']
    raiz = T10_PILOTO.parent.parent/'temporal_10_central'
    pasta = T10Path(T10_RETOMAR) if T10_RETOMAR else raiz/(t10_conf['cnn']+'_'+T10Datetime.now(T10Timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'))
    manifesto = {'assinatura':t10_selo,'configuracao':t10_conf,'tensorflow':tf10.__version__}
    if T10_RETOMAR:
        if t10_json.loads((pasta/'configuracao.json').read_text()) != manifesto:
            raise ValueError('Retomada incompatível: conteúdo, configuração ou TensorFlow diferente.')
    else:
        pasta.mkdir(parents=True,exist_ok=False)
        (pasta/'configuracao.json').write_text(t10_json.dumps(manifesto,indent=2),encoding='utf-8')
        t10_base.to_csv(pasta/'indice_base.csv',index=False)
        t10_np.savez(pasta/'divisoes.npz',**t10_div)
        t10_np.save(pasta/'indices_historico.npy',t10_hist)
    print('Pasta para acompanhar/retomar:',pasta,flush=True)

    def gravar_npz(destino, **dados):
        temporario = destino.with_name(destino.stem+'.tmp.npz')
        t10_np.savez(temporario,**dados)
        temporario.replace(destino)

    def construir(n, semente):
        tf10.keras.backend.clear_session()
        tf10.keras.utils.set_random_seed(semente)
        m = tf10.keras.Sequential([tf10.keras.layers.Input(shape=(n,)),
            tf10.keras.layers.Dense(32,activation='relu'),tf10.keras.layers.Dense(1)])
        m.compile(optimizer=tf10.keras.optimizers.Adam(learning_rate=cfg['taxa_aprendizado']),loss='mean_absolute_error')
        return m

    def entradas(colunas, ids, k):
        x = t10_np.asarray(t10_X[ids],dtype='float32')
        return t10_np.column_stack([x,colunas[ids,:k]]) if k else x

    temporal = t10_np.full((len(t10_base),10),t10_np.nan,dtype='float32')
    custos = []
    auditoria = []
    relogio = t10_clock()
    for numero, plano in enumerate(t10_planos):
        local = pasta/plano['nome']; local.mkdir(exist_ok=True)
        pool, alvos = plano['pool'], plano['alvos']
        t10_np.savez(local/'indices.npz',pool=pool,alvos=alvos)
        colunas = t10_np.full_like(temporal,t10_np.nan)
        tempos = t10_np.zeros(10,dtype=float)
        etapa = 0
        if (local/'checkpoint.npz').is_file():
            with t10_np.load(local/'checkpoint.npz',allow_pickle=False) as z:
                colunas, tempos, etapa = z['colunas'].copy(),z['tempos'].copy(),int(z['etapa'])
            if colunas.shape != temporal.shape or tempos.shape != (10,) or not 0 <= etapa <= 10:
                raise ValueError('Checkpoint inválido: '+str(local))
            if not t10_np.isfinite(colunas[t10_np.r_[pool,alvos],:etapa]).all():
                raise ValueError('Checkpoint incompleto nos frames utilizados.')
        for k in range(etapa,10):
            # Toda a cadeia usa SOMENTE y[pool]. Nunca recebe rótulos do bloco escondido/validação/teste.
            inicio = t10_clock()
            xt = entradas(colunas,pool,k)
            if not t10_np.isfinite(xt).all():
                raise ValueError('Histórico incompleto no fit auxiliar.')
            scaler = Scaler10().fit(xt)
            modelo = construir(xt.shape[1],int(cfg['semente'])+1000*(numero+1)+k)
            hist = modelo.fit(scaler.transform(xt).astype('float32'),t10_y[pool],
                epochs=t10_conf['epocas_aux'],batch_size=cfg['batch_regressao'],verbose=0,shuffle=True)
            # A coluna k+1 usa a previsão do frame imediatamente anterior, feita pelo modelo com k colunas.
            destinos = t10_np.flatnonzero((t10_hist[:,:k+1] >= 0).all(axis=1))
            fontes = t10_hist[destinos,0]
            xp = entradas(colunas,fontes,k)
            if not t10_np.isfinite(xp).all():
                raise ValueError('Histórico incompleto na geração da coluna.')
            pred = modelo.predict(scaler.transform(xp).astype('float32'),batch_size=cfg['batch_regressao'],verbose=0).ravel()
            if not t10_np.isfinite(pred).all():
                raise ValueError('Auxiliar produziu NaN/Inf.')
            colunas[destinos,k] = pred
            tempos[k] = t10_clock()-inicio
            modelo.save(local/f'auxiliar_{k+1:02d}.keras')
            gravar_npz(local/f'scaler_{k+1:02d}.npz',media=scaler.mean_,escala=scaler.scale_)
            t10_pd.DataFrame(hist.history).to_csv(local/f'historico_{k+1:02d}.csv',index=False)
            gravar_npz(local/'checkpoint.npz',colunas=colunas,tempos=tempos,etapa=t10_np.array(k+1))
            print(f'{plano["nome"]} | coluna {k+1}/10 | {tempos[k]:.1f} s',flush=True)
        temporal[alvos] = colunas[alvos]
        for k, segundos in enumerate(tempos,1):
            custos.append({'cadeia':plano['nome'],'coluna':k,'tempo_auxiliar_s':segundos})
        for idx in alvos:
            auditoria.append({'indice_dataset':int(idx),'cadeia':plano['nome']})
    usados = t10_np.concatenate(list(t10_div.values()))
    if not t10_np.isfinite(temporal[usados]).all():
        raise ValueError('Matriz temporal final incompleta.')
    t10_np.save(pasta/'features_temporais_10.npy',temporal)
    t10_pd.DataFrame(auditoria).to_csv(pasta/'origem_colunas.csv',index=False)
    custos = t10_pd.DataFrame(custos); custos.to_csv(pasta/'custos_auxiliares.csv',index=False)

    linhas, predicoes = [], []
    for k in t10_conf['avaliar']:
        local = pasta/f'final_{k:02d}'; local.mkdir(exist_ok=True)
        if (local/'concluido.json').is_file():
            linhas.extend(t10_pd.read_csv(local/'metricas.csv').to_dict('records'))
            predicoes.append(t10_pd.read_csv(local/'predicoes.csv'))
            print(f'Final {k}: checkpoint concluído reutilizado.',flush=True)
            continue
        tr, va, te = (t10_div[s] for s in ('treino','validacao','teste'))
        xt = entradas(temporal,tr,k)
        scaler = Scaler10().fit(xt)
        modelo = construir(xt.shape[1],int(cfg['semente']))
        parada = tf10.keras.callbacks.EarlyStopping(monitor='val_loss',patience=cfg['paciencia'],restore_best_weights=True)
        inicio = t10_clock()
        hist = modelo.fit(scaler.transform(xt).astype('float32'),t10_y[tr],
            validation_data=(scaler.transform(entradas(temporal,va,k)).astype('float32'),t10_y[va]),
            epochs=cfg['epocas'],batch_size=cfg['batch_regressao'],callbacks=[parada],verbose=0,shuffle=True)
        fit_s = t10_clock()-inicio
        registros, partes = [], []
        for s, ids in t10_div.items():
            yp = modelo.predict(scaler.transform(entradas(temporal,ids,k)).astype('float32'),batch_size=cfg['batch_regressao'],verbose=0).ravel()
            y = t10_y[ids]
            if not t10_np.isfinite(yp).all():
                raise ValueError('Predições finais não finitas.')
            registros.append({'colunas':k,'conjunto':s,'n':len(ids),'MAE_mm':float(mae10(y,yp)),
                'MAPE_pct':float(100*t10_np.mean(t10_np.abs((y-yp)/y))),
                'RMSE_mm':float(t10_np.sqrt(mse10(y,yp))),'R2':float(r210(y,yp)) if t10_np.var(y)>0 else t10_np.nan,
                'epocas':len(hist.history['loss']),'melhor_epoca':int(t10_np.argmin(hist.history['val_loss']))+1,
                'tempo_fit_final_s':fit_s,'tempo_auxiliar_acumulado_s':float(custos.loc[custos.coluna<=k,'tempo_auxiliar_s'].sum())})
            p = t10_base.iloc[ids][['indice_dataset','experimento','frame','y']].copy()
            p['colunas'],p['conjunto'],p['y_predito'] = k,s,yp
            partes.append(p)
        modelo.save(local/'regressor.keras')
        gravar_npz(local/'scaler.npz',media=scaler.mean_,escala=scaler.scale_)
        t10_pd.DataFrame(hist.history).to_csv(local/'historico.csv',index=False)
        t10_pd.DataFrame(registros).to_csv(local/'metricas.csv',index=False)
        t10_pd.concat(partes).to_csv(local/'predicoes.csv',index=False)
        (local/'concluido.json').write_text(t10_json.dumps({'colunas':k,'assinatura':t10_selo}),encoding='utf-8')
        linhas.extend(registros); predicoes.extend(partes)
        print(f'Final {k} colunas concluído | fit {fit_s:.1f} s',flush=True)
    metricas, predicoes = t10_pd.DataFrame(linhas),t10_pd.concat(predicoes,ignore_index=True)
    escolha = metricas.loc[metricas.conjunto=='validacao'].sort_values(['MAE_mm','colunas']).iloc[0]
    metricas.to_csv(pasta/'metricas.csv',index=False)
    predicoes.to_csv(pasta/'predicoes.csv',index=False)
    fim = {'assinatura':t10_selo,'colunas_escolhidas_por_validacao':int(escolha.colunas),
        'criterio':'menor MAE de validacao; empate favorece menor K',
        'fits_planejados':54,'segundos_nesta_chamada':t10_clock()-relogio}
    (pasta/'concluido.json').write_text(t10_json.dumps(fim,indent=2),encoding='utf-8')
    print('Concluído:',pasta)
    return {'pasta':pasta,'metricas':metricas,'predicoes':predicoes,'escolha':int(escolha.colunas),'temporal':temporal}

if T10_EXECUTAR:
    t10_resultado = t10_rodar()
else:
    print('Treinamento desativado. Após T10.2, defina T10_EXECUTAR=True e execute esta célula.')
    print('Para retomar: preencha T10_RETOMAR com a pasta impressa pela execução interrompida.')


### T10.4 — Interpretar os resultados

Mostra MAE, MAPE, RMSE, R², ganho frente ao estático e custo de modelagem. A quantidade de colunas é indicada pelo **menor MAE de validação**, não pelo teste. O gráfico interativo mantém os três painéis (treino, validação e teste), permitindo selecionar experimento e coluna. Nenhum parâmetro de treinamento muda aqui.

Resultados com uma semente são exploratórios. Ganho pequeno exige repetições antes de concluir melhora; empate não prova que os rótulos ou as features estejam corretos. Os custos acumulados compartilham auxiliares e não devem ser somados entre as linhas.


In [ ]:
# T10.4 — resultados principais e o gráfico de y × coluna temporal; sem novo treinamento.
import matplotlib.pyplot as t10_plt
import ipywidgets as t10_widgets
from IPython.display import clear_output as t10_clear
if t10_resultado is None:
    print('Execute T10.3 ou carregue uma pasta concluída com o bloco opcional abaixo.')
else:
    t10_m = t10_resultado['metricas'].copy()
    t10_t = t10_m.loc[t10_m.conjunto=='teste'].copy()
    referencia = float(t10_t.loc[t10_t.colunas==0,'MAE_mm'].iloc[0])
    t10_t['reducao_MAE_pct'] = 100*(referencia-t10_t.MAE_mm)/referencia if referencia > 0 else t10_np.nan
    t10_t['custo_modelagem_s'] = t10_t.tempo_fit_final_s+t10_t.tempo_auxiliar_acumulado_s
    print('TESTE CENTRAL — mesmos frames em 0, 1, 5 e 10 colunas')
    print('Quantidade escolhida SOMENTE pelo MAE de validação:',t10_resultado['escolha'])
    t10_display(t10_t.set_index('colunas')[['MAE_mm','MAPE_pct','RMSE_mm','R2','reducao_MAE_pct','custo_modelagem_s','n']]
        .style.format(precision=4).highlight_min(subset=['MAE_mm'],color='#d6f5df'))
    fig, ax = t10_plt.subplots(figsize=(8,3.4))
    for s in ['validacao','teste']:
        d = t10_m.loc[t10_m.conjunto==s].sort_values('colunas')
        ax.plot(d.colunas,d.MAE_mm,'o-',label=s)
    ax.set(xlabel='Colunas temporais',ylabel='MAE (mm)',xticks=[0,1,5,10],title='Ganho incremental — menor erro é melhor')
    ax.legend(); ax.grid(alpha=.2); fig.tight_layout(); t10_plt.show(); t10_plt.close(fig)
    print('Custo: auxiliares até K + fit final; exclui extração CNN e I/O de persistência.')
    print('Não somar os custos acumulados das linhas: auxiliares anteriores são compartilhados.')
    print('Execução exploratória, uma semente. Não escolher K repetidamente pelo teste.')

    t10_exp_widget = t10_widgets.Dropdown(options=sorted(t10_base.experimento.unique()),description='Experimento:')
    t10_k_widget = t10_widgets.Dropdown(options=list(range(1,11)),value=10,description='Coluna:')
    t10_saida = t10_widgets.Output()
    def t10_desenhar(_=None):
        with t10_saida:
            t10_clear(wait=True)
            fig, axes = t10_plt.subplots(3,1,figsize=(11,7),sharex=True,sharey=True)
            for ax,s in zip(axes,['treino','validacao','teste']):
                ids = t10_div[s]
                ids = ids[t10_base.experimento.to_numpy()[ids]==t10_exp_widget.value]
                ids = ids[t10_np.argsort(t10_base.frame.to_numpy()[ids])]
                ax.plot(t10_base.frame.to_numpy()[ids],t10_y[ids],'.',ms=2.5,color='#17324d',label='y real do frame atual')
                ax.plot(t10_base.frame.to_numpy()[ids],t10_resultado['temporal'][ids,t10_k_widget.value-1],'.',ms=2.5,
                    alpha=.65,color='#d17b21',label=f'Coluna {t10_k_widget.value}: previsão deslocada de t−1')
                ax.set_title(s,loc='left'); ax.set_ylabel('Largura (mm)'); ax.grid(alpha=.2); ax.legend(fontsize=8)
            axes[-1].set_xlabel('Número do frame — lacunas preservadas')
            fig.suptitle('Teste central | '+t10_exp_widget.value+' | qualidade da coluna, não previsão final')
            fig.tight_layout(); t10_plt.show(); t10_plt.close(fig)
    t10_exp_widget.observe(t10_desenhar,names='value'); t10_k_widget.observe(t10_desenhar,names='value')
    t10_display(t10_widgets.VBox([t10_widgets.HBox([t10_exp_widget,t10_k_widget]),t10_saida])); t10_desenhar()
    print('Resultados salvos em:',t10_resultado['pasta'])

# Opcional, em outra sessão: execute P1, T10.1 e T10.2; descomente para ler sem treinar.
# pasta10 = T10Path('/content/drive/MyDrive/TCC/resultados/cnn_new_estatico_v2/temporal_10_central/SUA_EXECUCAO')
# fim10 = t10_json.loads((pasta10/'concluido.json').read_text())
# if fim10['assinatura'] != t10_selo:
#     raise ValueError('Execução não corresponde à configuração/base carregada.')
# t10_resultado = {'pasta':pasta10,'metricas':t10_pd.read_csv(pasta10/'metricas.csv'),
#     'predicoes':t10_pd.read_csv(pasta10/'predicoes.csv'),
#     'escolha':fim10['colunas_escolhidas_por_validacao'],
#     'temporal':t10_np.load(pasta10/'features_temporais_10.npy',allow_pickle=False)}
# Depois execute novamente T10.4, mantendo o bloco de carregamento comentado.


## 14 — Extensão temporal: 50 e 100 colunas, sementes 42 e 43

**Ordem desta rodada:** gerar e avaliar até 100 colunas com a semente **42**; em seguida repetir toda a geração e o ajuste com a **43**. Comparações em **0, 1, 5, 10, 50 e 100 colunas**, sempre nos mesmos frames. Mantemos o teste central, a CNN/base/uso de parâmetros da execução de origem, os gaps, o regressor e as 50 épocas fixas dos auxiliares.

**Por que recalcular os menores K?** Exigir 100 predecessores reduz as linhas disponíveis e altera os pools auxiliares. Para separar esse efeito da quantidade de colunas, refazemos também o estático e K=1/5/10 nesse conjunto comum. A seção 13 e seus resultados permanecem preservados, mas não são a referência pareada desta nova tabela. Não reutilizamos checkpoints de dez colunas.

São **duas sementes × (500 auxiliares + 6 regressores finais) = 1.012 fits**. A CNN não é executada novamente. Cada nova coluna nasce da previsão de t−1 de um auxiliar que já recebe as colunas anteriores; não são 100 imagens diretamente na entrada, nem 100 rótulos reais passados. Mantém-se a limitação documentada: cadeias isoladas por bloco, com recursão in-sample dentro de cada pool, sem cross-fitting integralmente aninhado.

### T100.1 — Carregar e conferir a origem

Em sessão nova, execute **P1 da seção preliminar**, depois **T100.1 → T100.2 → T100.3 → T100.4**. Não execute a parte estática nem T10.3 para iniciar este lote. Confira a CNN, a base e `parâmetros` na saída. `TX_EXECUTAR=True` está habilitado para a execução solicitada; **não use “Executar tudo”**, pois outras seções também contêm treinamentos.

Para retomar uma interrupção, preencha `TX_RETOMAR` com a **pasta do lote** impressa em T100.3 (não a subpasta `seed_42`). Os caminhos, hashes e versões precisam coincidir. Não altere hiperparâmetros no meio da execução.


In [ ]:
# T100.1 — mesma base, 100 colunas e sementes 42/43; preparação sem treinar.
# Pré-requisito: execute somente P1 da seção preliminar para definir ab_ler e montar o Drive.
if 'ab_ler' not in globals():
    raise RuntimeError('Execute P1 da seção preliminar primeiro. Não precisa executar o estático nem P2–P6.')
from pathlib import Path as TXPath
from datetime import datetime as TXDatetime, timezone as TXTimezone
from time import perf_counter as tx_clock
import json as tx_json
import hashlib as tx_hash
import numpy as tx_np
import pandas as tx_pd
from IPython.display import display as tx_display

TX_PILOTO = TXPath('/content/drive/MyDrive/TCC/resultados/cnn_new_estatico_v2/temporal_pilotos/DenseNet169_20261014T204217_183926Z')
TX_COLUNAS = 100
TX_AVALIAR = (0, 1, 5, 10, 50, 100)
TX_SEMENTES = (42, 43)
TX_EPOCAS_AUX = 50  # Orçamento fixo pré-declarado; não escolhido olhando o teste.
TX_RETOMAR = "/content/drive/MyDrive/TCC/resultados/cnn_new_estatico_v2/temporal_100_central/DenseNet169_20261019T011057_736729Z"  # Pasta do LOTE 100 colunas para retomar, não a pasta antiga de 10 colunas.
TX_EXECUTAR = True  # Execução longa autorizada: semente 42, depois 43, automaticamente.

tx_cfg, tx_base, tx_X, tx_y, _, _, _, tx_fold, tx_origem = ab_ler(TX_PILOTO)
with tx_np.load(tx_fold/'divisao_original.npz', allow_pickle=False) as z:
    tx_div_original = {s: z[s].copy() for s in ('treino','validacao','teste')}
tx_trace_original = tx_pd.read_csv(tx_fold/'rastreabilidade_temporal_1.csv')
for tx_s, tx_ids in tx_div_original.items():
    if not tx_np.array_equal(tx_np.sort(tx_ids),tx_np.flatnonzero(tx_trace_original.conjunto==tx_s)):
        raise ValueError('Divisão original diverge da rastreabilidade: '+tx_s)

def tx_historico(base, div, profundidade):
    """Índices t-1...t-K: exatos, mesmo experimento e mesma partição externa."""
    n = len(base)
    grupos, frames = base.experimento.to_numpy(str), base.frame.to_numpy()
    if not tx_np.array_equal(frames, frames.astype(int)):
        raise ValueError('Frame não inteiro.')
    dono = tx_np.full(n, '', dtype=object)
    for nome, ids in div.items():
        if (ids.dtype.kind not in 'iu' or (ids < 0).any() or (ids >= n).any()
            or len(tx_np.unique(ids)) != len(ids) or (dono[ids] != '').any()):
            raise ValueError('Índices externos inválidos ou sobrepostos.')
        dono[ids] = nome
    lookup = {(str(e), int(f)): i for i, (e, f) in enumerate(zip(grupos, frames))}
    if len(lookup) != n:
        raise ValueError('Experimento/frame duplicado.')
    hist = tx_np.full((n, profundidade), -1, dtype=int)
    for i in tx_np.flatnonzero(dono != ''):
        for lag in range(1, profundidade+1):
            j = lookup.get((grupos[i], int(frames[i])-lag), -1)
            if j < 0 or dono[j] != dono[i]:
                break
            hist[i, lag-1] = j
    return hist, dono

def tx_planejar(base, div, hist, gap):
    """Quatro cadeias OOF: 2 partes de cada um dos 2 trechos de treino."""
    grupos, frames = base.experimento.to_numpy(str), base.frame.to_numpy()
    treino = div['treino']
    elegiveis = treino[(hist[treino] >= 0).all(axis=1)]
    alvos_por_cadeia = {k: [] for k in range(4)}
    for exp in sorted(set(grupos[treino])):
        ids = treino[grupos[treino] == exp]
        ids = ids[tx_np.argsort(frames[ids])]
        trechos = tx_np.split(ids, tx_np.flatnonzero(tx_np.diff(frames[ids]) != 1)+1)
        if len(trechos) != 2:
            raise ValueError(f'{exp}: esperado treino central em dois trechos contíguos.')
        for r, trecho in enumerate(trechos):
            for b, pedaco in enumerate(tx_np.array_split(trecho, 2)):
                alvo = tx_np.intersect1d(pedaco, elegiveis)
                if not len(alvo):
                    raise ValueError('Bloco sem 100 predecessores disponíveis.')
                alvos_por_cadeia[2*r+b].append(alvo)
    planos = []
    for k, partes in alvos_por_cadeia.items():
        alvos = tx_np.concatenate(partes)
        permitido = tx_np.zeros(len(base), dtype=bool)
        permitido[treino] = True
        intervalos = []
        for exp in sorted(set(grupos[alvos])):
            a = alvos[grupos[alvos] == exp]
            lo, hi = int(frames[hist[a, -1]].min())-gap, int(frames[a].max())+gap
            permitido[(grupos == exp) & (frames >= lo) & (frames <= hi)] = False
            intervalos.append([str(exp), lo, hi])
        # Todos os 100 predecessores dos frames usados em fit também ficam no pool permitido.
        pool = elegiveis[permitido[elegiveis] & permitido[hist[elegiveis]].all(axis=1)]
        if len(pool) < 100 or set(grupos[pool]) != set(grupos[treino]):
            raise ValueError('Pool auxiliar insuficiente depois de purgar o bloco e seu histórico.')
        planos.append({'nome':f'oof_{k+1}', 'pool':pool, 'alvos':alvos, 'purga':intervalos})
    planos.append({'nome':'externa', 'pool':elegiveis, 'alvos':tx_np.concatenate([
        ids[(hist[ids] >= 0).all(axis=1)] for s, ids in div.items() if s != 'treino']), 'purga':[]})
    return planos

tx_hist, tx_dono = tx_historico(tx_base, tx_div_original, TX_COLUNAS)
tx_div = {s: ids[(tx_hist[ids] >= 0).all(axis=1)] for s, ids in tx_div_original.items()}
tx_planos = tx_planejar(tx_base, tx_div_original, tx_hist, int(tx_cfg['gap_frames']))
tx_conf = {'versao':'cadeias_isoladas_100_duas_sementes_v1', 'sementes_planejadas':list(TX_SEMENTES), 'piloto':str(TX_PILOTO), 'origem_estatica':str(tx_origem),
    'base':tx_cfg['dataset_id'], 'cnn':tx_cfg['arquitetura'], 'parametros':tx_cfg['parametros_processo'],
    'colunas':TX_COLUNAS, 'avaliar':list(TX_AVALIAR), 'epocas_aux':TX_EPOCAS_AUX,
    'configuracao_herdada':tx_cfg, 'protocolo':'blocos_central',
    'limite_metodologico':'OOF por cadeia completa; recursao interna in-sample, nao crossfit aninhado integral'}
tx_resultados = {}
tx_lote = None
print('CNN:',tx_conf['cnn'],'| parâmetros:',tx_conf['parametros'],'| base:',tx_conf['base'])
print('Features estáticas reutilizadas de:',tx_origem)
tx_display(tx_pd.DataFrame([{'conjunto':s,'antes':len(tx_div_original[s]),'pareado_ate_100':len(ids)} for s,ids in tx_div.items()]))
tx_display(tx_pd.DataFrame([{'cadeia':p['nome'],'frames_fit':len(p['pool']),'frames_destino':len(p['alvos'])} for p in tx_planos]))
print('Orçamento: 2 sementes × (5 cadeias × 100 etapas + 6 finais) = 1.012 fits. Sem nova extração CNN.')
print('Nenhum treinamento iniciado. Confira a origem e execute T100.2.')


### T100.2 — Validação separada antes do treinamento

Confere os 100 predecessores exatos, as divisões externas, o isolamento dos blocos internos e os pools auxiliares. Calcula uma assinatura da configuração, matriz CNN, rótulos e índices. **Não treina; não há parâmetro a ajustar aqui.** Falhas interrompem o processo, sem reduzir gaps ou realinhar linhas silenciosamente.


In [ ]:
# T100.2 — verificações estruturais e assinatura dos artefatos; nenhum treinamento.
def tx_validar_plano(base, div, hist, planos, gap):
    grupos, frames = base.experimento.to_numpy(str), base.frame.to_numpy()
    usados = tx_np.concatenate(list(div.values()))
    if len(tx_np.unique(usados)) != len(usados):
        raise ValueError('Treino, validação e teste se sobrepõem.')
    for s, ids in div.items():
        if len(ids) < 2 or not (hist[ids] >= 0).all():
            raise ValueError('Partição sem histórico completo: '+s)
        for lag in range(hist.shape[1]):
            if not (grupos[hist[ids,lag]] == grupos[ids]).all() or not (frames[hist[ids,lag]] == frames[ids]-lag-1).all():
                raise ValueError('Histórico atravessou lacuna ou experimento.')
    tr = set(div['treino'])
    recebidos = []
    for p in planos:
        pool = p['pool']
        if not set(pool).issubset(tr):
            raise ValueError('Auxiliar usa dados externos ao treino.')
        if set(pool) & set(p['alvos']):
            raise ValueError('Auxiliar viu o rótulo do destino.')
        for exp, lo, hi in p['purga']:
            historico_pool = tx_np.concatenate([pool, hist[pool].ravel()])
            if ((grupos[historico_pool] == exp) & (frames[historico_pool] >= lo) & (frames[historico_pool] <= hi)).any():
                raise ValueError('Fit ou histórico do fit entra no bloco purgado.')
        if p['nome'] != 'externa':
            recebidos.extend(p['alvos'])
    if sorted(recebidos) != sorted(div['treino'].tolist()):
        raise ValueError('Cada frame de treino deve receber uma única cadeia OOF.')
    if not tx_np.array_equal(tx_np.sort(planos[-1]['alvos']), tx_np.sort(tx_np.r_[div['validacao'],div['teste']])):
        raise ValueError('Destinos externos divergentes.')

def tx_assinatura():
    h = tx_hash.sha256(tx_json.dumps(tx_conf,sort_keys=True).encode())
    for ar in [tx_X, tx_y, tx_hist, *tx_div.values()]:
        h.update(str(ar.shape).encode())
        h.update(str(ar.dtype).encode())
        for inicio in range(0, len(ar), 2048):
            h.update(tx_np.ascontiguousarray(ar[inicio:inicio+2048]).tobytes())
    h.update(tx_base.to_csv(index=False).encode())
    for p in tx_planos:
        h.update(p['pool'].tobytes()); h.update(p['alvos'].tobytes())
    return h.hexdigest()

if TX_COLUNAS != 100 or TX_AVALIAR != (0,1,5,10,50,100) or TX_SEMENTES != (42,43) or TX_EPOCAS_AUX < 1:
    raise ValueError('Plano esperado: 100 colunas, avaliações 0/1/5/10/50/100, sementes 42 e 43.')
tx_validar_plano(tx_base,tx_div,tx_hist,tx_planos,int(tx_cfg['gap_frames']))
tx_selo = tx_assinatura()
print('Validação estrutural OK. Assinatura:',tx_selo[:16])
print('As mesmas linhas serão usadas no estático e nos cinco modelos temporais.')
print('Importante: cada cadeia mantém seu pool fixo nas 100 etapas; não mistura colunas de outras cadeias.')
print('A validação estrutural não comprova o alinhamento físico entre imagens e rótulos.')


### T100.3 — Execução longa, com retomada automática das etapas concluídas

Executa a semente 42 e depois a 43, sem intervenção entre elas. Salva checkpoint **a cada coluna** e registra progresso a cada dez. Ao concluir uma semente, salva e mostra suas métricas antes de começar a próxima. O teste não é usado para treinar auxiliares nem para EarlyStopping; a validação seleciona a época do regressor final e indica K.

Se houver interrupção, copie a pasta do lote para `TX_RETOMAR` em T100.1, execute novamente T100.1/T100.2 e esta célula. As etapas completas são reaproveitadas; somente a última incompleta é refeita. Não há promessa de continuidade se o Colab encerrar a sessão. **Não desligue nem reinicie o runtime durante a execução.**

Os modelos e métricas desta extensão são salvos em `temporal_100_central`, sem sobrescrever os resultados anteriores. São resultados exploratórios com duas sementes, não um teste confirmatório independente após todas as análises já realizadas.


In [ ]:
# T100.3 — motor com checkpoints por coluna e execução sequencial das duas sementes.
def tx_rodar(conf, pasta, assinatura, retomar=False):
    if 'tx_selo' not in globals() or tx_assinatura() != tx_selo:
        raise RuntimeError('Dados/configuração mudaram. Reexecute T100.1 e T100.2.')
    import tensorflow as tf10
    from sklearn.preprocessing import StandardScaler as Scaler10
    from sklearn.metrics import mean_absolute_error as mae10, mean_squared_error as mse10, r2_score as r210
    cfg = conf['configuracao_herdada']
    profundidade = int(conf['colunas'])
    manifesto = {'assinatura':assinatura,'configuracao':conf,'tensorflow':tf10.__version__}
    if retomar:
        if tx_json.loads((pasta/'configuracao.json').read_text()) != manifesto:
            raise ValueError('Retomada incompatível: conteúdo, configuração ou TensorFlow diferente.')
    else:
        pasta.mkdir(parents=True,exist_ok=False)
        (pasta/'configuracao.json').write_text(tx_json.dumps(manifesto,indent=2),encoding='utf-8')
        tx_base.to_csv(pasta/'indice_base.csv',index=False)
        tx_np.savez(pasta/'divisoes.npz',**tx_div)
        tx_np.save(pasta/'indices_historico.npy',tx_hist)
    print('Pasta para acompanhar/retomar:',pasta,flush=True)

    def gravar_npz(destino, **dados):
        temporario = destino.with_name(destino.stem+'.tmp.npz')
        tx_np.savez(temporario,**dados)
        temporario.replace(destino)

    def construir(n, semente):
        tf10.keras.backend.clear_session()
        tf10.keras.utils.set_random_seed(semente)
        m = tf10.keras.Sequential([tf10.keras.layers.Input(shape=(n,)),
            tf10.keras.layers.Dense(32,activation='relu'),tf10.keras.layers.Dense(1)])
        m.compile(optimizer=tf10.keras.optimizers.Adam(learning_rate=cfg['taxa_aprendizado']),loss='mean_absolute_error')
        return m

    def entradas(colunas, ids, k):
        x = tx_np.asarray(tx_X[ids],dtype='float32')
        return tx_np.column_stack([x,colunas[ids,:k]]) if k else x

    temporal = tx_np.full((len(tx_base),profundidade),tx_np.nan,dtype='float32')
    custos = []
    auditoria = []
    relogio = tx_clock()
    for numero, plano in enumerate(tx_planos):
        local = pasta/plano['nome']; local.mkdir(exist_ok=True)
        pool, alvos = plano['pool'], plano['alvos']
        tx_np.savez(local/'indices.npz',pool=pool,alvos=alvos)
        colunas = tx_np.full_like(temporal,tx_np.nan)
        tempos = tx_np.zeros(profundidade,dtype=float)
        etapa = 0
        if (local/'checkpoint.npz').is_file():
            with tx_np.load(local/'checkpoint.npz',allow_pickle=False) as z:
                colunas, tempos, etapa = z['colunas'].copy(),z['tempos'].copy(),int(z['etapa'])
            if colunas.shape != temporal.shape or tempos.shape != (profundidade,) or not 0 <= etapa <= profundidade:
                raise ValueError('Checkpoint inválido: '+str(local))
            if not tx_np.isfinite(colunas[tx_np.r_[pool,alvos],:etapa]).all():
                raise ValueError('Checkpoint incompleto nos frames utilizados.')
        for k in range(etapa,profundidade):
            # Toda a cadeia usa SOMENTE y[pool]. Nunca recebe rótulos do bloco escondido/validação/teste.
            inicio = tx_clock()
            xt = entradas(colunas,pool,k)
            if not tx_np.isfinite(xt).all():
                raise ValueError('Histórico incompleto no fit auxiliar.')
            scaler = Scaler10().fit(xt)
            modelo = construir(xt.shape[1],int(cfg['semente'])+1000*(numero+1)+k)
            hist = modelo.fit(scaler.transform(xt).astype('float32'),tx_y[pool],
                epochs=conf['epocas_aux'],batch_size=cfg['batch_regressao'],verbose=0,shuffle=True)
            # A coluna k+1 usa a previsão do frame imediatamente anterior, feita pelo modelo com k colunas.
            destinos = tx_np.flatnonzero((tx_hist[:,:k+1] >= 0).all(axis=1))
            fontes = tx_hist[destinos,0]
            xp = entradas(colunas,fontes,k)
            if not tx_np.isfinite(xp).all():
                raise ValueError('Histórico incompleto na geração da coluna.')
            pred = modelo.predict(scaler.transform(xp).astype('float32'),batch_size=cfg['batch_regressao'],verbose=0).ravel()
            if not tx_np.isfinite(pred).all():
                raise ValueError('Auxiliar produziu NaN/Inf.')
            colunas[destinos,k] = pred
            tempos[k] = tx_clock()-inicio
            modelo.save(local/f'auxiliar_{k+1:02d}.keras')
            gravar_npz(local/f'scaler_{k+1:02d}.npz',media=scaler.mean_,escala=scaler.scale_)
            tx_pd.DataFrame(hist.history).to_csv(local/f'historico_{k+1:02d}.csv',index=False)
            gravar_npz(local/'checkpoint.npz',colunas=colunas,tempos=tempos,etapa=tx_np.array(k+1))
            if (k+1) % 10 == 0 or k == 0:
                print(f'seed={cfg["semente"]} | {plano["nome"]} | coluna {k+1}/{profundidade} | {tempos[k]:.1f} s',flush=True)
        temporal[alvos] = colunas[alvos]
        for k, segundos in enumerate(tempos,1):
            custos.append({'cadeia':plano['nome'],'coluna':k,'tempo_auxiliar_s':segundos})
        for idx in alvos:
            auditoria.append({'indice_dataset':int(idx),'cadeia':plano['nome']})
    usados = tx_np.concatenate(list(tx_div.values()))
    if not tx_np.isfinite(temporal[usados]).all():
        raise ValueError('Matriz temporal final incompleta.')
    tx_np.save(pasta/'features_temporais.npy',temporal)
    tx_pd.DataFrame(auditoria).to_csv(pasta/'origem_colunas.csv',index=False)
    custos = tx_pd.DataFrame(custos); custos.to_csv(pasta/'custos_auxiliares.csv',index=False)

    linhas, predicoes = [], []
    for k in conf['avaliar']:
        local = pasta/f'final_{k:02d}'; local.mkdir(exist_ok=True)
        if (local/'concluido.json').is_file():
            linhas.extend(tx_pd.read_csv(local/'metricas.csv').to_dict('records'))
            predicoes.append(tx_pd.read_csv(local/'predicoes.csv'))
            print(f'Final {k}: checkpoint concluído reutilizado.',flush=True)
            continue
        tr, va, te = (tx_div[s] for s in ('treino','validacao','teste'))
        xt = entradas(temporal,tr,k)
        scaler = Scaler10().fit(xt)
        modelo = construir(xt.shape[1],int(cfg['semente']))
        parada = tf10.keras.callbacks.EarlyStopping(monitor='val_loss',patience=cfg['paciencia'],restore_best_weights=True)
        inicio = tx_clock()
        hist = modelo.fit(scaler.transform(xt).astype('float32'),tx_y[tr],
            validation_data=(scaler.transform(entradas(temporal,va,k)).astype('float32'),tx_y[va]),
            epochs=cfg['epocas'],batch_size=cfg['batch_regressao'],callbacks=[parada],verbose=0,shuffle=True)
        fit_s = tx_clock()-inicio
        registros, partes = [], []
        for s, ids in tx_div.items():
            yp = modelo.predict(scaler.transform(entradas(temporal,ids,k)).astype('float32'),batch_size=cfg['batch_regressao'],verbose=0).ravel()
            y = tx_y[ids]
            if not tx_np.isfinite(yp).all():
                raise ValueError('Predições finais não finitas.')
            registros.append({'colunas':k,'conjunto':s,'n':len(ids),'MAE_mm':float(mae10(y,yp)),
                'MAPE_pct':float(100*tx_np.mean(tx_np.abs((y-yp)/y))),
                'RMSE_mm':float(tx_np.sqrt(mse10(y,yp))),'R2':float(r210(y,yp)) if tx_np.var(y)>0 else tx_np.nan,
                'epocas':len(hist.history['loss']),'melhor_epoca':int(tx_np.argmin(hist.history['val_loss']))+1,
                'tempo_fit_final_s':fit_s,'tempo_auxiliar_acumulado_s':float(custos.loc[custos.coluna<=k,'tempo_auxiliar_s'].sum())})
            p = tx_base.iloc[ids][['indice_dataset','experimento','frame','y']].copy()
            p['colunas'],p['conjunto'],p['y_predito'] = k,s,yp
            partes.append(p)
        modelo.save(local/'regressor.keras')
        gravar_npz(local/'scaler.npz',media=scaler.mean_,escala=scaler.scale_)
        tx_pd.DataFrame(hist.history).to_csv(local/'historico.csv',index=False)
        tx_pd.DataFrame(registros).to_csv(local/'metricas.csv',index=False)
        tx_pd.concat(partes).to_csv(local/'predicoes.csv',index=False)
        (local/'concluido.json').write_text(tx_json.dumps({'colunas':k,'assinatura':assinatura}),encoding='utf-8')
        linhas.extend(registros); predicoes.extend(partes)
        print(f'Final {k} colunas concluído | fit {fit_s:.1f} s',flush=True)
    metricas, predicoes = tx_pd.DataFrame(linhas),tx_pd.concat(predicoes,ignore_index=True)
    metricas['semente'] = int(cfg['semente'])
    predicoes['semente'] = int(cfg['semente'])
    escolha = metricas.loc[metricas.conjunto=='validacao'].sort_values(['MAE_mm','colunas']).iloc[0]
    metricas.to_csv(pasta/'metricas.csv',index=False)
    predicoes.to_csv(pasta/'predicoes.csv',index=False)
    fim = {'assinatura':assinatura,'colunas_escolhidas_por_validacao':int(escolha.colunas),
        'criterio':'menor MAE de validacao; empate favorece menor K',
        'fits_planejados':len(tx_planos)*profundidade+len(conf['avaliar']),'segundos_nesta_chamada':tx_clock()-relogio}
    (pasta/'concluido.json').write_text(tx_json.dumps(fim,indent=2),encoding='utf-8')
    print('Concluído:',pasta)
    return {'pasta':pasta,'metricas':metricas,'predicoes':predicoes,'escolha':int(escolha.colunas),'temporal':temporal}

def tx_executar_lote():
    global tx_lote, tx_resultados
    if 'tx_selo' not in globals() or tx_assinatura() != tx_selo:
        raise RuntimeError('Execute T100.1 e T100.2 antes. Dados/configuração não podem mudar.')
    import tensorflow as tf100
    tx_resultados = {}
    sementes = tuple(tx_conf['sementes_planejadas'])
    manifesto = {'assinatura':tx_selo,'tensorflow':tf100.__version__,
        'sementes':list(sementes),'configuracao':tx_conf}
    if TX_RETOMAR:
        pasta = TXPath(TX_RETOMAR)
    elif tx_lote is not None:
        pasta = TXPath(tx_lote)
    else:
        pasta = TXPath(str(TX_PILOTO.parent.parent))/'temporal_100_central'/(
            tx_conf['cnn']+'_'+TXDatetime.now(TXTimezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'))
    if pasta.exists():
        if tx_json.loads((pasta/'lote.json').read_text()) != manifesto:
            raise ValueError('Lote incompatível. Não reutilize pastas da versão com 10 colunas.')
    else:
        if TX_RETOMAR:
            raise FileNotFoundError('Pasta de retomada não encontrada: '+str(pasta))
        pasta.mkdir(parents=True,exist_ok=False)
        (pasta/'lote.json').write_text(tx_json.dumps(manifesto,indent=2),encoding='utf-8')
    tx_lote = pasta
    print('LOTE — copie este caminho para TX_RETOMAR se houver desconexão:',pasta,flush=True)
    print('Ordem automática: seed 42, depois seed 43. Mesma base, mesmos frames, CNN e hiperparâmetros.',flush=True)
    for seed in sementes:
        conf = tx_json.loads(tx_json.dumps(tx_conf))
        conf['configuracao_herdada']['semente'] = int(seed)
        conf['semente_execucao'] = int(seed)
        assinatura = tx_hash.sha256((tx_selo+tx_json.dumps(conf,sort_keys=True)).encode()).hexdigest()
        destino = pasta/f'seed_{seed}'
        print(f'\nINICIANDO SEMENTE {seed}',flush=True)
        resultado = tx_rodar(conf,destino,assinatura,retomar=destino.exists())
        tx_resultados[seed] = resultado
        # As métricas parciais permitem consultar a primeira semente enquanto a segunda roda.
        parcial = tx_pd.concat([r['metricas'] for r in tx_resultados.values()],ignore_index=True)
        parcial.to_csv(pasta/'metricas_sementes.csv',index=False)
        escolhas = {str(s):r['escolha'] for s,r in tx_resultados.items()}
        (pasta/'escolhas_por_validacao.json').write_text(tx_json.dumps(escolhas,indent=2),encoding='utf-8')
        tx_display(resultado['metricas'].loc[resultado['metricas'].conjunto=='teste',
            ['semente','colunas','MAE_mm','MAPE_pct','RMSE_mm','R2','n']].round(4))
        print('Escolha pela validação nesta semente:',resultado['escolha'],flush=True)
    (pasta/'concluido.json').write_text(tx_json.dumps({'assinatura':tx_selo,
        'sementes_concluidas':list(tx_resultados),'fits_planejados':1012},indent=2),encoding='utf-8')
    print('DUAS SEMENTES CONCLUÍDAS. Execute T100.4 para o resumo.',flush=True)
    return tx_resultados

if TX_EXECUTAR:
    tx_resultados = tx_executar_lote()
else:
    print('Desativado. Defina TX_EXECUTAR=True após conferir T100.1/T100.2.')


### T100.4 — Comparar quantidade de colunas e estabilidade

Mostra as métricas de cada semente, a redução de MAE frente ao seu estático pareado e média/desvio entre sementes. O gráfico compara MAE de validação e teste; o painel de largura permite escolher semente, experimento e coluna. **Não treina modelos.** Pode ser enfileirada após T100.3 para aparecer ao final.

Olhe primeiro se o ganho aparece nas duas sementes e se compensa o custo adicional. A escolha de K usa validação, não o menor erro de teste. Duas sementes dão uma indicação de estabilidade, mas não demonstram significância estatística. Os custos por K são acumulados e compartilham auxiliares: não some as linhas.


In [ ]:
# T100.4 — comparação pareada e estabilidade entre sementes; não treina.
import matplotlib.pyplot as tx_plt
import ipywidgets as tx_widgets
from IPython.display import clear_output as tx_clear
if not tx_resultados:
    print('Ainda não há uma semente concluída nesta sessão. Execute T100.3 ou retome o lote.')
else:
    tx_metricas = tx_pd.concat([r['metricas'] for r in tx_resultados.values()],ignore_index=True)
    tx_teste = tx_metricas.loc[tx_metricas.conjunto=='teste'].copy()
    tx_ref = tx_teste.loc[tx_teste.colunas==0].set_index('semente').MAE_mm
    tx_teste['reducao_MAE_pct'] = [100*(tx_ref.loc[s]-v)/tx_ref.loc[s] if tx_ref.loc[s]>0 else tx_np.nan
        for s,v in zip(tx_teste.semente,tx_teste.MAE_mm)]
    tx_teste['custo_modelagem_s'] = tx_teste.tempo_fit_final_s+tx_teste.tempo_auxiliar_acumulado_s
    print('TESTE CENTRAL — mesmos frames em todas as quantidades e sementes')
    tx_display(tx_teste.set_index(['semente','colunas'])[
        ['MAE_mm','MAPE_pct','RMSE_mm','R2','reducao_MAE_pct','custo_modelagem_s','n']].style.format(precision=4))
    tx_resumo = tx_teste.groupby('colunas').agg(
        sementes=('semente','nunique'),MAE_medio_mm=('MAE_mm','mean'),MAE_desvio_mm=('MAE_mm','std'),
        MAPE_medio_pct=('MAPE_pct','mean'),RMSE_medio_mm=('RMSE_mm','mean'),
        R2_medio_sementes=('R2','mean'),reducao_MAE_media_pct=('reducao_MAE_pct','mean'))
    tx_display(tx_resumo.style.format(precision=4))
    print('Duas sementes: média/desvio são descritivos; não provam significância ou generalização.')
    print('R² médio acima é média ENTRE SEMENTES no mesmo teste, não um R² de previsões agrupadas.')
    tx_va = tx_metricas.loc[tx_metricas.conjunto=='validacao'].groupby('colunas').MAE_mm.mean()
    print('Menor MAE médio de validação (não do teste):',int(tx_va.sort_values(kind='stable').index[0]),'colunas')
    print('Custos acumulam auxiliares até K e fit final; sem CNN. Não somar as linhas por K.')
    tx_resumo.to_csv(tx_lote/'resumo_sementes.csv')
    fig, axes = tx_plt.subplots(1,2,figsize=(11,3.8),sharey=True)
    for ax,conjunto in zip(axes,['validacao','teste']):
        for seed,d in tx_metricas.loc[tx_metricas.conjunto==conjunto].groupby('semente'):
            d=d.sort_values('colunas')
            ax.plot(d.colunas,d.MAE_mm,'o-',label=f'Semente {seed}')
        ax.set(xlabel='Colunas temporais',title=conjunto,xticks=[0,10,50,100])
        ax.grid(alpha=.2); ax.legend()
    axes[0].set_ylabel('MAE (mm) — menor é melhor')
    fig.tight_layout(); tx_plt.show(); tx_plt.close(fig)

    tx_seed_widget = tx_widgets.Dropdown(options=sorted(tx_resultados),description='Semente:')
    tx_exp_widget = tx_widgets.Dropdown(options=sorted(tx_base.experimento.unique()),description='Experimento:')
    tx_k_widget = tx_widgets.Dropdown(options=[1,5,10,50,100],value=100,description='Coluna:')
    tx_saida = tx_widgets.Output()
    def tx_desenhar(_=None):
        with tx_saida:
            tx_clear(wait=True)
            r=tx_resultados[tx_seed_widget.value]
            fig, axes=tx_plt.subplots(3,1,figsize=(11,7),sharex=True,sharey=True)
            for ax,s in zip(axes,['treino','validacao','teste']):
                ids=tx_div[s]
                ids=ids[tx_base.experimento.to_numpy()[ids]==tx_exp_widget.value]
                ids=ids[tx_np.argsort(tx_base.frame.to_numpy()[ids])]
                ax.plot(tx_base.frame.to_numpy()[ids],tx_y[ids],'.',ms=2.5,color='#17324d',label='y real atual')
                ax.plot(tx_base.frame.to_numpy()[ids],r['temporal'][ids,tx_k_widget.value-1],'.',ms=2.5,
                    alpha=.65,color='#d17b21',label=f'Coluna {tx_k_widget.value}: previsão deslocada de t−1')
                ax.set_title(s,loc='left'); ax.set_ylabel('Largura (mm)'); ax.grid(alpha=.2); ax.legend(fontsize=8)
            axes[-1].set_xlabel('Número do frame — lacunas preservadas')
            fig.suptitle(f'Teste central | {tx_exp_widget.value} | seed {tx_seed_widget.value} | qualidade da coluna')
            fig.tight_layout(); tx_plt.show(); tx_plt.close(fig)
    for w in [tx_seed_widget,tx_exp_widget,tx_k_widget]: w.observe(tx_desenhar,names='value')
    tx_display(tx_widgets.VBox([tx_widgets.HBox([tx_seed_widget,tx_exp_widget]),tx_k_widget,tx_saida])); tx_desenhar()
    print('Pasta do lote:',tx_lote)


# 15 — Comparação temporal entre extratores

Nova campanha independente, sem sobrescrever as seções anteriores. **Teste central por blocos**: 30% no meio; validação nas pontas (10% + 10%); treino nos dois trechos restantes, com gap de 100 frames. Compara **0, 1, 5, 10, 25, 50 e 100 colunas**, sementes **42 e 43**. O estático (K=0) é retreinado nos mesmos frames elegíveis até K=100. Não é uma repetição de LOEO nem do split aleatório.

Cada coluna vem de uma previsão auxiliar deslocada de t−1; a rede auxiliar é retreinada em cada etapa. Quatro cadeias geram as entradas de treino com blocos excluídos, e uma cadeia externa gera validação/teste. As cadeias mantêm pools fixos; a recursão interna é in-sample, **não cross-fitting aninhado integral**.

### TC1 — Escolher as extrações estáticas

Monte o Drive e selecione **uma execução por CNN** no seletor (Ctrl/Cmd + clique para várias). Depois das novas extrações estáticas, reexecute TC1 para atualizar a lista. Ajuste `TC_ROOT` se mudar a pasta e `TC_PARAMETROS` para incluir ou não os parâmetros em todas as CNNs. Os hiperparâmetros estão explícitos. `TC_EXECUTAR=False` mantém o treinamento desligado. Não precisa executar P1 nem T100.

Esta seção reutiliza somente as features CNN salvas, não modelos/colunas temporais antigos. Cada CNN exige até **1.014 fits**. Checkpoints permitem retomar a própria campanha.

In [ ]:
# TC1 — configurar campanha. Não extrai features nem treina modelos.
from pathlib import Path as TCPath
from datetime import datetime as TCDatetime, timezone as TCTimezone
from time import perf_counter as tc_clock
import json as tc_json
import hashlib as tc_hash
import numpy as tc_np
import pandas as tc_pd
import ipywidgets as tc_widgets
from IPython.display import display as tc_display, clear_output as tc_clear
from google.colab import drive as tc_drive
tc_drive.mount('/content/drive')

TC_ROOT = TCPath('/content/drive/MyDrive/TCC/resultados/cnn_new_estatico_v2')
TC_AVALIAR = (0, 1, 5, 10, 25, 50, 100)
TC_SEMENTES = (42, 43)
TC_PARAMETROS = False  # True inclui current, nozzle_speed e mass_flow_rate em TODOS os extratores.
TC_GAP = 100
TC_EPOCAS_AUX = 50
TC_EPOCAS = 300
TC_PACIENCIA = 20
TC_BATCH = 128
TC_LR = 0.001


TC_EXECUTAR = True  # Mude para True somente quando quiser iniciar TC3.
TC_RETOMAR = '/content/drive/MyDrive/TCC/resultados/cnn_new_estatico_v2/temporal_comparacao/20261021T203131_239418Z'  # Para retomar, cole o caminho CAMPANHA impresso por TC3.

tc_opcoes, tc_avisos = [], []
if not (TC_ROOT/'execucoes').is_dir():
    raise FileNotFoundError('Pasta execucoes ausente: confira TC_ROOT e a montagem do Drive.')
for tc_p in sorted((TC_ROOT/'execucoes').iterdir()):
    if not tc_p.is_dir():
        continue
    try:
        if not all((tc_p/n).is_file() for n in ['features.npy','indice_features.csv','configuracao.json','custo_extrator.csv']):
            continue
        tc_c = tc_json.loads((tc_p/'configuracao.json').read_text())
        tc_opcoes.append((f"{tc_c['arquitetura']} | base={tc_c['dataset_id']} | {tc_p.name}",str(tc_p)))
    except (ValueError, KeyError, OSError) as exc:
        tc_avisos.append(f'{tc_p.name}: {exc}')
if not tc_opcoes:
    raise ValueError('Nenhuma extração estática completa encontrada. Execute e salve a extração primeiro.')
tc_fontes = tc_widgets.SelectMultiple(options=tc_opcoes, value=(),
    description='Extrações:', rows=min(12,len(tc_opcoes)), layout=tc_widgets.Layout(width='100%'))
# Na retomada, recupere a seleção original sem escolher outra extração por engano.
if TC_RETOMAR:
    tc_manifesto_retomada = tc_json.loads((TCPath(TC_RETOMAR)/'campanha.json').read_text())
    tc_fontes.value = tuple(j['configuracao']['origem_estatica'] for j in tc_manifesto_retomada['jobs'])
    print('Fontes recuperadas da campanha:', TC_RETOMAR)
tc_display(tc_fontes)
print('Selecione uma extração por CNN (Ctrl/Cmd + clique permite várias). Confira a base e a data.')
print('TC2 exigirá os mesmos frames/rótulos nas fontes. TC_PARAMETROS controla a inclusão dos parâmetros, independentemente do ensaio estático anterior.')
print('Cada CNN: 2 × (5 cadeias × 100 auxiliares + 7 regressores finais) = 1.014 fits. Sem nova extração CNN.')
print('Nova campanha não reutiliza treinamentos antigos; retomada somente com TC_RETOMAR explícito.')
tc_jobs = []
tc_campanha = None
if tc_avisos:
    print('\n'.join(tc_avisos))


### TC2 — Preparação e validação da comparação

Execute depois de escolher as fontes em TC1. Confere identidade e ordem dos frames, rótulos, parâmetros e matriz CNN; constrói a divisão central e os históricos exatos. Exige a mesma base entre CNNs e usa os mesmos exemplos para todos os K e sementes. Gera uma assinatura de rastreabilidade. **Não treina**; não ajuste índices manualmente para contornar uma falha. A validação estrutural não comprova o alinhamento físico imagem–largura.

In [ ]:
# TC2 — funções e validação separada; não treina.
def tc_historico(base, div, profundidade):
    """Índices t-1...t-K: exatos, mesmo experimento e mesma partição externa."""
    n = len(base)
    grupos, frames = base.experimento.to_numpy(str), base.frame.to_numpy()
    if not tc_np.array_equal(frames, frames.astype(int)):
        raise ValueError('Frame não inteiro.')
    dono = tc_np.full(n, '', dtype=object)
    for nome, ids in div.items():
        if (ids.dtype.kind not in 'iu' or (ids < 0).any() or (ids >= n).any()
            or len(tc_np.unique(ids)) != len(ids) or (dono[ids] != '').any()):
            raise ValueError('Índices externos inválidos ou sobrepostos.')
        dono[ids] = nome
    lookup = {(str(e), int(f)): i for i, (e, f) in enumerate(zip(grupos, frames))}
    if len(lookup) != n:
        raise ValueError('Experimento/frame duplicado.')
    hist = tc_np.full((n, profundidade), -1, dtype=int)
    for i in tc_np.flatnonzero(dono != ''):
        for lag in range(1, profundidade+1):
            j = lookup.get((grupos[i], int(frames[i])-lag), -1)
            if j < 0 or dono[j] != dono[i]:
                break
            hist[i, lag-1] = j
    return hist, dono

def tc_planejar(base, div, hist, gap):
    """Quatro cadeias OOF: 2 partes de cada um dos 2 trechos de treino."""
    grupos, frames = base.experimento.to_numpy(str), base.frame.to_numpy()
    treino = div['treino']
    elegiveis = treino[(hist[treino] >= 0).all(axis=1)]
    alvos_por_cadeia = {k: [] for k in range(4)}
    for exp in sorted(set(grupos[treino])):
        ids = treino[grupos[treino] == exp]
        ids = ids[tc_np.argsort(frames[ids])]
        trechos = tc_np.split(ids, tc_np.flatnonzero(tc_np.diff(frames[ids]) != 1)+1)
        if len(trechos) != 2:
            raise ValueError(f'{exp}: esperado treino central em dois trechos contíguos.')
        for r, trecho in enumerate(trechos):
            for b, pedaco in enumerate(tc_np.array_split(trecho, 2)):
                alvo = tc_np.intersect1d(pedaco, elegiveis)
                if not len(alvo):
                    raise ValueError('Bloco sem 100 predecessores disponíveis.')
                alvos_por_cadeia[2*r+b].append(alvo)
    planos = []
    for k, partes in alvos_por_cadeia.items():
        alvos = tc_np.concatenate(partes)
        permitido = tc_np.zeros(len(base), dtype=bool)
        permitido[treino] = True
        intervalos = []
        for exp in sorted(set(grupos[alvos])):
            a = alvos[grupos[alvos] == exp]
            lo, hi = int(frames[hist[a, -1]].min())-gap, int(frames[a].max())+gap
            permitido[(grupos == exp) & (frames >= lo) & (frames <= hi)] = False
            intervalos.append([str(exp), lo, hi])
        # Todos os 100 predecessores dos frames usados em fit também ficam no pool permitido.
        pool = elegiveis[permitido[elegiveis] & permitido[hist[elegiveis]].all(axis=1)]
        if len(pool) < 100 or set(grupos[pool]) != set(grupos[treino]):
            raise ValueError('Pool auxiliar insuficiente depois de purgar o bloco e seu histórico.')
        planos.append({'nome':f'oof_{k+1}', 'pool':pool, 'alvos':alvos, 'purga':intervalos})
    planos.append({'nome':'externa', 'pool':elegiveis, 'alvos':tc_np.concatenate([
        ids[(hist[ids] >= 0).all(axis=1)] for s, ids in div.items() if s != 'treino']), 'purga':[]})
    return planos

def tc_validar_plano(base, div, hist, planos, gap):
    grupos, frames = base.experimento.to_numpy(str), base.frame.to_numpy()
    usados = tc_np.concatenate(list(div.values()))
    if len(tc_np.unique(usados)) != len(usados):
        raise ValueError('Treino, validação e teste se sobrepõem.')
    for s, ids in div.items():
        if len(ids) < 2 or not (hist[ids] >= 0).all():
            raise ValueError('Partição sem histórico completo: '+s)
        for lag in range(hist.shape[1]):
            if not (grupos[hist[ids,lag]] == grupos[ids]).all() or not (frames[hist[ids,lag]] == frames[ids]-lag-1).all():
                raise ValueError('Histórico atravessou lacuna ou experimento.')
    tr = set(div['treino'])
    recebidos = []
    for p in planos:
        pool = p['pool']
        if not set(pool).issubset(tr):
            raise ValueError('Auxiliar usa dados externos ao treino.')
        if set(pool) & set(p['alvos']):
            raise ValueError('Auxiliar viu o rótulo do destino.')
        for exp, lo, hi in p['purga']:
            historico_pool = tc_np.concatenate([pool, hist[pool].ravel()])
            if ((grupos[historico_pool] == exp) & (frames[historico_pool] >= lo) & (frames[historico_pool] <= hi)).any():
                raise ValueError('Fit ou histórico do fit entra no bloco purgado.')
        if p['nome'] != 'externa':
            recebidos.extend(p['alvos'])
    if sorted(recebidos) != sorted(div['treino'].tolist()):
        raise ValueError('Cada frame de treino deve receber uma única cadeia OOF.')
    if not tc_np.array_equal(tc_np.sort(planos[-1]['alvos']), tc_np.sort(tc_np.r_[div['validacao'],div['teste']])):
        raise ValueError('Destinos externos divergentes.')

def tc_assinatura(conf, base, X, y, hist, div, planos):
    h = tc_hash.sha256(tc_json.dumps(conf,sort_keys=True).encode())
    for ar in [X, y, hist, *div.values()]:
        h.update(str(ar.shape).encode())
        h.update(str(ar.dtype).encode())
        for inicio in range(0, len(ar), 2048):
            h.update(tc_np.ascontiguousarray(ar[inicio:inicio+2048]).tobytes())
    h.update(base.to_csv(index=False).encode())
    for p in planos:
        h.update(p['pool'].tobytes()); h.update(p['alvos'].tobytes())
    return h.hexdigest()

def tc_central(base, gap):
    grupos, frames = base.experimento.to_numpy(str), base.frame.to_numpy()
    partes = {s:[] for s in ['treino','validacao','teste']}
    for exp in sorted(set(grupos)):
        ids = tc_np.flatnonzero(grupos==exp)
        ids = ids[tc_np.argsort(frames[ids],kind='stable')]
        if len(ids)<10 or not (tc_np.diff(frames[ids])==1).all():
            raise ValueError(f'{exp}: este protocolo exige frames consecutivos, sem duplicação/lacuna.')
        a,b,c,d = [int(len(ids)*v) for v in (.10,.35,.65,.90)]
        va,te = tc_np.r_[ids[:a],ids[d:]],ids[b:c]
        left,right = ids[a:b],ids[c:d]
        left=left[(frames[left]>frames[ids[a-1]]+gap)&(frames[left]<frames[te[0]]-gap)]
        right=right[(frames[right]>frames[te[-1]]+gap)&(frames[right]<frames[ids[d]]-gap)]
        if min(len(left),len(right),len(va),len(te))<2:
            raise ValueError(f'{exp}: insuficiente após gaps; não reduzimos o gap automaticamente.')
        for s,v in zip(partes,[tc_np.r_[left,right],va,te]): partes[s].append(v)
    return {s:tc_np.concatenate(v) for s,v in partes.items()}

def tc_comparabilidade(ctx, regras):
    h=tc_hash.sha256(tc_json.dumps(regras,sort_keys=True).encode())
    # Same targets, order, covariates, split and hyperparameters; CNN features intentionally excluded.
    h.update(ctx['base'][TC_IDENTIDADE].to_csv(index=False,float_format='%.12g').encode())
    for s in ['treino','validacao','teste']:
        h.update(s.encode()); h.update(ctx['div'][s].astype('int64').tobytes())
    return h.hexdigest()

TC_IDENTIDADE=['indice_dataset','experimento','frame','caminho_relativo','y','current','nozzle_speed','mass_flow_rate']
if TC_AVALIAR!=(0,1,5,10,25,50,100) or TC_SEMENTES!=(42,43):
    raise ValueError('Plano esperado: 0/1/5/10/25/50/100, sementes 42/43.')
if type(TC_PARAMETROS) is not bool or TC_GAP<1 or min(TC_EPOCAS_AUX,TC_EPOCAS,TC_PACIENCIA,TC_BATCH)<1:
    raise ValueError('Configuração inválida.')
tc_selecionadas=list(tc_fontes.value)
if not tc_selecionadas:
    raise ValueError('Selecione as extrações em TC1 antes de executar TC2.')
tc_regras={'versao':'temporal_comparacao_v1','protocolo':'blocos_central',
    'teste':[.35,.65],'validacao':[[0,.10],[.90,1]],'gap':TC_GAP,'colunas':100,
    'avaliar':list(TC_AVALIAR),'sementes':list(TC_SEMENTES),'parametros':TC_PARAMETROS,
    'epocas_aux':TC_EPOCAS_AUX,'epocas':TC_EPOCAS,'paciencia':TC_PACIENCIA,
    'batch':TC_BATCH,'lr':TC_LR,'regressor':'Dense(32,relu)-Dense(1)',
    'padronizacao':'StandardScaler_fit_somente_treino',
    'limite':'OOF por cadeia completa; recursao interna in-sample, nao crossfit aninhado integral'}
tc_jobs=[]
for tc_fonte in tc_selecionadas:
    tc_p=TCPath(tc_fonte)
    tc_cfg=tc_json.loads((tc_p/'configuracao.json').read_text())
    tc_base=tc_pd.read_csv(tc_p/'indice_features.csv')
    if not tc_np.array_equal(tc_base.indice_dataset,tc_np.arange(len(tc_base))) or tc_base.duplicated(['experimento','frame']).any():
        raise ValueError(f'{tc_p.name}: identidade/ordem inválida.')
    if not tc_np.isfinite(tc_base[['frame','y','current','nozzle_speed','mass_flow_rate']].to_numpy()).all() or not (tc_base.y>0).all():
        raise ValueError('Valores ausentes/não finitos ou y não positivo.')
    if tc_jobs:
        tc_pd.testing.assert_frame_equal(tc_jobs[0]['base'][TC_IDENTIDADE],tc_base[TC_IDENTIDADE],check_dtype=False,check_exact=True)
    tc_custo=tc_pd.read_csv(tc_p/'custo_extrator.csv').iloc[0]
    tc_X=tc_np.load(tc_p/'features.npy',mmap_mode='r',allow_pickle=False)
    if tc_X.shape!=(len(tc_base),int(tc_custo.features_cnn)) or tc_custo.arquitetura!=tc_cfg['arquitetura']:
        raise ValueError('Features incompatíveis com índice/custo/configuração.')
    if not tc_np.isfinite(tc_X).all(): raise ValueError('Features contêm NaN/Inf.')
    if TC_PARAMETROS:
        tc_X=tc_np.column_stack([tc_X,tc_base[['current','nozzle_speed','mass_flow_rate']].to_numpy('float32')])
    tc_y=tc_base.y.to_numpy('float32')
    tc_original=tc_central(tc_base,TC_GAP)
    tc_hist,_=tc_historico(tc_base,tc_original,100)
    tc_div={s:ids[(tc_hist[ids]>=0).all(axis=1)] for s,ids in tc_original.items()}
    tc_planos=tc_planejar(tc_base,tc_original,tc_hist,TC_GAP)
    tc_validar_plano(tc_base,tc_div,tc_hist,tc_planos,TC_GAP)
    tc_conf={'versao':'temporal_comparacao_v1','cnn':tc_cfg['arquitetura'],
        'base':tc_cfg['dataset_id'],'origem_estatica':str(tc_p),'parametros':TC_PARAMETROS,
        'colunas':100,'avaliar':list(TC_AVALIAR),'epocas_aux':TC_EPOCAS_AUX,
        'regras':tc_regras,'configuracao_herdada':{'semente':42,'epocas':TC_EPOCAS,
        'paciencia':TC_PACIENCIA,'batch_regressao':TC_BATCH,'taxa_aprendizado':TC_LR}}
    tc_ctx={'base':tc_base,'X':tc_X,'y':tc_y,'hist':tc_hist,'div':tc_div,'planos':tc_planos,'conf':tc_conf}
    tc_ctx['grupo']=tc_comparabilidade(tc_ctx,tc_regras)
    tc_conf['grupo_comparacao']=tc_ctx['grupo']
    tc_ctx['assinatura']=tc_assinatura(tc_conf,tc_base,tc_X,tc_y,tc_hist,tc_div,tc_planos)
    if any(j['conf']['cnn']==tc_conf['cnn'] for j in tc_jobs):
        raise ValueError('Selecione somente uma extração de cada CNN por campanha.')
    tc_jobs.append(tc_ctx)
if len({j['grupo'] for j in tc_jobs})!=1:
    raise ValueError('Fontes têm dados/divisões não comparáveis.')
tc_selo_preparacao=tc_hash.sha256(tc_json.dumps([j['assinatura'] for j in tc_jobs]).encode()).hexdigest()
tc_display(tc_pd.DataFrame([{'CNN':j['conf']['cnn'],'base':j['conf']['base'],
    'parametros':TC_PARAMETROS,'entradas':j['X'].shape[1],**{s:len(ids) for s,ids in j['div'].items()},
    'origem':j['conf']['origem_estatica']} for j in tc_jobs]))
print('VALIDADO — mesmo teste, validação, treino e histórico de 100 frames para todos os K e CNNs.')
print('Grupo comparável:',tc_jobs[0]['grupo'][:16],'| fits planejados:',1014*len(tc_jobs))
print('Não comprova alinhamento físico dos rótulos. Nenhum treinamento iniciado.')


### TC3 — Rodar a campanha (execução longa)

Após TC2 concluir, defina `TC_EXECUTAR=True` em TC1 e reexecute TC1/TC2, mantendo a seleção, ou execute `TC_EXECUTAR=True` em uma célula antes desta. Esta célula percorre cada CNN selecionada e as sementes 42/43, gera 100 colunas iterativamente e avalia os sete K. Scalers são ajustados apenas no treino; EarlyStopping dos regressores finais usa a validação.

Para retomar após desconexão: execute TC1, selecione as mesmas fontes/configurações, preencha `TC_RETOMAR` com o caminho **CAMPANHA** e execute TC2/TC3. Não edite modelos salvos; configuração e TensorFlow precisam coincidir. Não use Executar tudo. O resultado de teste não deve orientar ajuste de K; a escolha registrada usa MAE de validação.

In [ ]:
# TC3 — geração iterativa, treinamentos finais e checkpoints.
def tc_rodar(ctx, conf, pasta, assinatura, retomar=False):
    tc_base, tc_X, tc_y = ctx['base'], ctx['X'], ctx['y']
    tc_hist, tc_div, tc_planos = ctx['hist'], ctx['div'], ctx['planos']
    import tensorflow as tf10
    from sklearn.preprocessing import StandardScaler as Scaler10
    from sklearn.metrics import mean_absolute_error as mae10, mean_squared_error as mse10, r2_score as r210
    cfg = conf['configuracao_herdada']
    profundidade = int(conf['colunas'])
    manifesto = {'assinatura':assinatura,'configuracao':conf,'tensorflow':tf10.__version__}
    if retomar:
        if tc_json.loads((pasta/'configuracao.json').read_text()) != manifesto:
            raise ValueError('Retomada incompatível: conteúdo, configuração ou TensorFlow diferente.')
    else:
        pasta.mkdir(parents=True,exist_ok=False)
        (pasta/'configuracao.json').write_text(tc_json.dumps(manifesto,indent=2),encoding='utf-8')
        tc_base.to_csv(pasta/'indice_base.csv',index=False)
        tc_np.savez(pasta/'divisoes.npz',**tc_div)
        tc_np.save(pasta/'indices_historico.npy',tc_hist)
    print('Pasta para acompanhar/retomar:',pasta,flush=True)

    def gravar_npz(destino, **dados):
        temporario = destino.with_name(destino.stem+'.tmp.npz')
        tc_np.savez(temporario,**dados)
        temporario.replace(destino)

    def construir(n, semente):
        tf10.keras.backend.clear_session()
        tf10.keras.utils.set_random_seed(semente)
        m = tf10.keras.Sequential([tf10.keras.layers.Input(shape=(n,)),
            tf10.keras.layers.Dense(32,activation='relu'),tf10.keras.layers.Dense(1)])
        m.compile(optimizer=tf10.keras.optimizers.Adam(learning_rate=cfg['taxa_aprendizado']),loss='mean_absolute_error')
        return m

    def entradas(colunas, ids, k):
        x = tc_np.asarray(tc_X[ids],dtype='float32')
        return tc_np.column_stack([x,colunas[ids,:k]]) if k else x

    temporal = tc_np.full((len(tc_base),profundidade),tc_np.nan,dtype='float32')
    custos = []
    auditoria = []
    relogio = tc_clock()
    for numero, plano in enumerate(tc_planos):
        local = pasta/plano['nome']; local.mkdir(exist_ok=True)
        pool, alvos = plano['pool'], plano['alvos']
        tc_np.savez(local/'indices.npz',pool=pool,alvos=alvos)
        colunas = tc_np.full_like(temporal,tc_np.nan)
        tempos = tc_np.zeros(profundidade,dtype=float)
        etapa = 0
        if (local/'checkpoint.npz').is_file():
            with tc_np.load(local/'checkpoint.npz',allow_pickle=False) as z:
                colunas, tempos, etapa = z['colunas'].copy(),z['tempos'].copy(),int(z['etapa'])
            if colunas.shape != temporal.shape or tempos.shape != (profundidade,) or not 0 <= etapa <= profundidade:
                raise ValueError('Checkpoint inválido: '+str(local))
            if not tc_np.isfinite(colunas[tc_np.r_[pool,alvos],:etapa]).all():
                raise ValueError('Checkpoint incompleto nos frames utilizados.')
        print(f"{conf['cnn']} | seed={cfg['semente']} | {plano['nome']} | checkpoint {etapa}/{profundidade}",flush=True)
        for k in range(etapa,profundidade):
            # Toda a cadeia usa SOMENTE y[pool]. Nunca recebe rótulos do bloco escondido/validação/teste.
            inicio = tc_clock()
            xt = entradas(colunas,pool,k)
            if not tc_np.isfinite(xt).all():
                raise ValueError('Histórico incompleto no fit auxiliar.')
            scaler = Scaler10().fit(xt)
            modelo = construir(xt.shape[1],int(cfg['semente'])+1000*(numero+1)+k)
            hist = modelo.fit(scaler.transform(xt).astype('float32'),tc_y[pool],
                epochs=conf['epocas_aux'],batch_size=cfg['batch_regressao'],verbose=0,shuffle=True)
            # A coluna k+1 usa a previsão do frame imediatamente anterior, feita pelo modelo com k colunas.
            destinos = tc_np.flatnonzero((tc_hist[:,:k+1] >= 0).all(axis=1))
            fontes = tc_hist[destinos,0]
            xp = entradas(colunas,fontes,k)
            if not tc_np.isfinite(xp).all():
                raise ValueError('Histórico incompleto na geração da coluna.')
            pred = modelo.predict(scaler.transform(xp).astype('float32'),batch_size=cfg['batch_regressao'],verbose=0).ravel()
            if not tc_np.isfinite(pred).all():
                raise ValueError('Auxiliar produziu NaN/Inf.')
            colunas[destinos,k] = pred
            tempos[k] = tc_clock()-inicio
            modelo.save(local/f'auxiliar_{k+1:02d}.keras')
            gravar_npz(local/f'scaler_{k+1:02d}.npz',media=scaler.mean_,escala=scaler.scale_)
            tc_pd.DataFrame(hist.history).to_csv(local/f'historico_{k+1:02d}.csv',index=False)
            gravar_npz(local/'checkpoint.npz',colunas=colunas,tempos=tempos,etapa=tc_np.array(k+1))
            if (k+1) % 10 == 0 or k == 0:
                print(f'seed={cfg["semente"]} | {plano["nome"]} | coluna {k+1}/{profundidade} | {tempos[k]:.1f} s',flush=True)
        temporal[alvos] = colunas[alvos]
        for k, segundos in enumerate(tempos,1):
            custos.append({'cadeia':plano['nome'],'coluna':k,'tempo_auxiliar_s':segundos})
        for idx in alvos:
            auditoria.append({'indice_dataset':int(idx),'cadeia':plano['nome']})
    usados = tc_np.concatenate(list(tc_div.values()))
    if not tc_np.isfinite(temporal[usados]).all():
        raise ValueError('Matriz temporal final incompleta.')
    tc_np.save(pasta/'features_temporais.npy',temporal)
    tc_pd.DataFrame(auditoria).to_csv(pasta/'origem_colunas.csv',index=False)
    custos = tc_pd.DataFrame(custos); custos.to_csv(pasta/'custos_auxiliares.csv',index=False)

    linhas, predicoes = [], []
    for k in conf['avaliar']:
        local = pasta/f'final_{k:02d}'; local.mkdir(exist_ok=True)
        if (local/'concluido.json').is_file():
            linhas.extend(tc_pd.read_csv(local/'metricas.csv').to_dict('records'))
            predicoes.append(tc_pd.read_csv(local/'predicoes.csv'))
            print(f'Final {k}: checkpoint concluído reutilizado.',flush=True)
            continue
        tr, va, te = (tc_div[s] for s in ('treino','validacao','teste'))
        xt = entradas(temporal,tr,k)
        scaler = Scaler10().fit(xt)
        modelo = construir(xt.shape[1],int(cfg['semente']))
        parada = tf10.keras.callbacks.EarlyStopping(monitor='val_loss',patience=cfg['paciencia'],restore_best_weights=True)
        inicio = tc_clock()
        hist = modelo.fit(scaler.transform(xt).astype('float32'),tc_y[tr],
            validation_data=(scaler.transform(entradas(temporal,va,k)).astype('float32'),tc_y[va]),
            epochs=cfg['epocas'],batch_size=cfg['batch_regressao'],callbacks=[parada],verbose=0,shuffle=True)
        fit_s = tc_clock()-inicio
        registros, partes = [], []
        for s, ids in tc_div.items():
            yp = modelo.predict(scaler.transform(entradas(temporal,ids,k)).astype('float32'),batch_size=cfg['batch_regressao'],verbose=0).ravel()
            y = tc_y[ids]
            if not tc_np.isfinite(yp).all():
                raise ValueError('Predições finais não finitas.')
            registros.append({'colunas':k,'conjunto':s,'n':len(ids),'MAE_mm':float(mae10(y,yp)),
                'MAPE_pct':float(100*tc_np.mean(tc_np.abs((y-yp)/y))),
                'RMSE_mm':float(tc_np.sqrt(mse10(y,yp))),'R2':float(r210(y,yp)) if tc_np.var(y)>0 else tc_np.nan,
                'epocas':len(hist.history['loss']),'melhor_epoca':int(tc_np.argmin(hist.history['val_loss']))+1,
                'tempo_fit_final_s':fit_s,'tempo_auxiliar_acumulado_s':float(custos.loc[custos.coluna<=k,'tempo_auxiliar_s'].sum())})
            p = tc_base.iloc[ids][['indice_dataset','experimento','frame','y']].copy()
            p['colunas'],p['conjunto'],p['y_predito'] = k,s,yp
            partes.append(p)
        modelo.save(local/'regressor.keras')
        gravar_npz(local/'scaler.npz',media=scaler.mean_,escala=scaler.scale_)
        tc_pd.DataFrame(hist.history).to_csv(local/'historico.csv',index=False)
        tc_pd.DataFrame(registros).to_csv(local/'metricas.csv',index=False)
        tc_pd.concat(partes).to_csv(local/'predicoes.csv',index=False)
        (local/'concluido.json').write_text(tc_json.dumps({'colunas':k,'assinatura':assinatura}),encoding='utf-8')
        linhas.extend(registros); predicoes.extend(partes)
        print(f'Final {k} colunas concluído | fit {fit_s:.1f} s',flush=True)
    metricas, predicoes = tc_pd.DataFrame(linhas),tc_pd.concat(predicoes,ignore_index=True)
    metricas['semente'] = int(cfg['semente'])
    predicoes['semente'] = int(cfg['semente'])
    escolha = metricas.loc[metricas.conjunto=='validacao'].sort_values(['MAE_mm','colunas']).iloc[0]
    metricas.to_csv(pasta/'metricas.csv',index=False)
    predicoes.to_csv(pasta/'predicoes.csv',index=False)
    fim = {'assinatura':assinatura,'colunas_escolhidas_por_validacao':int(escolha.colunas),
        'criterio':'menor MAE de validacao; empate favorece menor K',
        'fits_planejados':len(tc_planos)*profundidade+len(conf['avaliar']),'segundos_nesta_chamada':tc_clock()-relogio}
    (pasta/'concluido.json').write_text(tc_json.dumps(fim,indent=2),encoding='utf-8')
    print('Concluído:',pasta)
    return {'pasta':pasta,'metricas':metricas,'predicoes':predicoes,'escolha':int(escolha.colunas),'temporal':temporal}

def tc_executar_campanha():
    global tc_campanha
    import tensorflow as tf
    if not tc_jobs: raise RuntimeError('Execute TC1 e TC2 primeiro.')
    for ctx in tc_jobs:
        atual=tc_assinatura(ctx['conf'],ctx['base'],ctx['X'],ctx['y'],ctx['hist'],ctx['div'],ctx['planos'])
        if atual!=ctx['assinatura']: raise ValueError('Dados/configuração mudaram após TC2.')
    sementes=tc_jobs[0]['conf']['regras']['sementes']
    manifesto={'versao':1,'tensorflow':tf.__version__,'sementes':sementes,
        'jobs':[{'cnn':j['conf']['cnn'],'assinatura':j['assinatura'],'configuracao':j['conf']} for j in tc_jobs]}
    if TC_RETOMAR:
        pasta=TCPath(TC_RETOMAR)
    elif tc_campanha is not None:
        pasta=tc_campanha
    else:
        pasta=TC_ROOT/'temporal_comparacao'/TCDatetime.now(TCTimezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    if pasta.exists():
        if tc_json.loads((pasta/'campanha.json').read_text())!=manifesto:
            raise ValueError('Campanha incompatível. Restaure seleção/configuração/versão do TensorFlow para retomar.')
    else:
        if TC_RETOMAR: raise FileNotFoundError('TC_RETOMAR não existe.')
        pasta.mkdir(parents=True,exist_ok=False)
        (pasta/'campanha.json').write_text(tc_json.dumps(manifesto,indent=2),encoding='utf-8')
    tc_campanha=pasta
    print('CAMPANHA — copie para TC_RETOMAR em caso de desconexão:',pasta,flush=True)
    for ctx in tc_jobs:
        cnn=ctx['conf']['cnn']
        slug=''.join(c for c in cnn if c.isalnum())
        for seed in sementes:
            conf=tc_json.loads(tc_json.dumps(ctx['conf']))
            conf['configuracao_herdada']['semente']=seed
            sig=tc_hash.sha256((ctx['assinatura']+tc_json.dumps(conf,sort_keys=True)).encode()).hexdigest()
            destino=pasta/slug/f'seed_{seed}'
            print(f'\n{cnn} | seed {seed}',flush=True)
            resultado=tc_rodar(ctx,conf,destino,sig,retomar=destino.exists())
            tc_display(resultado['metricas'].query("conjunto=='teste'")[['semente','colunas','MAE_mm','RMSE_mm','R2']].round(4))
            print('K escolhido na validação:',resultado['escolha'],flush=True)
            del resultado
    (pasta/'concluido.json').write_text(tc_json.dumps({'assinaturas':[j['assinatura'] for j in tc_jobs]}),encoding='utf-8')
    print('CAMPANHA CONCLUÍDA. Execute TC4 para comparar as execuções salvas.')
    return pasta

if TC_EXECUTAR:
    tc_campanha=tc_executar_campanha()
else:
    print('Treinamento DESATIVADO. Prepare TC1/TC2 e defina TC_EXECUTAR=True para executar TC3.')


### TC4 — Comparação de todas as novas execuções temporais

Lê as campanhas salvas em `temporal_comparacao`; **não treina**. Execute novamente para atualizar após concluir uma semente. Escolha o grupo comparável (base, parâmetros, divisões e hiperparâmetros), a semente ou média 42/43 e o conjunto. A média exige as duas sementes completas da mesma execução. Campanhas antigas das seções anteriores permanecem preservadas, mas não são misturadas nesta comparação padronizada.

Três gráficos: **MAE × K**, **variação percentual frente ao estático da mesma CNN/semente** e **variação frente ao nível anterior de K** (0→1→5→10→25→50→100). Negativo = redução/melhora; positivo = aumento/piora. Os percentuais são calculados por semente antes da média. Curvas identificam CNN, parâmetros True/False e execução; duas sementes não provam significância estatística.

In [ ]:
# TC4 — lê todas as campanhas desta seção; não treina nem altera resultados.
from pathlib import Path as TCPath
import json as tc_json
import hashlib as tc_hash
import numpy as tc_np
import pandas as tc_pd
import matplotlib.pyplot as tc_plt
import ipywidgets as tc_widgets
from IPython.display import display as tc_display, clear_output as tc_clear, HTML as TCHTML
from html import escape as tc_escape

TC_HIST_ROOT=TCPath('/content/drive/MyDrive/TCC/resultados/cnn_new_estatico_v2/temporal_comparacao')
TC_KS=[0,1,5,10,25,50,100]

def tc_percentuais(d):
    """Positive = MAE increased (worse); negative = MAE decreased (better). Per seed/run."""
    out=[]
    for _,g in d.groupby(['execucao','CNN','semente','conjunto'],sort=False):
        g=g.sort_values('colunas').copy()
        if g.colunas.tolist()!=TC_KS: raise ValueError('Série incompleta de K; não calcular saltos silenciosamente.')
        ref=float(g.loc[g.colunas==0,'MAE_mm'].iloc[0])
        g['variacao_vs_estatico_pct']=100*(g.MAE_mm/ref-1) if ref>0 else tc_np.nan
        anterior=g.MAE_mm.shift(1)
        g['K_anterior']=g.colunas.shift(1)
        g['variacao_vs_anterior_pct']=100*(g.MAE_mm/anterior-1).where(anterior>0)
        out.append(g)
    return tc_pd.concat(out,ignore_index=True) if out else tc_pd.DataFrame()

tc_linhas,tc_pendentes=[],[]
if not TC_HIST_ROOT.is_dir():
    print('Ainda não existem campanhas novas salvas. O painel aparecerá após a primeira semente concluir.')
else:
    for arquivo in sorted(TC_HIST_ROOT.glob('*/campanha.json')):
        try:
            manifest=tc_json.loads(arquivo.read_text())
            for job in manifest['jobs']:
                conf=job['configuracao']; cnn=job['cnn']; slug=''.join(c for c in cnn if c.isalnum())
                for seed in manifest['sementes']:
                    local=arquivo.parent/slug/f'seed_{seed}'
                    if not (local/'concluido.json').is_file():
                        tc_pendentes.append(f'{arquivo.parent.name} | {cnn} | seed {seed}: pendente')
                        continue
                    try:
                        esperado=tc_json.loads(tc_json.dumps(conf)); esperado['configuracao_herdada']['semente']=seed
                        sig=tc_hash.sha256((job['assinatura']+tc_json.dumps(esperado,sort_keys=True)).encode()).hexdigest()
                        salvo=tc_json.loads((local/'configuracao.json').read_text())
                        if salvo['configuracao']!=esperado or salvo['assinatura']!=sig or tc_json.loads((local/'concluido.json').read_text())['assinatura']!=sig:
                            raise ValueError('Manifestos incompatíveis.')
                        d=tc_pd.read_csv(local/'metricas.csv')
                        if len(d)!=21 or d.duplicated(['conjunto','colunas']).any() or set(d.conjunto)!={'treino','validacao','teste'}:
                            raise ValueError('Métricas incompletas/duplicadas.')
                        if not (d.semente==seed).all() or not tc_np.isfinite(d[['MAE_mm','RMSE_mm','MAPE_pct']].to_numpy()).all():
                            raise ValueError('Métricas inválidas.')
                        for _,g in d.groupby('conjunto'):
                            if sorted(g.colunas)!=TC_KS or g.n.nunique()!=1: raise ValueError('K ou contagens divergentes.')
                        d['CNN']=cnn; d['execucao']=arquivo.parent.name; d['grupo']=conf['grupo_comparacao']
                        d['base']=conf['base']; d['parametros']=conf['parametros']; d['arquivo']=str(local)
                        tc_linhas.append(d)
                    except (ValueError,KeyError,OSError) as exc:
                        tc_pendentes.append(f'{local}: ignorado — {exc}')
        except (ValueError,KeyError,OSError) as exc:
            tc_pendentes.append(f'{arquivo}: ignorado — {exc}')

if tc_linhas:
    tc_historico=tc_percentuais(tc_pd.concat(tc_linhas,ignore_index=True))
    tc_groups=[]
    for grupo,d in tc_historico.groupby('grupo',sort=False):
        tc_groups.append((f"Base {d.base.iloc[0]} | parâmetros={d.parametros.iloc[0]} | {grupo[:10]}",grupo))
    tc_sel_grupo=tc_widgets.Dropdown(options=tc_groups,description='Comparação:',layout=tc_widgets.Layout(width='95%'))
    tc_sel_seed=tc_widgets.Dropdown(options=['Média 42/43','42','43'],description='Sementes:')
    tc_sel_conjunto=tc_widgets.Dropdown(options=['teste','validacao','treino'],value='teste',description='Conjunto:')
    tc_out=tc_widgets.Output()

    def tc_desenhar(_=None):
        with tc_out:
            tc_clear(wait=True)
            d=tc_historico.loc[(tc_historico.grupo==tc_sel_grupo.value)&(tc_historico.conjunto==tc_sel_conjunto.value)].copy()
            if tc_sel_seed.value=='Média 42/43':
                completos=d.groupby(['execucao','CNN']).semente.agg(lambda v:set(v)=={42,43})
                d=d.loc[[(r,c) in completos.index and completos.loc[(r,c)] for r,c in zip(d.execucao,d.CNN)]]
            else:
                d=d.loc[d.semente==int(tc_sel_seed.value)]
            if d.empty:
                print('Ainda não há resultados completos para estes filtros. Escolha uma semente concluída ou aguarde.')
                return
            print('Percentual negativo = REDUÇÃO do MAE (melhora); positivo = AUMENTO (piora).')
            print('Referências: estático da mesma execução/semente e K anterior: 0→1→5→10→25→50→100.')
            resumo=d.groupby(['execucao','CNN','parametros','colunas'],sort=False).agg(
                MAE_mm=('MAE_mm','mean'),MAE_desvio=('MAE_mm','std'),
                vs_estatico_pct=('variacao_vs_estatico_pct','mean'),
                vs_anterior_pct=('variacao_vs_anterior_pct','mean'),sementes=('semente','nunique')).reset_index()
            fig,axes=tc_plt.subplots(1,3,figsize=(18,4.6))
            for (run,cnn,param),g in resumo.groupby(['execucao','CNN','parametros'],sort=False):
                g=g.sort_values('colunas')
                rotulo=f'{cnn} | parâmetros={param} | {run}'
                for ax,col in zip(axes,['MAE_mm','vs_estatico_pct','vs_anterior_pct']):
                    ax.plot(range(len(g)),g[col],'o-',label=rotulo,linewidth=1.5,markersize=4)
            for ax,titulo,ylabel in zip(axes,['MAE — menor é melhor','Variação frente ao estático','Variação frente ao K anterior'],['MAE (mm)','Variação do MAE (%)','Variação do MAE (%)']):
                ax.set(title=titulo,xlabel='Colunas temporais (K)',ylabel=ylabel,xticks=range(7),xticklabels=TC_KS)
                ax.grid(alpha=.2)
            for ax in axes[1:]: ax.axhline(0,color='black',linewidth=.8)
            axes[0].legend(fontsize=7,loc='upper left',bbox_to_anchor=(0,-.24))
            fig.suptitle(f"Teste central | {tc_sel_conjunto.value} | {tc_sel_seed.value} | mesma base e protocolo")
            fig.tight_layout(); tc_plt.show(); tc_plt.close(fig)
            def cor(v):
                return '' if tc_pd.isna(v) else ('color:#b42318;font-weight:bold' if v>0 else 'color:#15803d' if v<0 else '')
            tc_display(resumo.drop(columns='MAE_desvio').set_index(['CNN','parametros','execucao','colunas']).style
                .format({'MAE_mm':'{:.4f}','vs_estatico_pct':'{:+.2f}%','vs_anterior_pct':'{:+.2f}%'},na_rep='—')
                .map(cor,subset=['vs_estatico_pct','vs_anterior_pct']))
            print('Média percentual = média das variações PAREADAS por semente, não percentual calculado das médias.')
            print('Duas sementes são descritivas. Use validação para escolher K; teste somente para avaliação.')
            print('Cada curva é uma execução; campanhas repetidas não são misturadas nem tratadas como sementes extras.')
    for w in [tc_sel_grupo,tc_sel_seed,tc_sel_conjunto]: w.observe(tc_desenhar,names='value')
    tc_display(tc_widgets.VBox([tc_sel_grupo,tc_widgets.HBox([tc_sel_seed,tc_sel_conjunto]),tc_out]))
    tc_desenhar()
else:
    print('Sem sementes completas desta nova campanha. Resultados antigos das seções 12–14 continuam preservados, fora desta comparação.')
if tc_pendentes:
    tc_display(TCHTML('<details><summary>Campanhas pendentes / arquivos ignorados</summary><pre>'+tc_escape('\n'.join(tc_pendentes))+'</pre></details>'))


In [ ]:
# TC5 — barras e custo × erro das campanhas temporais carregadas em TC4.
# Execute TC4 antes. Não treina nem altera os arquivos de resultados.
if 'tc_historico' not in globals() or tc_historico.empty:
    print('Execute TC4 para carregar as campanhas salvas primeiro.')
else:
    cp_dados = tc_historico.copy()
    cp_grupos = [(f'Base {g.base.iloc[0]} | parâmetros={g.parametros.iloc[0]} | {key[:10]}', key)
                 for key,g in cp_dados.groupby('grupo',sort=False)]
    cp_base = tc_widgets.Dropdown(options=cp_grupos,description='Base/grupo:',layout=tc_widgets.Layout(width='95%'))
    cp_k = tc_widgets.Dropdown(options=['Todos']+TC_KS,value=10,description='Colunas K:')
    cp_seed = tc_widgets.Dropdown(options=['Média 42/43','42','43'],description='Sementes:')
    cp_split = tc_widgets.Dropdown(options=['teste','validacao','treino'],description='Conjunto:')
    cp_param = tc_widgets.Dropdown(options=['Todos','False','True'],description='Parâmetros:')
    cp_metric = tc_widgets.Dropdown(options=[('MAE (mm) ↓','MAE_mm'),('MAPE (%) ↓','MAPE_pct'),('RMSE (mm) ↓','RMSE_mm'),('R² ↑','R2')],description='Métrica:')
    cp_refresh = tc_widgets.Button(description='Atualizar gráficos',icon='refresh')
    cp_out = tc_widgets.Output()

    def cp_desenhar(_=None):
        with cp_out:
            tc_clear(wait=True)
            d=cp_dados.loc[(cp_dados.grupo==cp_base.value)&(cp_dados.conjunto==cp_split.value)].copy()
            if cp_param.value!='Todos': d=d.loc[d.parametros.astype(str)==cp_param.value]
            if cp_k.value!='Todos': d=d.loc[d.colunas==cp_k.value]
            if cp_seed.value=='Média 42/43':
                d=d.groupby(['execucao','CNN','colunas'],group_keys=False).filter(lambda g:set(g.semente)=={42,43})
            else: d=d.loc[d.semente==int(cp_seed.value)]
            if d.empty:
                print('Nenhum resultado completo para estes filtros.'); return
            # Custo por configuração, não soma de toda a busca de hiperparâmetros.
            d['custo_s']=d['tempo_auxiliar_acumulado_s']+d['tempo_fit_final_s']
            cols=['MAE_mm','MAPE_pct','RMSE_mm','R2','custo_s']
            r=d.groupby(['execucao','CNN','parametros','colunas'],as_index=False)[cols].mean()
            r=r.sort_values(cp_metric.value,ascending=cp_metric.value!='R2').reset_index(drop=True)
            r['ID']=[f'E{i+1:02d}' for i in range(len(r))]
            def data_br(v):
                try: return tc_pd.to_datetime(v,format='%Y%m%dT%H%M%S_%fZ',utc=True).tz_convert('America/Sao_Paulo').strftime('%d/%m/%Y %H:%M:%S')
                except ValueError: return v
            labels=[f'{a.ID} | {a.CNN} | parâmetros={a.parametros} | K={a.colunas} | {data_br(a.execucao)}' for a in r.itertuples()]
            contexto=f'Teste central | {cp_split.value} | {cp_seed.value} | K={cp_k.value}'
            print(contexto)
            if cp_seed.value=='Média 42/43': print('Métricas e tempos: médias das duas sementes; R² é média dos R², não R² de previsões agrupadas.')
            fig,ax=tc_plt.subplots(figsize=(13,max(3,.45*len(r)+1.6)))
            bars=ax.barh(range(len(r)),r[cp_metric.value],color=['#d17b21' if bool(p) else '#386cb0' for p in r.parametros])
            ax.set_yticks(range(len(r)),labels,fontsize=8); ax.invert_yaxis()
            ax.bar_label(bars,labels=[f'{v:.4f}' for v in r[cp_metric.value]],padding=4)
            ax.set_title(f'Comparação temporal — {cp_metric.label}',loc='left',fontweight='bold')
            ax.axvline(0,color='gray',linewidth=.8); ax.grid(axis='x',alpha=.2); ax.set_axisbelow(True); ax.margins(x=.2)
            fig.tight_layout(); tc_plt.show(); tc_plt.close(fig)
            v=r.loc[tc_np.isfinite(r.custo_s)&(r.custo_s>0)&tc_np.isfinite(r.R2)].copy()
            if not v.empty:
                from matplotlib.colors import Normalize
                norm=Normalize(vmin=min(0,float(v.R2.min())),vmax=max(1,float(v.R2.max())))
                fig,ax=tc_plt.subplots(figsize=(12,5.5))
                for param,g in v.groupby('parametros'):
                    sc=ax.scatter(g.custo_s,g.MAE_mm,c=g.R2,cmap='viridis',norm=norm,
                        marker='D' if bool(param) else 'o',s=90,edgecolors='#444444',label=f'parâmetros={param}')
                    for a in g.itertuples(): ax.annotate(f'{a.ID} · K={a.colunas}',(a.custo_s,a.MAE_mm),xytext=(5,7),textcoords='offset points',fontsize=8)
                ax.set_xscale('log'); ax.set_xlabel('Auxiliares acumulados até K + treino final (s, escala logarítmica)')
                ax.set_ylabel('MAE (mm) — menor é melhor'); ax.set_title('Custo × erro — melhor direção: esquerda e abaixo',loc='left',fontweight='bold')
                ax.grid(alpha=.2); ax.legend(); fig.colorbar(sc,ax=ax,label='R² médio' if cp_seed.value=='Média 42/43' else 'R²')
                fig.tight_layout(); tc_plt.show(); tc_plt.close(fig)
            print('Custo exclui extração CNN, gravação dos artefatos e busca dos outros K. Hardware e retomadas podem afetar os tempos.')
            print('Compare no mesmo grupo e K. “Todos” é exploratório; escolha de K deve usar validação, não teste.')
            tc_display(TCHTML('<details><summary>Identificação das execuções e valores</summary>'+r.to_html(index=False,float_format=lambda x:f'{x:.4f}',escape=True)+'</details>'))
    for w in [cp_base,cp_k,cp_seed,cp_split,cp_param,cp_metric]: w.observe(cp_desenhar,names='value')
    cp_refresh.on_click(cp_desenhar)
    tc_display(tc_widgets.VBox([cp_base,tc_widgets.HBox([cp_k,cp_seed,cp_split]),tc_widgets.HBox([cp_param,cp_metric,cp_refresh]),cp_out]))
    cp_desenhar()
